<a href="https://colab.research.google.com/github/tryan01/gravitational-waves/blob/main/microquant_v2_02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import auth
import os

print("Authenticating with Google Cloud...")
auth.authenticate_user()

# Define the bucket and the specific archives to pull
bucket_name = "tryanheider-kalshi-bucket"
archives_to_download = [
    "raw_vault_2026-04-16.tar.gz",
    "raw_vault_2026-04-17.tar.gz",
    "raw_vault_2026-04-18.tar.gz",
    "raw_vault_2026-04-19.tar.gz",
    "raw_vault_2026-04-20.tar.gz",
    "raw_vault_2026-04-22.tar.gz",
    "raw_vault_2026-04-23.tar.gz",
    "raw_vault_2026-04-24.tar.gz",
    "raw_vault_2026-04-25.tar.gz",
    "raw_vault_2026-04-26.tar.gz",
    "raw_vault_2026-04-27.tar.gz",
    "raw_vault_2026-04-28.tar.gz",
    "raw_vault_2026-04-29.tar.gz",
    "raw_vault_2026-04-30.tar.gz"
]

# Create a local directory in Colab to hold the archives
download_dir = '/content/kalshi_raw_archives'
os.makedirs(download_dir, exist_ok=True)

print(f"Downloading archives from gs://{bucket_name} to {download_dir}...")

for archive in archives_to_download:
    gcs_path = f"gs://{bucket_name}/{archive}"
    local_path = os.path.join(download_dir, archive)

    # Use gsutil for fast, multi-threaded downloading
    print(f"Fetching {archive}...")
    !gcloud storage cp {gcs_path} {local_path}

print("\nDownload complete. Archives available on local Colab disk:")
print(os.listdir(download_dir))

In [ ]:
import os
import glob
import tarfile
import ast
import pandas as pd
import numpy as np
import concurrent.futures
from typing import Dict, List, Any, Optional, Tuple

# --- CONFIGURATION ---
DOWNLOAD_DIR = '/content/kalshi_raw_archives'
TARGET_ARCHIVES: List[str] = glob.glob(f"{DOWNLOAD_DIR}/*.tar.gz")
WORKING_DIR: str = '/content/kalshi_working_data'

OUTPUT_DENSE_DIR: str = '/content/kalshi_dense_state_v2'   # For the ML Model
OUTPUT_RAW_DIR: str = '/content/kalshi_clean_raw_v2'       # For the Zero-Assumption Simulator

LEVELS: int = 5
STRIDE_SEC: float = 1.0
WINDOW_HOURS: float = 4.0
WINDOW_SEC: float = WINDOW_HOURS * 3600.0

os.makedirs(WORKING_DIR, exist_ok=True)
os.makedirs(OUTPUT_DENSE_DIR, exist_ok=True)
os.makedirs(OUTPUT_RAW_DIR, exist_ok=True)


def _extract_archive(archive_path: str, extract_path: str) -> None:
    """Extracts a tar archive safely."""
    if not os.path.exists(archive_path):
        print(f"WARNING: Archive not found at {archive_path}")
        return

    archive_name: str = os.path.basename(archive_path).replace('.tar', '').replace('.tar.gz', '')
    target_folder: str = os.path.join(extract_path, archive_name)

    if os.path.exists(target_folder):
        print(f"Archive {archive_name} already extracted. Skipping decompression.")
        return

    print(f"Extracting {archive_path} to {target_folder}...")
    os.makedirs(target_folder, exist_ok=True)
    with tarfile.open(archive_path, 'r:gz') as tar:
        tar.extractall(path=target_folder)
    print(f"Finished extracting {archive_name}.")


def unify_timestamp(ts_series: pd.Series) -> pd.Series:
    """
    Convert mixed timestamp representations into UNIX float seconds.

    Handles:
    - numeric epoch seconds
    - numeric epoch milliseconds
    - ISO strings
    - missing values (preserved as NaN)

    Crucially: this does NOT turn missing timestamps into the huge negative sentinel.
    """
    out = pd.Series(np.nan, index=ts_series.index, dtype=float)

    numeric = pd.to_numeric(ts_series, errors='coerce')
    numeric_mask = numeric.notna()
    if numeric_mask.any():
        numeric_vals = numeric.loc[numeric_mask].astype(float)
        numeric_vals = np.where(numeric_vals > 1e12, numeric_vals / 1000.0, numeric_vals)
        out.loc[numeric_mask] = numeric_vals

    remaining_mask = ~numeric_mask
    if remaining_mask.any():
        dt = pd.to_datetime(ts_series.loc[remaining_mask], format='mixed', errors='coerce', utc=True)
        dt_mask = dt.notna()
        if dt_mask.any():
            out.loc[dt.index[dt_mask]] = dt.loc[dt_mask].astype('int64') / 1e9

    return out


def safe_float(x: Any, default: Optional[float] = np.nan) -> float:
    try:
        if x is None or pd.isna(x):
            return default
        return float(x)
    except Exception:
        return default


def parse_ladder(raw_ladder):
    """
    Parse snapshot ladders stored as nested numpy arrays / lists / tuples / strings.

    Expected shape after normalization:
      [
        ['0.0100', '757201.00'],
        ['0.0200', '4223.00'],
        ...
      ]
    """
    import numpy as np
    import pandas as pd
    import ast

    if raw_ladder is None:
        return []

    try:
        if isinstance(raw_ladder, float) and pd.isna(raw_ladder):
            return []
    except Exception:
        pass

    ladder_obj = raw_ladder

    # pyarrow scalar support
    if hasattr(ladder_obj, "as_py"):
        try:
            ladder_obj = ladder_obj.as_py()
        except Exception:
            pass

    # convert top-level numpy arrays / array-likes
    if hasattr(ladder_obj, "tolist") and not isinstance(ladder_obj, (str, bytes)):
        try:
            ladder_obj = ladder_obj.tolist()
        except Exception:
            pass

    # string representation support
    if isinstance(ladder_obj, str):
        s = ladder_obj.strip()
        if s == "" or s.lower() == "none":
            return []
        try:
            ladder_obj = ast.literal_eval(s)
        except Exception:
            return []

    if not isinstance(ladder_obj, (list, tuple)):
        return []

    out = []
    for item in ladder_obj:
        if hasattr(item, "as_py"):
            try:
                item = item.as_py()
            except Exception:
                pass

        if hasattr(item, "tolist") and not isinstance(item, (str, bytes)):
            try:
                item = item.tolist()
            except Exception:
                pass

        if not isinstance(item, (list, tuple)) or len(item) != 2:
            continue

        try:
            px = float(item[0])
            vol = float(item[1])
        except Exception:
            continue

        if np.isnan(px) or np.isnan(vol) or vol <= 0:
            continue

        out.append((round(px, 2), vol))

    return out


def apply_snapshot_row(row: pd.Series) -> Tuple[Dict[float, float], Dict[float, float]]:
    """
    Build yes-bid / yes-ask books from a snapshot row.

    Observed raw schema:
      yes_dollars_fp -> YES resting bids
      no_dollars_fp  -> NO resting bids, which imply YES asks via ask = 1 - no_bid
    """
    bids: Dict[float, float] = {}
    asks: Dict[float, float] = {}

    yes_ladder = parse_ladder(row.get('yes_dollars_fp'))
    no_ladder = parse_ladder(row.get('no_dollars_fp'))

    for px, vol in yes_ladder:
        if vol > 1e-5:
            bids[round(px, 2)] = float(vol)

    for no_px, vol in no_ladder:
        yes_ask_px = round(1.0 - float(no_px), 2)
        if vol > 1e-5:
            asks[yes_ask_px] = float(vol)

    return bids, asks


def _snapshot_state(
    bids: Dict[float, float],
    asks: Dict[float, float],
    vol_yes: float,
    vol_no: float,
    min_trade: float,
    max_trade: float,
    current_local_ts: float,
    current_exchange_ts: float,
    max_ts: float
) -> Dict[str, float]:
    sorted_bids = sorted(bids.items(), key=lambda x: x[0], reverse=True)
    sorted_asks = sorted(asks.items(), key=lambda x: x[0])

    state = {
        "ts": float(current_exchange_ts) if pd.notna(current_exchange_ts) else np.nan,
        "local_recv_ts": float(current_local_ts),
    }

    for i in range(LEVELS):
        state[f"bid_px_{i+1}"] = float(sorted_bids[i][0]) if i < len(sorted_bids) else 0.0
        state[f"bid_vol_{i+1}"] = float(sorted_bids[i][1]) if i < len(sorted_bids) else 0.0
        state[f"ask_px_{i+1}"] = float(sorted_asks[i][0]) if i < len(sorted_asks) else 1.0
        state[f"ask_vol_{i+1}"] = float(sorted_asks[i][1]) if i < len(sorted_asks) else 0.0

    state["taker_vol_yes"] = vol_yes
    state["taker_vol_no"] = vol_no
    state["window_min_trade_price"] = min_trade
    state["window_max_trade_price"] = max_trade
    state["time_fraction"] = (max_ts - current_local_ts) / WINDOW_SEC

    return state


def process_single_market(market_dir: str) -> str:
    """
    Processes a single market:
    - normalizes timestamps without sentinel corruption
    - saves cleaned raw rows
    - reconstructs the book using true snapshot parsing
    - treats snapshots as hard resets
    - does NOT invalidate on seq jumps inside a single market file
    """
    ticker: str = os.path.basename(market_dir)
    dense_output_path: str = os.path.join(OUTPUT_DENSE_DIR, f"dense_{ticker}.parquet")
    raw_output_path = os.path.join(OUTPUT_RAW_DIR, f"clean_raw_{ticker}.parquet")

    if os.path.exists(dense_output_path) and os.path.exists(raw_output_path):
        return f"[{ticker}] SKIPPED: Both V2 files already exist."

    partition_files: List[str] = glob.glob(f"{market_dir}/*.parquet")
    if not partition_files:
        return f"[{ticker}] SKIPPED: No parquet files."

    try:
        # 1. Concatenate all partitions for this market
        df_list: List[pd.DataFrame] = [pd.read_parquet(f) for f in partition_files]
        master_df: pd.DataFrame = pd.concat(df_list, ignore_index=True)

        # 2. Normalize clocks safely
        if 'ts' in master_df.columns:
            master_df['ts'] = unify_timestamp(master_df['ts'])
        else:
            master_df['ts'] = np.nan

        if 'local_recv_ts' in master_df.columns:
            master_df['local_recv_ts'] = unify_timestamp(master_df['local_recv_ts'])
        else:
            raise ValueError("local_recv_ts missing from raw data")

        # 3. Sort by local receive time; seq is only a stable tie-breaker, not a continuity rule
        if 'seq' not in master_df.columns:
            master_df['seq'] = np.nan
        master_df = master_df.sort_values(by=['local_recv_ts', 'seq'], kind='mergesort').reset_index(drop=True)

        # 4. Save cleaned raw rows for simulator / audits
        master_df.to_parquet(raw_output_path, index=False)

        # 5. Time grid setup
        max_ts: float = float(master_df['local_recv_ts'].max())
        min_ts_clipped: float = max(float(master_df['local_recv_ts'].min()), max_ts - WINDOW_SEC)
        time_grid: np.ndarray = np.arange(min_ts_clipped, max_ts, STRIDE_SEC)

        # 6. Reconstruction state
        resting_bids: Dict[float, float] = {}
        resting_asks: Dict[float, float] = {}

        book_valid: bool = False
        grid_idx: int = 0
        grid_features: List[Dict[str, float]] = []

        rolling_taker_yes: float = 0.0
        rolling_taker_no: float = 0.0
        window_min_trade_price: float = np.inf
        window_max_trade_price: float = -np.inf
        last_exchange_ts: float = np.nan

        # Diagnostics
        snapshot_count: int = 0
        skipped_delta_pre_snapshot_count: int = 0
        empty_snapshot_count: int = 0
        valid_grid_count: int = 0
        total_grid_count: int = 0

        # 7. Walk forward through the full market
        for _, row in master_df.iterrows():
            current_event_ts: float = float(row['local_recv_ts'])

            while grid_idx < len(time_grid) and current_event_ts > time_grid[grid_idx]:
                grid_features.append(_snapshot_state(
                    resting_bids,
                    resting_asks,
                    rolling_taker_yes,
                    rolling_taker_no,
                    window_min_trade_price,
                    window_max_trade_price,
                    float(time_grid[grid_idx]),
                    last_exchange_ts,
                    max_ts
                ))
                total_grid_count += 1
                if book_valid:
                    valid_grid_count += 1

                grid_idx += 1
                rolling_taker_yes, rolling_taker_no = 0.0, 0.0
                window_min_trade_price, window_max_trade_price = np.inf, -np.inf

            if pd.notna(row.get('ts')):
                last_exchange_ts = float(row['ts'])

            event_type = str(row.get('event_type'))

            # -------------------------
            # ORDER BOOK EVENTS
            # -------------------------
            if event_type == 'orderbook_snapshot':
                new_bids, new_asks = apply_snapshot_row(row)
                snapshot_count += 1

                # Ignore terminal / empty snapshots instead of wiping the book
                if len(new_bids) == 0 and len(new_asks) == 0:
                    empty_snapshot_count += 1
                    continue

                resting_bids = new_bids
                resting_asks = new_asks
                book_valid = True
                continue

            elif event_type == 'orderbook_delta':
                if not book_valid:
                    skipped_delta_pre_snapshot_count += 1
                    continue

                tick_side = str(row.get('side'))
                price_val = safe_float(row.get('price_dollars'), default=np.nan)
                delta = safe_float(row.get('delta_fp'), default=0.0)

                if pd.isna(price_val) or tick_side not in ['yes', 'no']:
                    continue

                price_yes = round(price_val, 2) if tick_side == 'yes' else round(1.0 - price_val, 2)
                book = resting_bids if tick_side == 'yes' else resting_asks

                new_vol = book.get(price_yes, 0.0) + delta
                if new_vol > 1e-5:
                    book[price_yes] = new_vol
                else:
                    book.pop(price_yes, None)

            # -------------------------
            # TRADE EVENTS
            # -------------------------
            elif event_type == 'trade':
                taker_side = str(row.get('taker_side'))

                yes_px = safe_float(row.get('yes_price_dollars'), default=np.nan)
                no_px = safe_float(row.get('no_price_dollars'), default=np.nan)

                if not pd.isna(yes_px):
                    price = yes_px
                elif not pd.isna(no_px):
                    price = round(1.0 - no_px, 2)
                else:
                    continue

                trade_count = safe_float(row.get('count_fp'), default=0.0)

                if taker_side == 'yes':
                    rolling_taker_yes += trade_count
                elif taker_side == 'no':
                    rolling_taker_no += trade_count

                if price < window_min_trade_price:
                    window_min_trade_price = price
                if price > window_max_trade_price:
                    window_max_trade_price = price

        # 8. Flush trailing grid points
        while grid_idx < len(time_grid):
            grid_features.append(_snapshot_state(
                resting_bids,
                resting_asks,
                rolling_taker_yes,
                rolling_taker_no,
                window_min_trade_price,
                window_max_trade_price,
                float(time_grid[grid_idx]),
                last_exchange_ts,
                max_ts
            ))
            total_grid_count += 1
            if book_valid:
                valid_grid_count += 1

            grid_idx += 1
            rolling_taker_yes, rolling_taker_no = 0.0, 0.0
            window_min_trade_price, window_max_trade_price = np.inf, -np.inf

        # 9. Save dense output
        df_grid: pd.DataFrame = pd.DataFrame(grid_features)
        if df_grid.empty:
            return f"[{ticker}] FAILED: Output grid empty."

        same_mask = (
            df_grid["ts"].notna()
            & df_grid["local_recv_ts"].notna()
            & (df_grid["ts"].to_numpy() == df_grid["local_recv_ts"].to_numpy())
        )
        if len(df_grid) > 0 and same_mask.all():
            raise ValueError("Clock bug: dense snapshots still have identical ts and local_recv_ts.")

        df_grid.to_parquet(dense_output_path, index=False)

        valid_grid_rate = (valid_grid_count / total_grid_count) if total_grid_count > 0 else 0.0
        return (
            f"[{ticker}] SUCCESS: Wrote Clean Raw & {len(df_grid)} Dense rows | "
            f"snapshots={snapshot_count} | "
            f"empty_snapshots={empty_snapshot_count} | "
            f"skipped_pre_snapshot_deltas={skipped_delta_pre_snapshot_count} | "
            f"valid_grid_rate={valid_grid_rate:.3f}"
        )

    except Exception as e:
        return f"[{ticker}] ERROR: {str(e)}"


if __name__ == '__main__':
    print("--- Starting Pipeline V2 Initialization ---")

    for archive in TARGET_ARCHIVES:
        _extract_archive(archive, WORKING_DIR)

    all_parquets: List[str] = glob.glob(f"{WORKING_DIR}/**/*.parquet", recursive=True)
    if not all_parquets:
        raise FileNotFoundError(f"No .parquet files found in {WORKING_DIR}. Check target archives.")

    market_directories: List[str] = list(set([os.path.dirname(p) for p in all_parquets]))
    print(f"\nDiscovered {len(market_directories)} uncompressed markets. Beginning parallel V2 generation...")

    with concurrent.futures.ProcessPoolExecutor() as executor:
        results: List[str] = list(executor.map(process_single_market, market_directories))

    for r in results:
        print(r)

    print("\nExtraction Complete. Ready for Colab Phase 2.")

In [ ]:
%%bash
set -e

# Check sizes first
du -sh /content/kalshi_dense_state_v2 /content/kalshi_clean_raw_v2

# Make processed-cache folder in your bucket
gcloud storage mkdir gs://tryanheider-kalshi-bucket/processed_cache || true

# Compress each folder
tar -czf /content/kalshi_dense_state_v2.tar.gz -C /content kalshi_dense_state_v2
tar -czf /content/kalshi_clean_raw_v2.tar.gz -C /content kalshi_clean_raw_v2

# Check archive sizes
ls -lh /content/kalshi_dense_state_v2.tar.gz /content/kalshi_clean_raw_v2.tar.gz

# Upload to GCS
gcloud storage cp /content/kalshi_dense_state_v2.tar.gz gs://tryanheider-kalshi-bucket/processed_cache/
gcloud storage cp /content/kalshi_clean_raw_v2.tar.gz gs://tryanheider-kalshi-bucket/processed_cache/

# Confirm
gcloud storage ls -lh gs://tryanheider-kalshi-bucket/processed_cache/

In [ ]:
# ============================================================
# MicroQuant Stage 1 Dataset: Hot-Density Regime Target
# ============================================================
# Drop-in replacement for KalshiDenseDataset when training a Stage 1
# hot-regime detector.
#
# Core target:
#   For an anchor time t, scan candidate entry offsets e = t, ..., t + candidate_lookahead.
#   At each e, ask whether the then-current bid/ask pair would BOTH fill within
#   fill_lookahead seconds.
#
#   hot_count = number of candidate offsets where both sides fill.
#   label = 1 if hot_count > k_hot else 0.
#
# This is deliberately NOT simulator-PnL. It is a sim-agnostic-ish
# opportunity-density target meant for Stage 1.
#
# Expected model change:
#   model = KalshiTransformer(feature_dim=23, ..., num_classes=2)
#

import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from typing import List, Dict, Optional, Any


class KalshiHotDensityDataset(Dataset):
    def __init__(
        self,
        file_paths: List[str],
        seq_len: int = 60,
        candidate_lookahead: int = 30,
        fill_lookahead: int = 30,
        stride: int = 2,
        k_hot: int = 5,
        min_spread: float = 0.02,
        inference_mode: bool = False,
        return_diagnostics: bool = False,
        use_grouped_ofi: bool = True,
        explicit_feature_cols: Optional[List[str]] = None,
    ):
        """
        Binary hot-regime dataset.

        Parameters
        ----------
        file_paths:
            Dense parquet files, same as the current KalshiDenseDataset.

        seq_len:
            Number of historical seconds fed to the transformer.

        candidate_lookahead:
            How far forward from anchor t to scan possible future entry times.
            For the first pass, 30 is natural because the best diagnostic policy
            was a 30s hot-window variant.

        fill_lookahead:
            For each candidate entry time e, how far forward to check whether
            the then-current bid and ask both fill. 30 matches the old target.

        stride:
            Dataset sampling stride. Use 2 for training initially; use 1 for
            inference/evaluation if you want maximum temporal coverage.

        k_hot:
            Binary target is hot_count > k_hot. If k_hot=5, then at least 6
            candidate offsets must both-fill. Change the comparison line below
            if you prefer >= semantics.

        min_spread:
            Candidate quote pairs are only counted as good/bad if ask-bid >= min_spread.
            This keeps Stage 1 focused on structurally tradable regimes without
            using full simulator PnL.

        inference_mode:
            If True, __getitem__ returns ticker/timestamps/features, matching your
            existing inference pattern.

        return_diagnostics:
            If True in training mode, __getitem__ returns (features, label, diag_dict).
            Useful for audits, slower for normal training.

        use_grouped_ofi:
            True computes OFI within each ticker file, fixing cross-file shift bleed.
            False mimics the old global-shift behavior more closely.
        """
        self.seq_len = int(seq_len)
        self.candidate_lookahead = int(candidate_lookahead)
        self.fill_lookahead = int(fill_lookahead)
        self.stride = int(stride)
        self.k_hot = int(k_hot)
        self.min_spread = float(min_spread)
        self.inference_mode = bool(inference_mode)
        self.return_diagnostics = bool(return_diagnostics)
        self.use_grouped_ofi = bool(use_grouped_ofi)

        self.bid_col = "bid_px_1"
        self.ask_col = "ask_px_1"
        self.trade_min_col = "window_min_trade_price"
        self.trade_max_col = "window_max_trade_price"

        if explicit_feature_cols is None:
            feature_cols = []
            for level in range(1, 6):
                feature_cols += [
                    f"bid_px_{level}",
                    f"bid_vol_{level}",
                    f"ask_px_{level}",
                    f"ask_vol_{level}",
                ]
            feature_cols += ["taker_vol_yes", "taker_vol_no", "OFI"]
            self.feature_cols = feature_cols
        else:
            self.feature_cols = list(explicit_feature_cols)

        print(f"Loading {len(file_paths)} parquet files...")
        dfs = []
        for f in file_paths:
            df = pd.read_parquet(f)
            df = df.copy()
            df["ticker"] = os.path.basename(f)
            dfs.append(df)

        if not dfs:
            raise ValueError("No dense files supplied.")

        self.data = pd.concat(dfs, ignore_index=True)

        required_cols = {
            "ticker",
            "ts",
            "local_recv_ts",
            self.bid_col,
            self.ask_col,
            self.trade_min_col,
            self.trade_max_col,
        }
        missing_required = required_cols - set(self.data.columns)
        if missing_required:
            raise ValueError(f"Missing required columns: {sorted(missing_required)}")

        # --------------------------------------------------------
        # Feature engineering: OFI + log-volume normalization
        # --------------------------------------------------------
        self._add_ofi()

        missing_features = [c for c in self.feature_cols if c not in self.data.columns]
        if missing_features:
            raise ValueError(f"Missing feature columns after OFI construction: {missing_features}")

        vol_cols = [col for col in self.data.columns if "vol" in col.lower()]
        self.data[vol_cols] = np.log1p(self.data[vol_cols].astype(float))

        # Stable arrays.
        self.ticker_series = self.data["ticker"].astype(str).values
        self.ts_array = self.data["ts"].values
        self.local_recv_ts_array = self.data["local_recv_ts"].values

        self.bid_array = pd.to_numeric(self.data[self.bid_col], errors="coerce").fillna(0.0).values.astype(float)
        self.ask_array = pd.to_numeric(self.data[self.ask_col], errors="coerce").fillna(1.0).values.astype(float)

        self.trade_min_array = (
            pd.to_numeric(self.data[self.trade_min_col], errors="coerce")
            .fillna(np.inf)
            .values
            .astype(float)
        )
        self.trade_max_array = (
            pd.to_numeric(self.data[self.trade_max_col], errors="coerce")
            .fillna(-np.inf)
            .values
            .astype(float)
        )

        self.features_array = self.data[self.feature_cols].astype(float).values
        self.features_array = np.nan_to_num(self.features_array, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

        # --------------------------------------------------------
        # Label precomputation
        # --------------------------------------------------------
        self._precompute_hot_density_labels()
        self._build_valid_indices()

        print(f"Dataset built with {len(self.indices)} valid samples.")
        self._print_label_summary()

    # ============================================================
    # Feature engineering
    # ============================================================
    def _add_ofi(self) -> None:
        """Adds OFI using either grouped or legacy global shift semantics."""
        if not self.use_grouped_ofi:
            self.data["OFI"] = self._compute_ofi_for_frame(self.data)
            return

        ofi = np.zeros(len(self.data), dtype=float)
        for _, idx in self.data.groupby("ticker", sort=False).indices.items():
            idx = np.asarray(idx)
            ofi[idx] = self._compute_ofi_for_frame(self.data.iloc[idx]).to_numpy(dtype=float)
        self.data["OFI"] = ofi
        self.data["OFI"] = self.data["OFI"].fillna(0.0)

    @staticmethod
    def _compute_ofi_for_frame(df: pd.DataFrame) -> pd.Series:
        bid_px = pd.to_numeric(df["bid_px_1"], errors="coerce")
        bid_vol = pd.to_numeric(df["bid_vol_1"], errors="coerce")
        ask_px = pd.to_numeric(df["ask_px_1"], errors="coerce")
        ask_vol = pd.to_numeric(df["ask_vol_1"], errors="coerce")

        bid_px_prev = bid_px.shift(1)
        bid_vol_prev = bid_vol.shift(1)
        ask_px_prev = ask_px.shift(1)
        ask_vol_prev = ask_vol.shift(1)

        bid_ofi = np.zeros(len(df), dtype=float)
        bid_ofi = np.where(bid_px > bid_px_prev, bid_vol, bid_ofi)
        bid_ofi = np.where(bid_px == bid_px_prev, bid_vol - bid_vol_prev, bid_ofi)
        bid_ofi = np.where(bid_px < bid_px_prev, -bid_vol_prev, bid_ofi)

        ask_ofi = np.zeros(len(df), dtype=float)
        ask_ofi = np.where(ask_px < ask_px_prev, ask_vol, ask_ofi)
        ask_ofi = np.where(ask_px == ask_px_prev, ask_vol - ask_vol_prev, ask_ofi)
        ask_ofi = np.where(ask_px > ask_px_prev, -ask_vol_prev, ask_ofi)

        raw_ofi = bid_ofi - ask_ofi
        out = np.sign(raw_ofi) * np.log1p(np.abs(raw_ofi))
        return pd.Series(out, index=df.index).fillna(0.0)

    # ============================================================
    # Label construction
    # ============================================================
    @staticmethod
    def _forward_min(arr: np.ndarray, horizon: int) -> np.ndarray:
        """At j, min(arr[j+1 : j+horizon+1])."""
        s = pd.Series(arr).shift(-1)
        return s.iloc[::-1].rolling(window=horizon, min_periods=1).min().iloc[::-1].to_numpy(dtype=float)

    @staticmethod
    def _forward_max(arr: np.ndarray, horizon: int) -> np.ndarray:
        """At j, max(arr[j+1 : j+horizon+1])."""
        s = pd.Series(arr).shift(-1)
        return s.iloc[::-1].rolling(window=horizon, min_periods=1).max().iloc[::-1].to_numpy(dtype=float)

    @staticmethod
    def _range_count(binary_arr: np.ndarray, start_offset: int, end_offset: int) -> np.ndarray:
        """
        For each position p, count binary_arr[p+start_offset : p+end_offset+1].
        Offsets are inclusive.
        """
        n = len(binary_arr)
        vals = binary_arr.astype(np.int64)
        csum = np.concatenate([[0], np.cumsum(vals)])
        p = np.arange(n)
        start = np.clip(p + start_offset, 0, n)
        end_excl = np.clip(p + end_offset + 1, 0, n)
        return csum[end_excl] - csum[start]

    def _precompute_hot_density_labels(self) -> None:
        n_total = len(self.data)

        self.candidate_good = np.zeros(n_total, dtype=np.int8)
        self.candidate_bad = np.zeros(n_total, dtype=np.int8)

        self.hot_count_0_30 = np.zeros(n_total, dtype=np.int16)
        self.bad_count_0_30 = np.zeros(n_total, dtype=np.int16)

        self.hot_count_0_10 = np.zeros(n_total, dtype=np.int16)
        self.hot_count_10_20 = np.zeros(n_total, dtype=np.int16)
        self.hot_count_20_30 = np.zeros(n_total, dtype=np.int16)

        self.bad_count_0_10 = np.zeros(n_total, dtype=np.int16)
        self.bad_count_10_20 = np.zeros(n_total, dtype=np.int16)
        self.bad_count_20_30 = np.zeros(n_total, dtype=np.int16)

        self.hot_label = np.zeros(n_total, dtype=np.int64)

        for _, idx in self.data.groupby("ticker", sort=False).indices.items():
            idx = np.asarray(idx)

            bid = self.bid_array[idx]
            ask = self.ask_array[idx]
            trade_min = self.trade_min_array[idx]
            trade_max = self.trade_max_array[idx]

            future_min_trade = self._forward_min(trade_min, self.fill_lookahead)
            future_max_trade = self._forward_max(trade_max, self.fill_lookahead)
            future_min_ask = self._forward_min(ask, self.fill_lookahead)
            future_max_bid = self._forward_max(bid, self.fill_lookahead)

            spread = ask - bid
            tradable = (
                np.isfinite(bid)
                & np.isfinite(ask)
                & (bid > 0.0)
                & (ask < 1.0)
                & (ask > bid)
                & (spread >= self.min_spread)
            )

            bid_fills = future_min_trade < bid
            ask_fills = future_max_trade > ask

            good = tradable & bid_fills & ask_fills

            # Toxic-ish candidate, mirroring the spirit of the old class-0 logic:
            # one side fills, the other does not, and the book reprices against us.
            ask_repriced_down = future_min_ask < ask
            bid_repriced_up = future_max_bid > bid
            bad = tradable & (
                (bid_fills & ~ask_fills & ask_repriced_down)
                | (ask_fills & ~bid_fills & bid_repriced_up)
            )

            good_i = good.astype(np.int8)
            bad_i = bad.astype(np.int8)

            # Full candidate window, default offsets 0..30.
            hot_0_30 = self._range_count(good_i, 0, self.candidate_lookahead)
            bad_0_30 = self._range_count(bad_i, 0, self.candidate_lookahead)

            # Diagnostic timing buckets. If candidate_lookahead < 30 these still work,
            # but later buckets will be clipped by valid-index construction.
            hot_0_10 = self._range_count(good_i, 0, min(10, self.candidate_lookahead))
            hot_10_20 = self._range_count(good_i, 10, min(20, self.candidate_lookahead))
            hot_20_30 = self._range_count(good_i, 20, min(30, self.candidate_lookahead))

            bad_0_10 = self._range_count(bad_i, 0, min(10, self.candidate_lookahead))
            bad_10_20 = self._range_count(bad_i, 10, min(20, self.candidate_lookahead))
            bad_20_30 = self._range_count(bad_i, 20, min(30, self.candidate_lookahead))

            # Primary target: binary hotness from good-count density.
            # User-requested semantics: hot_count > k.
            label = (hot_0_30 > self.k_hot).astype(np.int64)

            self.candidate_good[idx] = good_i
            self.candidate_bad[idx] = bad_i

            self.hot_count_0_30[idx] = hot_0_30.astype(np.int16)
            self.bad_count_0_30[idx] = bad_0_30.astype(np.int16)

            self.hot_count_0_10[idx] = hot_0_10.astype(np.int16)
            self.hot_count_10_20[idx] = hot_10_20.astype(np.int16)
            self.hot_count_20_30[idx] = hot_20_30.astype(np.int16)

            self.bad_count_0_10[idx] = bad_0_10.astype(np.int16)
            self.bad_count_10_20[idx] = bad_10_20.astype(np.int16)
            self.bad_count_20_30[idx] = bad_20_30.astype(np.int16)

            self.hot_label[idx] = label

    def _build_valid_indices(self) -> None:
        """Builds sample anchors whose history and all label lookahead windows stay within ticker."""
        valid_indices = []
        farthest_lookahead = self.candidate_lookahead + self.fill_lookahead
        max_idx = len(self.data) - farthest_lookahead - 1

        for i in range(self.seq_len, max_idx, self.stride):
            history_start = i - self.seq_len
            future_end = i + farthest_lookahead
            if self.ticker_series[history_start] == self.ticker_series[i] == self.ticker_series[future_end]:
                valid_indices.append(i)

        self.indices = np.asarray(valid_indices, dtype=np.int64)

    # ============================================================
    # Dataset API
    # ============================================================
    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        current_idx = int(self.indices[idx])
        history_window = self.features_array[current_idx - self.seq_len : current_idx]
        features = torch.tensor(history_window, dtype=torch.float32)

        if self.inference_mode:
            ticker = str(self.ticker_series[current_idx])
            ts_exchange = float(self.ts_array[current_idx])
            ts_recv = float(self.local_recv_ts_array[current_idx])
            return ticker, ts_exchange, ts_recv, features

        label = torch.tensor(int(self.hot_label[current_idx]), dtype=torch.long)

        if not self.return_diagnostics:
            return features, label

        diag = self.get_diagnostics_for_index(current_idx)
        return features, label, diag

    def get_diagnostics_for_index(self, current_idx: int) -> Dict[str, Any]:
        current_idx = int(current_idx)
        return {
            "ticker": str(self.ticker_series[current_idx]),
            "ts": float(self.ts_array[current_idx]),
            "local_recv_ts": float(self.local_recv_ts_array[current_idx]),
            "label": int(self.hot_label[current_idx]),
            "hot_count_0_30": int(self.hot_count_0_30[current_idx]),
            "bad_count_0_30": int(self.bad_count_0_30[current_idx]),
            "hot_count_0_10": int(self.hot_count_0_10[current_idx]),
            "hot_count_10_20": int(self.hot_count_10_20[current_idx]),
            "hot_count_20_30": int(self.hot_count_20_30[current_idx]),
            "bad_count_0_10": int(self.bad_count_0_10[current_idx]),
            "bad_count_10_20": int(self.bad_count_10_20[current_idx]),
            "bad_count_20_30": int(self.bad_count_20_30[current_idx]),
            "candidate_good_now": int(self.candidate_good[current_idx]),
            "candidate_bad_now": int(self.candidate_bad[current_idx]),
            "bid_px_1": float(self.bid_array[current_idx]),
            "ask_px_1": float(self.ask_array[current_idx]),
            "spread": float(self.ask_array[current_idx] - self.bid_array[current_idx]),
        }

    def diagnostics_frame(self, max_rows: Optional[int] = None) -> pd.DataFrame:
        """Returns diagnostics for valid sample anchors. Useful for class-balance audits."""
        idxs = self.indices if max_rows is None else self.indices[:max_rows]
        rows = [self.get_diagnostics_for_index(int(i)) for i in idxs]
        return pd.DataFrame(rows)

    def _print_label_summary(self) -> None:
        if len(self.indices) == 0:
            print("No valid indices; cannot summarize labels.")
            return

        labels = self.hot_label[self.indices]
        counts = pd.Series(labels).value_counts().sort_index()
        rates = counts / counts.sum()

        print("\nHot-density label distribution:")
        for label_value in [0, 1]:
            c = int(counts.get(label_value, 0))
            r = float(rates.get(label_value, 0.0))
            print(f"  class {label_value}: {c} ({r:.4f})")

        hot_counts = self.hot_count_0_30[self.indices]
        bad_counts = self.bad_count_0_30[self.indices]
        print("\nhot_count_0_30 quantiles:")
        print(pd.Series(hot_counts).quantile([0.50, 0.75, 0.90, 0.95, 0.99]).to_string())
        print("\nbad_count_0_30 quantiles:")
        print(pd.Series(bad_counts).quantile([0.50, 0.75, 0.90, 0.95, 0.99]).to_string())


# ============================================================
# Helper audit functions
# ============================================================

def audit_hot_density_dataset(dataset: KalshiHotDensityDataset, by_ticker: bool = True) -> Dict[str, pd.DataFrame]:
    """Quick diagnostic summaries for the dataset labels and counts."""
    df = dataset.diagnostics_frame()
    out = {}

    label_summary = (
        df.groupby("label")
        .agg(
            samples=("label", "size"),
            mean_hot_count=("hot_count_0_30", "mean"),
            median_hot_count=("hot_count_0_30", "median"),
            mean_bad_count=("bad_count_0_30", "mean"),
            median_bad_count=("bad_count_0_30", "median"),
            mean_spread=("spread", "mean"),
        )
        .reset_index()
    )
    label_summary["rate"] = label_summary["samples"] / len(df)
    out["label_summary"] = label_summary

    if by_ticker:
        ticker_summary = (
            df.groupby("ticker")
            .agg(
                samples=("label", "size"),
                hot_rate=("label", "mean"),
                mean_hot_count=("hot_count_0_30", "mean"),
                mean_bad_count=("bad_count_0_30", "mean"),
            )
            .reset_index()
            .sort_values("hot_rate", ascending=False)
        )
        out["ticker_summary"] = ticker_summary

    print("\n=== Label summary ===")
    print(out["label_summary"].to_string(index=False))

    if by_ticker:
        print("\n=== Ticker summary ===")
        print(out["ticker_summary"].to_string(index=False))

    return out


# ============================================================
# Example usage
# ============================================================
# train_hot_dataset = KalshiHotDensityDataset(
#     file_paths=train_files,
#     seq_len=60,
#     candidate_lookahead=30,
#     fill_lookahead=30,
#     stride=2,
#     k_hot=5,
#     min_spread=0.02,
#     inference_mode=False,
# )
#
# val_hot_dataset = KalshiHotDensityDataset(
#     file_paths=val_files,
#     seq_len=60,
#     candidate_lookahead=30,
#     fill_lookahead=30,
#     stride=2,
#     k_hot=5,
#     min_spread=0.02,
#     inference_mode=False,
# )
#
# audit_hot_density_dataset(train_hot_dataset)
# audit_hot_density_dataset(val_hot_dataset)
#
# train_loader = DataLoader(train_hot_dataset, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
# val_loader = DataLoader(val_hot_dataset, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
#
# model = KalshiTransformer(feature_dim=23, d_model=64, nhead=4, num_layers=2, num_classes=2).to(device)


In [ ]:
# edited dataloading for clean data experiment

import os
import glob
from torch.utils.data import DataLoader

print("Scanning local disk for Dense Tensors...")
local_path = '/content/kalshi_dense_state_v2'
all_dense_files = glob.glob(f"{local_path}/*.parquet")

if not all_dense_files:
    raise FileNotFoundError(f"No parquet files found in {local_path}.")

# Group files by Base Game to prevent leakage across dual-market files
market_groups = {}
for file_path in all_dense_files:
    filename = os.path.basename(file_path)
    parts = filename.split('-')
    if len(parts) >= 2:
        base_game = parts[1]   # e.g. '26APR221845NYYBOS'
        market_groups.setdefault(base_game, []).append(file_path)

unique_games = sorted(market_groups.keys())
print(f"Found {len(unique_games)} unique MLB games.")

# Hold out APR 22 games only
APR23_PREFIX = "26APR29"

train_games = [g for g in unique_games if not g.startswith(APR23_PREFIX)]
val_games   = [g for g in unique_games if g.startswith(APR23_PREFIX)]

train_files = [f for game in train_games for f in market_groups[game]]
val_files   = [f for game in val_games for f in market_groups[game]]

print(f"Train Set (non-APR22): {len(train_games)} games -> {len(train_files)} dual-market files")
print(f"Val Set (APR22 only):  {len(val_games)} games -> {len(val_files)} dual-market files")

if len(val_games) == 0:
    raise ValueError("No APR 22 games found in dense files.")
if len(train_games) == 0:
    raise ValueError("No non-APR 22 games found in dense files.")

print("\nBuilding Training Dataset...")
train_dataset = KalshiHotDensityDataset(
        file_paths=train_files,
        seq_len=60,
        candidate_lookahead=30,
        fill_lookahead=30,
        stride=1,
        k_hot=5,
        min_spread=0.02,
        inference_mode=False,
)

print("\nBuilding Validation Dataset...")
val_dataset = KalshiHotDensityDataset(
        file_paths=val_files,
        seq_len=60,
        candidate_lookahead=30,
        fill_lookahead=30,
        stride=1,
        k_hot=5,
        min_spread=0.02,
        inference_mode=False,
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

train_dataset.return_diagnostics = True
val_dataset.return_diagnostics = True

print("\nDataLoaders Ready.")
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor

class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 5000):
        super().__init__()
        pe: Tensor = torch.zeros(max_len, d_model)
        position: Tensor = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term: Tensor = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x: Tensor) -> Tensor:
        x = x + self.pe[:, :x.size(1), :]
        return x

class KalshiTransformer(nn.Module):
    def __init__(self, feature_dim: int = 23, d_model: int = 64, nhead: int = 4, num_layers: int = 2, num_classes: int = 3):
        super().__init__()
        self.d_model: int = d_model


        # 1. Input Projection
        self.input_proj = nn.Linear(feature_dim, d_model)
        self.pos_encoder = PositionalEncoding(d_model)

        # 2. Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True, dropout=0.1)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # 3. Classification Head
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 32),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(32, num_classes)
        )

    def forward(self, x: Tensor) -> Tensor:

        # Project and encode
        x = self.input_proj(x) * math.sqrt(self.d_model)
        x = self.pos_encoder(x)

        # Pass through attention layers
        x = self.transformer_encoder(x)

        # Extract the final time-step representation
        final_step: Tensor = x[:, -1, :]

        logits: Tensor = self.classifier(final_step)
        return logits

class FocalLoss(nn.Module):
    def __init__(self, alpha: Tensor, gamma: float = 2.0):
        super().__init__()
        self.alpha: Tensor = alpha
        self.gamma: float = gamma

    def forward(self, inputs: Tensor, targets: Tensor) -> Tensor:
        ce_loss: Tensor = F.cross_entropy(inputs, targets, reduction='none')
        pt: Tensor = torch.exp(-ce_loss)
        alpha_t: Tensor = self.alpha.gather(0, targets)
        focal_loss: Tensor = alpha_t * (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean()

In [ ]:
import os
import json
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import confusion_matrix, classification_report, precision_recall_fscore_support
from tqdm import tqdm

# ============================================================
# CONFIG
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")

EPOCHS = 5
LR = 1e-4
WEIGHT_DECAY = 1e-5
GRAD_CLIP = 1.0

SAVE_DIR = "/content/hot_density_checkpoints"
os.makedirs(SAVE_DIR, exist_ok=True)

BEST_VAL_LOSS_PATH = os.path.join(SAVE_DIR, "best_hot_density_val_loss.pth")
BEST_HOT_F1_PATH = os.path.join(SAVE_DIR, "best_hot_density_hot_f1.pth")

# ============================================================
# MODEL: binary hot/cold classifier
# ============================================================

model = KalshiTransformer(
    feature_dim=23,
    d_model=64,
    nhead=4,
    num_layers=2,
    num_classes=2,   # IMPORTANT: binary [0=cold, 1=hot]
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)

# ============================================================
# HARD-NEGATIVE WEIGHTED CROSS ENTROPY
# ============================================================

class HardNegativeCELoss(nn.Module):
    def __init__(
        self,
        pos_weight=6.0,
        toxic_boost=4.0,
        bad_count_scale=10.0,
        max_weight=12.0,
    ):
        super().__init__()
        self.pos_weight = pos_weight
        self.toxic_boost = toxic_boost
        self.bad_count_scale = bad_count_scale
        self.max_weight = max_weight

    def forward(self, logits, targets, bad_counts):
        # Per-sample CE
        ce = torch.nn.functional.cross_entropy(
            logits,
            targets,
            reduction="none",
        )

        targets_f = targets.float()
        bad_counts = bad_counts.float()

        # Map bad_count into roughly [0, 1], clipped.
        bad_strength = torch.clamp(bad_counts / self.bad_count_scale, 0.0, 1.0)

        # Positive examples get base recall pressure.
        # Negative examples get extra weight only if they are high-bad-count.
        weights = torch.where(
            targets == 1,
            torch.full_like(targets_f, self.pos_weight),
            1.0 + self.toxic_boost * bad_strength,
        )

        weights = torch.clamp(weights, 1.0, self.max_weight)

        return (ce * weights).mean()


criterion = HardNegativeCELoss(
    pos_weight=8.0,
    toxic_boost=2.0,
    bad_count_scale=10.0,
    max_weight=12.0,
).to(device)

print("Using HardNegativeCELoss")

# ============================================================
# HELPERS
# ============================================================

def unpack_batch(batch):
    """
    Works whether dataset returns:
        (features, labels)
    or:
        (features, labels, diagnostics)
    """
    if len(batch) == 2:
        batch_features, batch_labels = batch
    elif len(batch) == 3:
        batch_features, batch_labels, _ = batch
    else:
        raise ValueError(f"Unexpected batch length: {len(batch)}")

    return batch_features, batch_labels


def summarize_validation(all_targets, all_preds, all_p_hot):
    all_targets = np.asarray(all_targets)
    all_preds = np.asarray(all_preds)
    all_p_hot = np.asarray(all_p_hot)

    print("\nConfusion Matrix (Rows=true, Cols=pred [0=cold, 1=hot]):")
    cm = confusion_matrix(all_targets, all_preds, labels=[0, 1])
    print(cm)

    print("\nClassification Report:")
    report = classification_report(
        all_targets,
        all_preds,
        labels=[0, 1],
        target_names=["0 cold", "1 hot"],
        zero_division=0,
    )
    print(report)

    precision, recall, f1, support = precision_recall_fscore_support(
        all_targets,
        all_preds,
        labels=[0, 1],
        zero_division=0,
    )

    hot_precision = precision[1]
    hot_recall = recall[1]
    hot_f1 = f1[1]

    print("p_hot quantiles:")
    for q in [0.50, 0.75, 0.90, 0.95, 0.975, 0.99, 0.995, 0.999]:
        print(f"  q={q:>5}: {np.quantile(all_p_hot, q):.6f}")

    print("\nThreshold sweep:")
    for thr in [0.50, 0.60, 0.70, 0.80, 0.90, 0.93, 0.95, 0.97]:
        mask = all_p_hot >= thr
        n_pred = int(mask.sum())

        if n_pred == 0:
            print(f"  p_hot >= {thr:.2f}: n=0")
            continue

        precision_at_thr = float(all_targets[mask].mean())
        recall_at_thr = float(all_targets[mask].sum() / max(1, all_targets.sum()))

        print(
            f"  p_hot >= {thr:.2f}: "
            f"n={n_pred:6d} | precision={precision_at_thr:.4f} | recall={recall_at_thr:.4f}"
        )

    print("\nTop-N ranking audit:")
    order = np.argsort(-all_p_hot)
    total_hot = max(1, int(all_targets.sum()))

    for n in [100, 300, 500, 700, 1000, 1500, 2000, 5000]:
        n = min(n, len(order))
        idx = order[:n]
        hits = int(all_targets[idx].sum())
        precision_at_n = hits / n
        recall_at_n = hits / total_hot
        min_p = float(all_p_hot[idx[-1]])

        print(
            f"  top {n:5d}: "
            f"hits={hits:5d} | precision={precision_at_n:.4f} | "
            f"recall={recall_at_n:.4f} | cutoff_p_hot={min_p:.6f}"
        )

    return {
        "hot_precision_argmax": float(hot_precision),
        "hot_recall_argmax": float(hot_recall),
        "hot_f1_argmax": float(hot_f1),
        "num_pred_hot_argmax": int((all_preds == 1).sum()),
        "num_true_hot": int((all_targets == 1).sum()),
    }


# ============================================================
# TRAIN LOOP
# ============================================================

best_val_loss = float("inf")
best_hot_f1 = -float("inf")
history = []

for epoch in range(1, EPOCHS + 1):
    print("\n" + "=" * 60)
    print(f"EPOCH {epoch} / {EPOCHS}")
    print("=" * 60)

    # -------------------------
    # TRAIN
    # -------------------------
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    train_bar = tqdm(train_loader, desc="Training", leave=False)

    for batch in train_bar:
        batch_features, batch_labels, batch_diag = batch

        batch_features = batch_features.to(device)
        batch_labels = batch_labels.to(device)

        optimizer.zero_grad(set_to_none=True)

        logits = model(batch_features)
        bad_counts = batch_diag["bad_count_0_30"].to(device)
        loss = criterion(logits, batch_labels, bad_counts)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)
        optimizer.step()

        train_loss += loss.item()

        preds = torch.argmax(logits, dim=1)
        train_correct += (preds == batch_labels).sum().item()
        train_total += batch_labels.numel()

        train_bar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "acc": f"{train_correct / max(1, train_total):.4f}",
        })

    avg_train_loss = train_loss / max(1, len(train_loader))
    train_acc = train_correct / max(1, train_total)

    # -------------------------
    # VALIDATE
    # -------------------------
    model.eval()
    val_loss = 0.0

    all_targets = []
    all_preds = []
    all_p_hot = []

    val_bar = tqdm(val_loader, desc="Validating", leave=False)

    with torch.no_grad():
        for batch in val_bar:
            batch_features, batch_labels, batch_diag = batch

            batch_features = batch_features.to(device)
            batch_labels = batch_labels.to(device)

            logits = model(batch_features)
            bad_counts = batch_diag["bad_count_0_30"].to(device)
            loss = criterion(logits, batch_labels, bad_counts)
            val_loss += loss.item()

            probs = torch.softmax(logits, dim=1)
            p_hot = probs[:, 1]
            preds = torch.argmax(probs, dim=1)

            all_targets.extend(batch_labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_p_hot.extend(p_hot.cpu().numpy())

    avg_val_loss = val_loss / max(1, len(val_loader))

    print(f"\nTrain Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss:   {avg_val_loss:.4f}")

    metrics = summarize_validation(all_targets, all_preds, all_p_hot)

    epoch_path = os.path.join(SAVE_DIR, f"hot_density_epoch_{epoch}.pth")
    torch.save(model.state_dict(), epoch_path)
    print(f"\nSaved epoch checkpoint: {epoch_path}")

    if avg_val_loss < best_val_loss:
        print(
            f"[+] Val loss improved from {best_val_loss:.4f} "
            f"to {avg_val_loss:.4f}. Saving: {BEST_VAL_LOSS_PATH}"
        )
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), BEST_VAL_LOSS_PATH)

    if metrics["hot_f1_argmax"] > best_hot_f1:
        print(
            f"[+] Hot F1 improved from {best_hot_f1:.4f} "
            f"to {metrics['hot_f1_argmax']:.4f}. Saving: {BEST_HOT_F1_PATH}"
        )
        best_hot_f1 = metrics["hot_f1_argmax"]
        torch.save(model.state_dict(), BEST_HOT_F1_PATH)

    row = {
        "epoch": epoch,
        "train_loss": float(avg_train_loss),
        "train_acc": float(train_acc),
        "val_loss": float(avg_val_loss),
        **metrics,
    }
    history.append(row)

    with open(os.path.join(SAVE_DIR, "training_history.json"), "w") as f:
        json.dump(history, f, indent=2)

print("\nTraining Complete.")
print(f"Best val-loss checkpoint: {BEST_VAL_LOSS_PATH}")
print(f"Best hot-F1 checkpoint:   {BEST_HOT_F1_PATH}")
print(f"All epoch checkpoints in: {SAVE_DIR}")

The simulator code begins here.  An attempt at a near perfect market simulator to efficiently and accurately test and iterate model architecture and execution logic.  Execution logic is also kept below.

In [ ]:
### STAGE 2 MULTI ORDER SIM ###

import os
import math
import pandas as pd
import numpy as np
from typing import Dict, Optional, Any, Tuple, List


def safe_float(x: Any, default: Optional[float] = np.nan) -> float:
    try:
        if x is None or pd.isna(x):
            return default
        return float(x)
    except Exception:
        return default


def parse_ladder(raw_ladder: Any) -> List[Tuple[float, float]]:
    """
    Parse snapshot ladders stored as nested numpy arrays / lists / tuples / strings.

    Expected normalized shape:
      [
        ['0.0100', '757201.00'],
        ['0.0200', '4223.00'],
        ...
      ]
    """
    if raw_ladder is None:
        return []

    try:
        if isinstance(raw_ladder, float) and pd.isna(raw_ladder):
            return []
    except Exception:
        pass

    ladder_obj = raw_ladder

    # pyarrow scalar support
    if hasattr(ladder_obj, "as_py"):
        try:
            ladder_obj = ladder_obj.as_py()
        except Exception:
            pass

    # top-level numpy arrays / array-likes
    if hasattr(ladder_obj, "tolist") and not isinstance(ladder_obj, (str, bytes)):
        try:
            ladder_obj = ladder_obj.tolist()
        except Exception:
            pass

    # string representation support
    if isinstance(ladder_obj, str):
        s = ladder_obj.strip()
        if s == "" or s.lower() == "none":
            return []
        try:
            ladder_obj = ast.literal_eval(s)
        except Exception:
            return []

    if not isinstance(ladder_obj, (list, tuple)):
        return []

    out: List[Tuple[float, float]] = []
    for item in ladder_obj:
        if hasattr(item, "as_py"):
            try:
                item = item.as_py()
            except Exception:
                pass

        if hasattr(item, "tolist") and not isinstance(item, (str, bytes)):
            try:
                item = item.tolist()
            except Exception:
                pass

        if not isinstance(item, (list, tuple)) or len(item) != 2:
            continue

        try:
            px = float(item[0])
            vol = float(item[1])
        except Exception:
            continue

        if np.isnan(px) or np.isnan(vol) or vol <= 0:
            continue

        out.append((round(px, 2), vol))

    return out


def apply_snapshot_row(row: dict) -> Tuple[Dict[float, float], Dict[float, float]]:
    """
    Build yes-bid / yes-ask books from a snapshot row.

    Observed schema:
      yes_dollars_fp -> YES resting bids
      no_dollars_fp  -> NO resting bids, which imply YES asks via ask = 1 - no_bid
    """
    bids: Dict[float, float] = {}
    asks: Dict[float, float] = {}

    yes_ladder = parse_ladder(row.get("yes_dollars_fp"))
    no_ladder = parse_ladder(row.get("no_dollars_fp"))

    for px, vol in yes_ladder:
        if vol > 1e-5:
            bids[round(px, 2)] = float(vol)

    for no_px, vol in no_ladder:
        yes_ask_px = round(1.0 - float(no_px), 2)
        if vol > 1e-5:
            asks[yes_ask_px] = float(vol)

    return bids, asks


# ============================================================
# MULTI-ORDER MATCHING ENGINE, Q=1
# ============================================================

class MultiOrderKalshiMatchingEngine:
    """
    Q=1 multi-order matching engine.

    Design goals:
      - Preserve the old single-order semantics when only one order is live.
      - Support multiple pending orders using order_id.
      - Allocate finite trade volume across eligible orders, so one trade event
        cannot fill unlimited simulated orders.
      - Do not include our own simulated orders in best bid / ask.
    """

    def __init__(self) -> None:
        self.resting_bids: Dict[float, float] = {}
        self.resting_asks: Dict[float, float] = {}

        self.snapshot_count: int = 0
        self.empty_snapshot_count: int = 0

        self.next_order_id: int = 1
        self.active_orders: Dict[int, Dict[str, Any]] = {}

        # Diagnostics
        self.total_submitted_orders: int = 0
        self.total_canceled_orders: int = 0
        self.total_filled_orders: int = 0
        self.total_trade_events: int = 0

    def get_best_bid(self) -> Optional[float]:
        valid_bids = [px for px, vol in self.resting_bids.items() if vol > 1e-5]
        return max(valid_bids) if valid_bids else None

    def get_best_ask(self) -> Optional[float]:
        valid_asks = [px for px, vol in self.resting_asks.items() if vol > 1e-5]
        return min(valid_asks) if valid_asks else None

    def get_order(self, order_id: Optional[int]) -> Optional[Dict[str, Any]]:
        if order_id is None:
            return None
        return self.active_orders.get(order_id)

    def get_receipt(self, order_id: Optional[int]) -> Optional[Dict[str, Any]]:
        order = self.get_order(order_id)
        if order is None:
            return None
        return order.get("receipt")

    def _pending_orders(self) -> List[Dict[str, Any]]:
        return [
            o for o in self.active_orders.values()
            if o["receipt"]["status"] == "pending"
        ]

    def _refresh_queue_ahead_from_snapshot(self) -> None:
        """
        Snapshot refresh, generalized for many pending orders.

        For orders at the same side/price, later simulated orders sit behind
        earlier simulated orders. External queue ahead comes from the snapshot.
        """
        pending = self._pending_orders()

        groups: Dict[Tuple[str, float], List[Dict[str, Any]]] = {}
        for order in pending:
            key = (order["side"], order["order_price"])
            groups.setdefault(key, []).append(order)

        for (side, px), orders in groups.items():
            book = self.resting_bids if side == "yes" else self.resting_asks
            external_v_ahead = float(book.get(px, 0.0))

            orders.sort(key=lambda o: (o["receipt"]["arrival_ts"], o["order_id"]))

            own_ahead = 0.0
            for order in orders:
                order["v_ahead"] = external_v_ahead + own_ahead
                own_ahead += order["remaining_volume"]

    def process_tick(self, row: dict) -> None:
        event_type = str(row.get("event_type"))

        if event_type not in ["orderbook_snapshot", "orderbook_delta", "trade"]:
            return

        # ---------------------------------------------------------
        # 1. Snapshot
        # ---------------------------------------------------------
        if event_type == "orderbook_snapshot":
            self.snapshot_count += 1

            new_bids, new_asks = apply_snapshot_row(row)

            # Same behavior as old engine: ignore empty terminal snapshots.
            if len(new_bids) == 0 and len(new_asks) == 0:
                self.empty_snapshot_count += 1
                return

            self.resting_bids = new_bids
            self.resting_asks = new_asks
            self._refresh_queue_ahead_from_snapshot()
            return

        # ---------------------------------------------------------
        # 2. Delta
        # ---------------------------------------------------------
        if event_type == "orderbook_delta":
            tick_side = str(row.get("side"))
            price_val = safe_float(row.get("price_dollars"), default=np.nan)
            delta = safe_float(row.get("delta_fp"), default=0.0)

            if pd.isna(price_val) or tick_side not in ["yes", "no"]:
                return

            price = round(price_val, 2) if tick_side == "yes" else round(1.0 - price_val, 2)
            book = self.resting_bids if tick_side == "yes" else self.resting_asks

            new_vol = float(book.get(price, 0.0)) + float(delta)
            if new_vol > 1e-5:
                book[price] = new_vol
            else:
                book.pop(price, None)

            # Preserve old pull heuristic: same side + same price pulls reduce queue ahead.
            if delta < 0:
                pulled = abs(float(delta))
                for order in self._pending_orders():
                    if order["side"] == tick_side and order["order_price"] == price:
                        order["v_ahead"] = max(0.0, order["v_ahead"] - pulled * 0.5)
                        order["receipt"]["pulls_while_waiting"] += pulled

            return

        # ---------------------------------------------------------
        # 3. Trade-driven finite fill allocation
        # ---------------------------------------------------------
        if event_type == "trade":
            raw_price = row.get("yes_price_dollars")
            if raw_price is None or pd.isna(raw_price):
                return

            price = float(raw_price)
            tick_side = str(row.get("taker_side"))
            current_ts = float(row.get("local_recv_ts"))
            traded_total = float(row.get("count_fp", 0.0))

            if traded_total <= 0:
                return

            self.total_trade_events += 1

            eligible: List[Dict[str, Any]] = []

            for order in self._pending_orders():
                if current_ts <= order["receipt"]["arrival_ts"]:
                    continue

                o_side = order["side"]
                if tick_side == o_side:
                    continue

                is_hit = False
                if o_side == "yes" and price <= order["order_price"]:
                    is_hit = True
                elif o_side == "no" and price >= order["order_price"]:
                    is_hit = True

                if is_hit:
                    eligible.append(order)

            if not eligible:
                return

            # Price/time priority approximation.
            # YES bids: higher price first.
            # NO/ask side: lower YES-ask price first.
            def priority_key(order):
                if order["side"] == "yes":
                    price_priority = -order["order_price"]
                else:
                    price_priority = order["order_price"]
                return (price_priority, order["receipt"]["arrival_ts"], order["order_id"])

            eligible.sort(key=priority_key)

            remaining_trade = traded_total

            for order in eligible:
                if remaining_trade <= 1e-9:
                    break

                if order["receipt"]["status"] != "pending":
                    continue

                # Diagnostic: this order saw this much eligible taker flow while waiting.
                order["receipt"]["trades_ahead_of_us"] += traded_total

                # First consume queue ahead.
                if order["v_ahead"] > 1e-5:
                    consumed_ahead = min(order["v_ahead"], remaining_trade)
                    order["v_ahead"] -= consumed_ahead
                    remaining_trade -= consumed_ahead

                if remaining_trade <= 1e-9:
                    break

                # Then fill our Q=1 order if queue ahead is gone.
                if order["v_ahead"] <= 1e-5:
                    fill_qty = min(order["remaining_volume"], remaining_trade)
                    order["remaining_volume"] -= fill_qty
                    remaining_trade -= fill_qty

                    if order["remaining_volume"] <= 1e-5:
                        order["receipt"]["status"] = "filled"
                        order["receipt"]["fill_ts"] = current_ts
                        order["receipt"]["fill_price"] = order["order_price"]
                        order["receipt"]["filled_qty"] = order["my_volume"]
                        self.total_filled_orders += 1

            return

    def submit_limit_order(
        self,
        side: str,
        price: float,
        arrival_ts: float,
        signal_ts: float,
        volume: float = 1.0,
        slot_id: Optional[int] = None,
        role: Optional[str] = None,
    ) -> Optional[int]:
        """
        Places a Q=1 simulated limit order and returns order_id.
        """
        if side not in ["yes", "no"]:
            return None

        order_price = round(float(price), 2)
        book = self.resting_bids if side == "yes" else self.resting_asks
        external_v_ahead = float(book.get(order_price, 0.0))

        # Later own orders at same side/price sit behind earlier own pending orders.
        own_ahead = 0.0
        for order in self._pending_orders():
            if order["side"] == side and order["order_price"] == order_price:
                own_ahead += order["remaining_volume"]

        v_ahead = external_v_ahead + own_ahead

        order_id = self.next_order_id
        self.next_order_id += 1
        self.total_submitted_orders += 1

        receipt = {
            "order_id": order_id,
            "slot_id": slot_id,
            "role": role,
            "side": side,
            "signal_ts": signal_ts,
            "arrival_ts": arrival_ts,
            "target_price": order_price,
            "initial_v_ahead": v_ahead,
            "pulls_while_waiting": 0.0,
            "trades_ahead_of_us": 0.0,
            "fill_ts": None,
            "fill_price": None,
            "filled_qty": 0.0,
            "status": "pending",
        }

        self.active_orders[order_id] = {
            "order_id": order_id,
            "slot_id": slot_id,
            "role": role,
            "side": side,
            "order_price": order_price,
            "my_volume": float(volume),
            "remaining_volume": float(volume),
            "v_ahead": v_ahead,
            "receipt": receipt,
        }

        return order_id

    def cancel_order(self, order_id: Optional[int], cancel_ts: float) -> Optional[Dict[str, Any]]:
        if order_id is None:
            return None

        order = self.active_orders.get(order_id)
        if order is None:
            return None

        receipt = order["receipt"]

        if receipt["status"] == "pending":
            receipt["status"] = "canceled"
            receipt["exit_ts"] = cancel_ts
            self.total_canceled_orders += 1

        return receipt.copy()


# ============================================================
# TRADE SLOT FSM
# ============================================================

class Q1MakerMakerSlot:
    """
    One independent maker-maker trade lifecycle.

    States:
      MAKER_PENDING:
          two orders live, one YES bid and one NO/YES-ask.
      ACTIVE_PEGGING:
          one side filled, chase the other side.
      TERMINAL:
          receipt logged.
    """

    def __init__(
        self,
        slot_id: int,
        signal: dict,
        yes_order_id: int,
        no_order_id: int,
        start_ts: float,
        whipsaw_timeout: float = 30.0,
        peg_cycle_sec: float = 3.0,
        ttl_puke_sec: float = 60.0,
        min_spread_bailout: float = 0.01,
    ):
        self.slot_id = slot_id
        self.signal = signal

        self.yes_order_id = yes_order_id
        self.no_order_id = no_order_id

        self.state = "MAKER_PENDING"
        self.start_ts = float(start_ts)
        self.whipsaw_start_ts = float(start_ts)

        self.whipsaw_timeout = float(whipsaw_timeout)
        self.peg_cycle_sec = float(peg_cycle_sec)
        self.ttl_puke_sec = float(ttl_puke_sec)
        self.min_spread_bailout = float(min_spread_bailout)

        self.filled_leg = None
        self.entry_receipt = None

        self.peg_start_ts = None
        self.last_peg_ts = None

        self.terminal = False
        self.receipt = None

    def update(self, current_ts: float, engine: MultiOrderKalshiMatchingEngine) -> Optional[dict]:
        if self.terminal:
            return None

        if self.state == "MAKER_PENDING":
            return self._update_maker_pending(current_ts, engine)

        if self.state == "ACTIVE_PEGGING":
            return self._update_active_pegging(current_ts, engine)

        return None

    def _update_maker_pending(self, current_ts: float, engine: MultiOrderKalshiMatchingEngine) -> Optional[dict]:
        yes_receipt = engine.get_receipt(self.yes_order_id)
        no_receipt = engine.get_receipt(self.no_order_id)

        yes_filled = yes_receipt is not None and yes_receipt["status"] == "filled"
        no_filled = no_receipt is not None and no_receipt["status"] == "filled"

        # 1. Clean two-sided fill.
        if yes_filled and no_filled:
            pnl = round(float(no_receipt["fill_price"]) - float(yes_receipt["fill_price"]), 2)
            return self._finish_full_whipsaw(yes_receipt, no_receipt, current_ts, pnl)

        # 2. Timeout.
        if current_ts - self.whipsaw_start_ts > self.whipsaw_timeout:
            if not yes_filled and not no_filled:
                engine.cancel_order(self.yes_order_id, current_ts)
                engine.cancel_order(self.no_order_id, current_ts)
                return self._finish_scratch("Scratch - Dead Market, No Fills")

            self.filled_leg = "yes" if yes_filled else "no"
            self.entry_receipt = yes_receipt if yes_filled else no_receipt

            self.state = "ACTIVE_PEGGING"
            self.peg_start_ts = current_ts
            self.last_peg_ts = current_ts - self.peg_cycle_sec

        return None

    def _update_active_pegging(self, current_ts: float, engine: MultiOrderKalshiMatchingEngine) -> Optional[dict]:
        exit_side = "no" if self.filled_leg == "yes" else "yes"
        exit_order_id = self.no_order_id if exit_side == "no" else self.yes_order_id
        exit_receipt = engine.get_receipt(exit_order_id)

        entry_px = float(self.entry_receipt["fill_price"])
        time_in_peg = current_ts - float(self.peg_start_ts)

        # A. Exit order got hit naturally.
        if exit_receipt is not None and exit_receipt["status"] == "filled":
            exit_px = float(exit_receipt["fill_price"])
            pnl = exit_px - entry_px if self.filled_leg == "yes" else entry_px - exit_px
            return self._finish_closed_trade(
                exit_reason="Scratch/Loss - Peg Hit",
                exit_px=round(exit_px, 2),
                exit_ts=current_ts,
                pnl=round(pnl, 2),
                exit_receipt=exit_receipt,
            )

        # B. Hard timeout.
        if time_in_peg > self.ttl_puke_sec:
            engine.cancel_order(exit_order_id, current_ts)
            exit_px, pnl = self._execute_market_exit(engine, entry_px, self.filled_leg)
            return self._finish_closed_trade(
                exit_reason="Loss - 60s Chase Timeout",
                exit_px=exit_px,
                exit_ts=current_ts,
                pnl=pnl,
                exit_receipt=None,
            )

        # C. Peg cycle.
        if current_ts - self.last_peg_ts >= self.peg_cycle_sec:
            self.last_peg_ts = current_ts

            best_bid = engine.get_best_bid()
            best_ask = engine.get_best_ask()

            if best_bid is None or best_ask is None:
                return None

            current_spread = round(float(best_ask) - float(best_bid), 2)

            # Bail out if spread compressed.
            if current_spread <= self.min_spread_bailout:
                engine.cancel_order(exit_order_id, current_ts)
                exit_px, pnl = self._execute_market_exit(engine, entry_px, self.filled_leg)
                return self._finish_closed_trade(
                    exit_reason="Loss/Scratch - 1c Spread Bailout",
                    exit_px=exit_px,
                    exit_ts=current_ts,
                    pnl=pnl,
                    exit_receipt=None,
                )

            current_order = engine.get_order(exit_order_id)
            current_order_px = None
            if current_order is not None and current_order["receipt"]["status"] == "pending":
                current_order_px = current_order["order_price"]

            # Preserve old top-of-book retention / replacement logic.
            if self.filled_leg == "yes":
                # Need to exit on NO side, represented as YES ask.
                if current_order_px is not None and current_order_px <= best_ask:
                    return None

                target_px = round(float(best_ask) - 0.01, 2)
                if target_px <= best_bid:
                    target_px = round(float(best_bid) + 0.01, 2)

                engine.cancel_order(exit_order_id, current_ts)
                new_id = engine.submit_limit_order(
                    side="no",
                    price=target_px,
                    arrival_ts=current_ts,
                    signal_ts=current_ts,
                    volume=1.0,
                    slot_id=self.slot_id,
                    role="peg_exit",
                )
                self.no_order_id = new_id

            else:
                # Filled NO first; need to exit on YES side.
                if current_order_px is not None and current_order_px >= best_bid:
                    return None

                target_px = round(float(best_bid) + 0.01, 2)
                if target_px >= best_ask:
                    target_px = round(float(best_ask) - 0.01, 2)

                engine.cancel_order(exit_order_id, current_ts)
                new_id = engine.submit_limit_order(
                    side="yes",
                    price=target_px,
                    arrival_ts=current_ts,
                    signal_ts=current_ts,
                    volume=1.0,
                    slot_id=self.slot_id,
                    role="peg_exit",
                )
                self.yes_order_id = new_id

        return None

    def _execute_market_exit(self, engine: MultiOrderKalshiMatchingEngine, entry_px: float, filled_leg: str):
        if filled_leg == "yes":
            exit_px = engine.get_best_bid()
            if exit_px is None:
                exit_px = 0.01
            pnl = float(exit_px) - entry_px
        else:
            exit_px = engine.get_best_ask()
            if exit_px is None:
                exit_px = 0.99
            pnl = entry_px - float(exit_px)

        return round(float(exit_px), 2), round(float(pnl), 2)

    def _base_receipt(self, outcome: str, gross_pnl: float) -> dict:
        return {
            "slot_id": self.slot_id,
            "ticker": self.signal["ticker"],
            "signal_side": self.signal.get("predicted_side", "maker"),
            "confidence": self.signal.get("confidence"),
            "signal_ts": self.signal.get("local_recv_ts"),
            "outcome": outcome,
            "gross_pnl": float(gross_pnl),
        }

    def _finish_scratch(self, reason: str) -> dict:
        rec = self._base_receipt(reason, 0.0)
        rec.update({
            "entry_price": None,
            "exit_price": None,
            "yes_order_id": self.yes_order_id,
            "no_order_id": self.no_order_id,
            "yes_fill_ts": None,
            "no_fill_ts": None,
            "first_fill_ts": None,
            "second_fill_ts": None,
            "time_between_fills": None,
            "time_from_signal_to_first_fill": None,
            "time_from_order_to_first_fill": None,
            "same_timestamp_fill": None,
        })
        return self._mark_terminal(rec)

    def _finish_full_whipsaw(self, yes_receipt: dict, no_receipt: dict, exit_ts: float, pnl: float) -> dict:
        yes_fill_ts = yes_receipt.get("fill_ts")
        no_fill_ts = no_receipt.get("fill_ts")

        first_fill_ts = min(yes_fill_ts, no_fill_ts)
        second_fill_ts = max(yes_fill_ts, no_fill_ts)

        signal_ts = self.signal.get("local_recv_ts")
        order_arrival_ts = min(yes_receipt.get("arrival_ts"), no_receipt.get("arrival_ts"))

        time_between_fills = second_fill_ts - first_fill_ts
        time_from_signal_to_first_fill = first_fill_ts - signal_ts if signal_ts is not None else None
        time_from_order_to_first_fill = first_fill_ts - order_arrival_ts if order_arrival_ts is not None else None

        rec = self._base_receipt("Win - Full Whipsaw Arb", pnl)
        rec.update({
            "entry_price": yes_receipt["fill_price"],
            "exit_price": no_receipt["fill_price"],
            "yes_order_id": self.yes_order_id,
            "no_order_id": self.no_order_id,

            "yes_initial_v_ahead": yes_receipt.get("initial_v_ahead"),
            "no_initial_v_ahead": no_receipt.get("initial_v_ahead"),
            "yes_trades_ahead_of_us": yes_receipt.get("trades_ahead_of_us"),
            "no_trades_ahead_of_us": no_receipt.get("trades_ahead_of_us"),
            "yes_pulls_while_waiting": yes_receipt.get("pulls_while_waiting"),
            "no_pulls_while_waiting": no_receipt.get("pulls_while_waiting"),

            "yes_fill_ts": yes_fill_ts,
            "no_fill_ts": no_fill_ts,
            "first_fill_ts": first_fill_ts,
            "second_fill_ts": second_fill_ts,
            "time_between_fills": time_between_fills,
            "time_from_signal_to_first_fill": time_from_signal_to_first_fill,
            "time_from_order_to_first_fill": time_from_order_to_first_fill,
            "same_timestamp_fill": bool(time_between_fills == 0.0),
            "time_in_trade": time_between_fills,
            "exit_ts": exit_ts,
            "legacy_time_from_yes_fill": exit_ts - yes_fill_ts if yes_fill_ts is not None else None,
        })
        return self._mark_terminal(rec)

    def _finish_closed_trade(
        self,
        exit_reason: str,
        exit_px: float,
        exit_ts: float,
        pnl: float,
        exit_receipt: Optional[dict] = None,
    ) -> dict:
        entry_fill_ts = self.entry_receipt.get("fill_ts") if self.entry_receipt is not None else None
        exit_fill_ts = exit_receipt.get("fill_ts") if exit_receipt is not None else exit_ts

        signal_ts = self.signal.get("local_recv_ts")
        order_arrival_ts = self.entry_receipt.get("arrival_ts") if self.entry_receipt is not None else None

        time_in_trade = (
            exit_fill_ts - entry_fill_ts
            if entry_fill_ts is not None and exit_fill_ts is not None
            else None
        )
        time_from_signal_to_first_fill = (
            entry_fill_ts - signal_ts
            if entry_fill_ts is not None and signal_ts is not None
            else None
        )
        time_from_order_to_first_fill = (
            entry_fill_ts - order_arrival_ts
            if entry_fill_ts is not None and order_arrival_ts is not None
            else None
        )

        rec = self._base_receipt(exit_reason, pnl)
        rec.update({
            "entry_price": self.entry_receipt["fill_price"] if self.entry_receipt is not None else None,
            "exit_price": exit_px,
            "yes_order_id": self.yes_order_id,
            "no_order_id": self.no_order_id,
            "filled_leg": self.filled_leg,
            "entry_fill_ts": entry_fill_ts,
            "exit_fill_ts": exit_fill_ts,
            "time_in_trade": time_in_trade,
            "time_from_signal_to_first_fill": time_from_signal_to_first_fill,
            "time_from_order_to_first_fill": time_from_order_to_first_fill,
            "exit_ts": exit_ts,
        })
        return self._mark_terminal(rec)

    def _mark_terminal(self, rec: dict) -> dict:
        self.state = "TERMINAL"
        self.terminal = True
        self.receipt = rec
        return rec


# ============================================================
# MULTI-SLOT EXECUTOR
# ============================================================

class MultiSlotKalshiExecutor:
    """
    Signal-driven multi-slot executor.

    Important semantics:
      - Signals are NOT queued stale like the old single-FSM executor.
      - When a signal is due, we either open a slot now or log why it was skipped.
      - entry_cooldown_sec is per ticker/executor instance.
      - max_active_slots controls concurrent independent Q=1 slots.
    """

    def __init__(
        self,
        predictions_df: pd.DataFrame,
        max_active_slots: int = 1,
        entry_cooldown_sec: float = 2.0,
        min_entry_spread: float = 0.02,
        latency_penalty: float = 0.020,
        whipsaw_timeout: float = 30.0,
        peg_cycle_sec: float = 3.0,
        ttl_puke_sec: float = 60.0,
        min_spread_bailout: float = 0.01,
        log_slot_skips: bool = True,
    ):
        self.predictions = predictions_df.sort_values(by="local_recv_ts").to_dict("records")
        self.pred_idx = 0

        self.max_active_slots = int(max_active_slots)
        self.entry_cooldown_sec = float(entry_cooldown_sec)
        self.min_entry_spread = float(min_entry_spread)
        self.latency_penalty = float(latency_penalty)

        self.whipsaw_timeout = float(whipsaw_timeout)
        self.peg_cycle_sec = float(peg_cycle_sec)
        self.ttl_puke_sec = float(ttl_puke_sec)
        self.min_spread_bailout = float(min_spread_bailout)

        self.log_slot_skips = bool(log_slot_skips)

        self.next_slot_id = 1
        self.active_slots: List[Q1MakerMakerSlot] = []
        self.completed_trades: List[dict] = []

        self.last_entry_ts: Optional[float] = None

    def update(self, row: pd.Series, engine: MultiOrderKalshiMatchingEngine) -> None:
        current_ts = float(row["local_recv_ts"])

        # 1. Update active slots first. This lets fills from engine.process_tick(row)
        # resolve before we decide whether a new slot is available.
        still_active = []
        for slot in self.active_slots:
            rec = slot.update(current_ts, engine)
            if rec is not None:
                self.completed_trades.append(rec)

            if not slot.terminal:
                still_active.append(slot)

        self.active_slots = still_active

        # 2. Process all due signals at this tick. No stale queueing.
        while self.pred_idx < len(self.predictions):
            sig = self.predictions[self.pred_idx]
            signal_ts = float(sig["local_recv_ts"])

            if current_ts < signal_ts + self.latency_penalty:
                break

            self.pred_idx += 1
            self._try_open_slot(sig, current_ts, engine)

    def _try_open_slot(self, signal: dict, current_ts: float, engine: MultiOrderKalshiMatchingEngine) -> None:
        if len(self.active_slots) >= self.max_active_slots:
            self._log_skipped_signal(signal, "Skipped - Max Active Slots")
            return

        if self.last_entry_ts is not None:
            if current_ts - self.last_entry_ts < self.entry_cooldown_sec:
                self._log_skipped_signal(signal, "Skipped - Entry Cooldown")
                return

        best_bid = engine.get_best_bid()
        best_ask = engine.get_best_ask()

        if best_bid is None or best_ask is None:
            self._log_skipped_signal(signal, "Skipped - Vacuum or Spread < 2c")
            return

        spread = round(float(best_ask) - float(best_bid), 2)

        if spread < self.min_entry_spread:
            self._log_skipped_signal(signal, "Skipped - Vacuum or Spread < 2c")
            return

        slot_id = self.next_slot_id
        self.next_slot_id += 1

        yes_order_id = engine.submit_limit_order(
            side="yes",
            price=best_bid,
            arrival_ts=current_ts,
            signal_ts=float(signal["local_recv_ts"]),
            volume=1.0,
            slot_id=slot_id,
            role="entry_yes",
        )
        no_order_id = engine.submit_limit_order(
            side="no",
            price=best_ask,
            arrival_ts=current_ts,
            signal_ts=float(signal["local_recv_ts"]),
            volume=1.0,
            slot_id=slot_id,
            role="entry_no",
        )

        if yes_order_id is None or no_order_id is None:
            self._log_skipped_signal(signal, "Skipped - Order Submit Failed")
            return

        slot = Q1MakerMakerSlot(
            slot_id=slot_id,
            signal=signal,
            yes_order_id=yes_order_id,
            no_order_id=no_order_id,
            start_ts=current_ts,
            whipsaw_timeout=self.whipsaw_timeout,
            peg_cycle_sec=self.peg_cycle_sec,
            ttl_puke_sec=self.ttl_puke_sec,
            min_spread_bailout=self.min_spread_bailout,
        )

        self.active_slots.append(slot)
        self.last_entry_ts = current_ts

    def _log_skipped_signal(self, signal: dict, reason: str) -> None:
        if not self.log_slot_skips:
            return

        self.completed_trades.append({
            "slot_id": None,
            "ticker": signal["ticker"],
            "signal_side": signal.get("predicted_side", "maker"),
            "confidence": signal.get("confidence"),
            "signal_ts": signal.get("local_recv_ts"),
            "outcome": reason,
            "entry_price": None,
            "exit_price": None,
            "gross_pnl": 0.0,
            "yes_fill_ts": None,
            "no_fill_ts": None,
            "first_fill_ts": None,
            "second_fill_ts": None,
            "time_between_fills": None,
            "time_from_signal_to_first_fill": None,
            "time_from_order_to_first_fill": None,
            "same_timestamp_fill": None,
        })


# ============================================================
# MASTER LOOP HELPER
# ============================================================

def run_multislot_zero_assumption_sim(
    predictions_path: str,
    raw_local_dir: str = "/content/kalshi_clean_raw_v2",
    output_audit_path: str = "trade_receipts_multislot.parquet",
    max_active_slots: int = 1,
    entry_cooldown_sec: float = 2.0,
    min_entry_spread: float = 0.02,
    log_slot_skips: bool = True,
) -> pd.DataFrame:
    """
    Drop-in-style master loop for the new multi-order/multi-slot simulator.

    predictions_path should be a parquet with at least:
      ticker
      local_recv_ts
      predicted_side or confidence optional
    """

    print("=== INITIATING MULTI-SLOT ZERO-ASSUMPTION SIMULATOR ===")
    print("Predictions:", predictions_path)
    print("Raw dir:", raw_local_dir)
    print("max_active_slots:", max_active_slots)
    print("entry_cooldown_sec:", entry_cooldown_sec)
    print("min_entry_spread:", min_entry_spread)

    df_preds = pd.read_parquet(predictions_path).copy()
    df_preds = df_preds.sort_values("local_recv_ts").reset_index(drop=True)

    if "predicted_side" not in df_preds.columns:
        df_preds["predicted_side"] = "maker"

    if "confidence" not in df_preds.columns:
        if "p_hot" in df_preds.columns:
            df_preds["confidence"] = df_preds["p_hot"]
        else:
            df_preds["confidence"] = np.nan

    tickers = df_preds["ticker"].unique()
    print(f"Loaded {len(df_preds)} signals across {len(tickers)} unique markets.")

    all_receipts = []

    for ticker in tqdm(tickers, desc="Simulating Markets"):
        ticker_preds = df_preds[df_preds["ticker"] == ticker].copy()

        clean_ticker = ticker.replace("dense_", "").replace(".parquet", "")
        raw_file_path = os.path.join(raw_local_dir, f"clean_raw_{clean_ticker}.parquet")

        if not os.path.exists(raw_file_path):
            print(f"\n[!] Raw data missing for {clean_ticker}. Skipping.")
            continue

        df_raw = pd.read_parquet(raw_file_path)

        engine = MultiOrderKalshiMatchingEngine()
        executor = MultiSlotKalshiExecutor(
            predictions_df=ticker_preds,
            max_active_slots=max_active_slots,
            entry_cooldown_sec=entry_cooldown_sec,
            min_entry_spread=min_entry_spread,
            log_slot_skips=log_slot_skips,
        )

        raw_records = df_raw.to_dict("records")

        for row in raw_records:
            engine.process_tick(row)
            executor.update(row, engine)

        all_receipts.extend(executor.completed_trades)

    df_audit = pd.DataFrame(all_receipts)

    print("\n=== MULTI-SLOT SIMULATION COMPLETE ===")

    if not df_audit.empty:
        df_audit.to_parquet(output_audit_path, index=False)

        print(f"\nSaved receipts: {output_audit_path}")
        print(f"Total Logged Rows: {len(df_audit)}")

        print("\n--- OUTCOME DISTRIBUTION ---")
        print(df_audit["outcome"].value_counts())

        print("\n--- PNL SUMMARY ---")
        gross_pnl = df_audit["gross_pnl"].sum()
        print(f"Total Gross PnL: ${gross_pnl:.2f}")

        actionable = df_audit[~df_audit["outcome"].str.contains("Skipped", na=False)]
        if not actionable.empty:
            print(f"\nActionable trades: {len(actionable)}")
            print(f"Gross PnL / actionable trade: ${actionable['gross_pnl'].mean():.4f}")
        else:
            print("\nNo actionable trades.")

    else:
        print("\n[!] No trades were logged.")

    return df_audit

In [ ]:
df_audit_ms = run_multislot_zero_assumption_sim(
    predictions_path="/content/hot_density_signal_files/predictions_hot_epoch1_top400.parquet",
    raw_local_dir="/content/kalshi_clean_raw_v2",
    output_audit_path="/content/trade_receipts_multislot_top400_slots1_cd2.parquet",
    max_active_slots=1,
    entry_cooldown_sec=1.0,
    min_entry_spread=0.02,
    log_slot_skips=True,
)

In [ ]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm

# ============================================================
# 1. RECEIPT / ACCOUNTING AUDIT
# ============================================================

def audit_receipts(receipts_path_or_df, name="run"):
    if isinstance(receipts_path_or_df, str):
        df = pd.read_parquet(receipts_path_or_df).copy()
    else:
        df = receipts_path_or_df.copy()

    print("\n" + "=" * 90)
    print(f"AUDIT: {name}")
    print("=" * 90)

    if df.empty:
        print("Empty receipt dataframe.")
        return df

    df["gross_pnl"] = pd.to_numeric(df["gross_pnl"], errors="coerce").fillna(0.0)
    df["is_skipped"] = df["outcome"].astype(str).str.contains("Skipped", na=False)
    df["is_win"] = df["outcome"].eq("Win - Full Whipsaw Arb")
    df["is_rescue"] = df["outcome"].astype(str).str.contains("Bailout|Peg Hit|Timeout", na=False)
    df["is_dead"] = df["outcome"].astype(str).str.contains("Dead Market", na=False)
    df["is_actionable"] = ~df["is_skipped"]

    print("\nRows:", len(df))
    print("Actionable:", int(df["is_actionable"].sum()))
    print("Skipped:", int(df["is_skipped"].sum()))
    print("Gross PnL:", round(df["gross_pnl"].sum(), 4))
    if df["is_actionable"].sum() > 0:
        print("Gross/actionable:", round(df.loc[df["is_actionable"], "gross_pnl"].mean(), 5))

    print("\nOutcome counts:")
    print(df["outcome"].value_counts().to_string())

    print("\nComponent PnL:")
    comp = df.groupby("outcome")["gross_pnl"].agg(["count", "sum", "mean"]).sort_values("sum", ascending=False)
    print(comp.to_string())

    print("\nBy ticker:")
    by_ticker = (
        df.groupby("ticker")
        .agg(
            rows=("outcome", "size"),
            actionable=("is_actionable", "sum"),
            wins=("is_win", "sum"),
            rescue=("is_rescue", "sum"),
            skipped=("is_skipped", "sum"),
            gross_pnl=("gross_pnl", "sum"),
            gross_per_actionable=("gross_pnl", lambda x: np.nan),
        )
        .reset_index()
    )
    by_ticker["gross_per_actionable"] = by_ticker["gross_pnl"] / by_ticker["actionable"].replace(0, np.nan)
    by_ticker = by_ticker.sort_values("gross_pnl", ascending=False)
    print(by_ticker.head(20).to_string(index=False))

    # ------------------------------------------------------------
    # Basic impossible-value checks
    # ------------------------------------------------------------
    issues = []

    for col in ["entry_price", "exit_price"]:
        if col in df.columns:
            vals = pd.to_numeric(df[col], errors="coerce")
            bad = df[vals.notna() & ((vals < 0.0) | (vals > 1.0))]
            if len(bad):
                issues.append((f"{col} outside [0,1]", len(bad)))

    time_cols = [
        "time_between_fills",
        "time_from_signal_to_first_fill",
        "time_from_order_to_first_fill",
        "time_in_trade",
    ]
    for col in time_cols:
        if col in df.columns:
            vals = pd.to_numeric(df[col], errors="coerce")
            bad = df[vals.notna() & (vals < -1e-9)]
            if len(bad):
                issues.append((f"{col} negative", len(bad)))

    # Clean whipsaw PnL check.
    if {"entry_price", "exit_price", "gross_pnl"}.issubset(df.columns):
        wins = df[df["is_win"]].copy()
        if not wins.empty:
            expected = (
                pd.to_numeric(wins["exit_price"], errors="coerce")
                - pd.to_numeric(wins["entry_price"], errors="coerce")
            ).round(2)
            actual = pd.to_numeric(wins["gross_pnl"], errors="coerce").round(2)
            mismatch = wins[(expected - actual).abs() > 1e-9]
            if len(mismatch):
                issues.append(("Win PnL != exit_price - entry_price", len(mismatch)))

    # New executor slot sanity.
    # New executor slot/order sanity with per-ticker IDs.
    if "slot_id" in df.columns:
        action = df[df["is_actionable"]].copy()
        action = action[action["slot_id"].notna()].copy()

        if not action.empty:
            dup_slot = action.duplicated(["ticker", "slot_id"]).sum()
            if dup_slot:
                issues.append(("Duplicate actionable terminal (ticker, slot_id)", int(dup_slot)))

    for oid_col in ["yes_order_id", "no_order_id"]:
        if oid_col in df.columns:
            action = df[df["is_actionable"]].copy()
            action = action[action[oid_col].notna()].copy()

            if not action.empty:
                dup_oid = action.duplicated(["ticker", oid_col]).sum()
                if dup_oid:
                    issues.append((f"Duplicate terminal (ticker, {oid_col})", int(dup_oid)))

    print("\nSanity issues:")
    if issues:
        for issue, n in issues:
            print(f"  [!] {issue}: {n}")
    else:
        print("  None found by basic audit.")

    # ------------------------------------------------------------
    # Fill timing buckets
    # ------------------------------------------------------------
    if "time_from_signal_to_first_fill" in df.columns:
        tmp = df[df["is_actionable"]].copy()
        vals = pd.to_numeric(tmp["time_from_signal_to_first_fill"], errors="coerce")
        tmp["signal_to_fill_bucket"] = pd.cut(
            vals,
            bins=[-np.inf, 0, 2, 5, 10, 20, 30, 60, np.inf],
            labels=["<=0", "0-2", "2-5", "5-10", "10-20", "20-30", "30-60", ">60"],
        )
        age = (
            tmp.groupby("signal_to_fill_bucket", observed=False)
            .agg(
                n=("outcome", "size"),
                wins=("is_win", "sum"),
                rescue=("is_rescue", "sum"),
                gross_pnl=("gross_pnl", "sum"),
                mean_pnl=("gross_pnl", "mean"),
            )
            .reset_index()
        )
        print("\nPnL by signal-to-first-fill bucket:")
        print(age.to_string(index=False))

    # ------------------------------------------------------------
    # Confidence buckets
    # ------------------------------------------------------------
    if "confidence" in df.columns:
        tmp = df[df["is_actionable"]].copy()
        conf = pd.to_numeric(tmp["confidence"], errors="coerce")
        if conf.notna().sum() > 0:
            tmp["confidence_bucket"] = pd.qcut(conf.rank(method="first"), q=min(5, len(tmp)), duplicates="drop")
            cb = (
                tmp.groupby("confidence_bucket", observed=False)
                .agg(
                    n=("outcome", "size"),
                    wins=("is_win", "sum"),
                    rescue=("is_rescue", "sum"),
                    conf_min=("confidence", "min"),
                    conf_max=("confidence", "max"),
                    gross_pnl=("gross_pnl", "sum"),
                    mean_pnl=("gross_pnl", "mean"),
                )
                .reset_index()
            )
            print("\nActionable PnL by confidence quintile:")
            print(cb.to_string(index=False))

    return df


# ============================================================
# 2. RANDOM SPREAD BASELINE SIGNAL GENERATOR
# ============================================================

def generate_random_spread_signals_from_dense(
    dense_files,
    n_signals=100,
    min_spread=0.02,
    cooldown_sec=2.0,
    seed=0,
    output_path="/content/random_spread_signals.parquet",
):
    """
    Generates random maker signals from dense files at moments where current spread >= min_spread.

    This ignores Stage 1 entirely. It answers:
      "If I randomly trade spreaded moments, does the simulator itself print money?"
    """

    rng = np.random.default_rng(seed)
    candidates = []

    for f in tqdm(dense_files, desc="Scanning dense files for random spread candidates"):
        df = pd.read_parquet(f)
        base = os.path.basename(f)

        required = {"local_recv_ts", "bid_px_1", "ask_px_1"}
        missing = required - set(df.columns)
        if missing:
            print(f"[skip] {base} missing {missing}")
            continue

        tmp = df[["local_recv_ts", "bid_px_1", "ask_px_1"]].copy()
        tmp["spread"] = pd.to_numeric(tmp["ask_px_1"], errors="coerce") - pd.to_numeric(tmp["bid_px_1"], errors="coerce")
        tmp = tmp[np.isfinite(tmp["spread"]) & (tmp["spread"] >= min_spread)].copy()

        if tmp.empty:
            continue

        tmp["ticker"] = base
        candidates.append(tmp[["ticker", "local_recv_ts", "spread"]])

    if not candidates:
        raise ValueError("No spread candidates found.")

    cand = pd.concat(candidates, ignore_index=True)
    cand = cand.sort_values("local_recv_ts").reset_index(drop=True)

    # Shuffle candidate order, then greedily enforce per-ticker cooldown.
    perm = rng.permutation(len(cand))
    chosen = []
    last_ts_by_ticker = {}

    for idx in perm:
        row = cand.iloc[int(idx)]
        ticker = row["ticker"]
        ts = float(row["local_recv_ts"])

        last = last_ts_by_ticker.get(ticker)
        if last is not None and abs(ts - last) < cooldown_sec:
            continue

        chosen.append({
            "ticker": ticker,
            "local_recv_ts": ts,
            "ts": ts,
            "ts_exchange": ts,
            "predicted_side": "maker",
            "confidence": float(row["spread"]),   # use spread as dummy confidence
            "random_baseline": True,
            "spread_at_signal": float(row["spread"]),
        })
        last_ts_by_ticker[ticker] = ts

        if len(chosen) >= n_signals:
            break

    out = pd.DataFrame(chosen).sort_values("local_recv_ts").reset_index(drop=True)

    if len(out) < n_signals:
        print(f"[WARN] Only generated {len(out)} signals, requested {n_signals}")

    out.to_parquet(output_path, index=False)

    print("\nRandom spread baseline signals saved:", output_path)
    print("Signals:", len(out))
    print("Unique tickers:", out["ticker"].nunique())
    print("Spread quantiles:")
    print(out["spread_at_signal"].quantile([0.0, 0.25, 0.5, 0.75, 0.9, 0.99, 1.0]))

    return out


def run_random_spread_baseline_repeats(
    dense_files,
    raw_local_dir="/content/kalshi_clean_raw_v2",
    n_signals=100,
    min_spread=0.02,
    signal_cooldown_sec=2.0,
    executor_cooldown_sec=1.0,
    max_active_slots=1,
    n_repeats=10,
    output_dir="/content/random_spread_baselines",
):
    os.makedirs(output_dir, exist_ok=True)

    rows = []

    for seed in range(n_repeats):
        sig_path = os.path.join(
            output_dir,
            f"random_spread_n{n_signals}_seed{seed}.parquet"
        )
        rec_path = os.path.join(
            output_dir,
            f"receipts_random_spread_n{n_signals}_seed{seed}.parquet"
        )

        generate_random_spread_signals_from_dense(
            dense_files=dense_files,
            n_signals=n_signals,
            min_spread=min_spread,
            cooldown_sec=signal_cooldown_sec,
            seed=seed,
            output_path=sig_path,
        )

        df_rec = run_multislot_zero_assumption_sim(
            predictions_path=sig_path,
            raw_local_dir=raw_local_dir,
            output_audit_path=rec_path,
            max_active_slots=max_active_slots,
            entry_cooldown_sec=executor_cooldown_sec,
            min_entry_spread=min_spread,
            log_slot_skips=True,
        )

        actionable = df_rec[~df_rec["outcome"].astype(str).str.contains("Skipped", na=False)]
        rows.append({
            "seed": seed,
            "signals": n_signals,
            "logged_rows": len(df_rec),
            "actionable": len(actionable),
            "gross_pnl": df_rec["gross_pnl"].sum() if not df_rec.empty else 0.0,
            "gross_per_actionable": actionable["gross_pnl"].mean() if len(actionable) else np.nan,
            "wins": int((df_rec["outcome"] == "Win - Full Whipsaw Arb").sum()) if not df_rec.empty else 0,
            "rescue_or_loss": int(df_rec["outcome"].astype(str).str.contains("Bailout|Peg Hit|Timeout", na=False).sum()) if not df_rec.empty else 0,
        })

    summary = pd.DataFrame(rows)
    summary_path = os.path.join(output_dir, f"summary_random_spread_n{n_signals}.csv")
    summary.to_csv(summary_path, index=False)

    print("\n" + "=" * 90)
    print("RANDOM SPREAD BASELINE SUMMARY")
    print("=" * 90)
    print(summary.to_string(index=False))
    print("\nAverages:")
    print(summary[["actionable", "gross_pnl", "gross_per_actionable", "wins", "rescue_or_loss"]].mean(numeric_only=True))
    print("\nSaved:", summary_path)

    return summary


# ============================================================
# 3. ISOLATED-SIGNAL TEST GENERATOR
# ============================================================

def make_isolated_signal_file(
    predictions_path,
    min_gap_sec=90.0,
    output_path="/content/isolated_signals.parquet",
):
    """
    Keeps only signals spaced at least min_gap_sec apart per ticker.
    Useful because old and new max_slots=1 should behave much closer
    on isolated signals. Large divergence here is a bug warning.
    """

    df = pd.read_parquet(predictions_path).copy()
    df = df.sort_values(["ticker", "local_recv_ts"]).reset_index(drop=True)

    kept = []
    last_ts = {}

    for row in df.to_dict("records"):
        ticker = row["ticker"]
        ts = float(row["local_recv_ts"])

        if ticker not in last_ts or ts - last_ts[ticker] >= min_gap_sec:
            kept.append(row)
            last_ts[ticker] = ts

    out = pd.DataFrame(kept).sort_values("local_recv_ts").reset_index(drop=True)
    out.to_parquet(output_path, index=False)

    print("Isolated signal file saved:", output_path)
    print("Original rows:", len(df))
    print("Isolated rows:", len(out))
    print("Unique tickers:", out["ticker"].nunique())

    return out

In [ ]:
old_receipts = audit_receipts(
    "/content/hot_density_signal_files/trade_receipts_top400.parquet",
    name="OLD FIFO top400",
)

new_receipts = audit_receipts(
    "/content/trade_receipts_multislot_top400_slots1_cd2.parquet",
    name="NEW fresh-slot top400 max_slots=1 cd=1",
)

In [ ]:
random_summary = run_random_spread_baseline_repeats(
    dense_files=val_files,
    raw_local_dir="/content/kalshi_clean_raw_v2",
    n_signals=400,              # match your top400 test
    min_spread=0.02,
    signal_cooldown_sec=2.0,
    executor_cooldown_sec=1.0,
    max_active_slots=1,
    n_repeats=10,
    output_dir="/content/random_spread_baselines_apr26",
)

In [ ]:
isolated = make_isolated_signal_file(
    predictions_path="/content/hot_density_signal_files/predictions_hot_epoch1_top400.parquet",
    min_gap_sec=90.0,
    output_path="/content/hot_density_signal_files/predictions_top400_isolated_gap90.parquet",
)

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

OUT_DIR = "/content/hot_density_signal_files"
os.makedirs(OUT_DIR, exist_ok=True)

CHECKPOINT = "/content/hot_density_checkpoints/hot_density_epoch_1.pth"
TOP_N_LIST = [100, 400, 700]

SIM_FILES = val_files  # IMPORTANT: set this to the exact APR25 files used for the $1.28 baseline


def load_hot_model(checkpoint_path):
    model = KalshiTransformer(
        feature_dim=23,
        d_model=64,
        nhead=4,
        num_layers=2,
        num_classes=2,
    ).to(device)

    try:
        state = torch.load(checkpoint_path, map_location=device, weights_only=True)
    except TypeError:
        state = torch.load(checkpoint_path, map_location=device)

    model.load_state_dict(state)
    model.eval()
    return model


def generate_hot_outputs(checkpoint_path, file_paths):
    model = load_hot_model(checkpoint_path)

    infer_dataset = KalshiHotDensityDataset(
        file_paths=file_paths,
        seq_len=60,
        candidate_lookahead=30,
        fill_lookahead=30,
        stride=1,
        k_hot=5,
        min_spread=0.02,
        inference_mode=True,
    )

    infer_loader = DataLoader(
        infer_dataset,
        batch_size=256,
        shuffle=False,
        num_workers=2,
        pin_memory=True,
    )

    rows = []

    with torch.no_grad():
        for tickers, ts_exchange_batch, ts_recv_batch, x_batch in tqdm(infer_loader, desc="Hot-density inference"):
            x_batch = x_batch.to(device)
            logits = model(x_batch)
            probs = torch.softmax(logits, dim=1)
            p_hot = probs[:, 1].cpu().numpy()

            ts_exchange_np = (
                ts_exchange_batch.detach().cpu().numpy()
                if torch.is_tensor(ts_exchange_batch)
                else np.asarray(ts_exchange_batch)
            )
            ts_recv_np = (
                ts_recv_batch.detach().cpu().numpy()
                if torch.is_tensor(ts_recv_batch)
                else np.asarray(ts_recv_batch)
            )

            for i, ticker in enumerate(tickers):
                rows.append({
                    "ticker": str(ticker),
                    "ts_exchange": float(ts_exchange_np[i]),
                    "local_recv_ts": float(ts_recv_np[i]),
                    "p_hot": float(p_hot[i]),
                })

    df = pd.DataFrame(rows).sort_values("local_recv_ts").reset_index(drop=True)

    print("\np_hot quantiles:")
    print(df["p_hot"].quantile([0.50, 0.75, 0.90, 0.95, 0.975, 0.99, 0.995, 0.999]))

    return df


def write_top_n_prediction_files(outputs_df, top_n_list):
    paths = []

    for top_n in top_n_list:
        sigs = (
            outputs_df
            .sort_values("p_hot", ascending=False)
            .head(top_n)
            .copy()
            .sort_values("local_recv_ts")
            .reset_index(drop=True)
        )

        sigs["predicted_side"] = "maker"
        sigs["confidence"] = sigs["p_hot"]
        sigs["ts"] = sigs["ts_exchange"]

        # Match your existing simulator prediction schema.
        sigs = sigs[
            [
                "ticker",
                "ts",
                "ts_exchange",
                "local_recv_ts",
                "predicted_side",
                "confidence",
                "p_hot",
            ]
        ]

        path = os.path.join(OUT_DIR, f"predictions_hot_epoch1_top{top_n}.parquet")
        sigs.to_parquet(path, index=False)
        paths.append(path)

        print(f"\nSaved top-{top_n}: {path}")
        print(f"  rows: {len(sigs)}")
        print(f"  unique tickers: {sigs['ticker'].nunique()}")
        print(f"  p_hot cutoff: {sigs['p_hot'].min():.6f}")

    return paths


hot_outputs = generate_hot_outputs(CHECKPOINT, SIM_FILES)
prediction_paths = write_top_n_prediction_files(hot_outputs, TOP_N_LIST)

print("\nPrediction files ready:")
for p in prediction_paths:
    print(p)

In [ ]:
import ast
import pandas as pd
import numpy as np
from typing import Dict, Optional, Any, Tuple, List


def safe_float(x: Any, default: Optional[float] = np.nan) -> float:
    try:
        if x is None or pd.isna(x):
            return default
        return float(x)
    except Exception:
        return default


def parse_ladder(raw_ladder: Any) -> List[Tuple[float, float]]:
    """
    Parse snapshot ladders stored as nested numpy arrays / lists / tuples / strings.

    Expected normalized shape:
      [
        ['0.0100', '757201.00'],
        ['0.0200', '4223.00'],
        ...
      ]
    """
    if raw_ladder is None:
        return []

    try:
        if isinstance(raw_ladder, float) and pd.isna(raw_ladder):
            return []
    except Exception:
        pass

    ladder_obj = raw_ladder

    # pyarrow scalar support
    if hasattr(ladder_obj, "as_py"):
        try:
            ladder_obj = ladder_obj.as_py()
        except Exception:
            pass

    # top-level numpy arrays / array-likes
    if hasattr(ladder_obj, "tolist") and not isinstance(ladder_obj, (str, bytes)):
        try:
            ladder_obj = ladder_obj.tolist()
        except Exception:
            pass

    # string representation support
    if isinstance(ladder_obj, str):
        s = ladder_obj.strip()
        if s == "" or s.lower() == "none":
            return []
        try:
            ladder_obj = ast.literal_eval(s)
        except Exception:
            return []

    if not isinstance(ladder_obj, (list, tuple)):
        return []

    out: List[Tuple[float, float]] = []
    for item in ladder_obj:
        if hasattr(item, "as_py"):
            try:
                item = item.as_py()
            except Exception:
                pass

        if hasattr(item, "tolist") and not isinstance(item, (str, bytes)):
            try:
                item = item.tolist()
            except Exception:
                pass

        if not isinstance(item, (list, tuple)) or len(item) != 2:
            continue

        try:
            px = float(item[0])
            vol = float(item[1])
        except Exception:
            continue

        if np.isnan(px) or np.isnan(vol) or vol <= 0:
            continue

        out.append((round(px, 2), vol))

    return out


def apply_snapshot_row(row: dict) -> Tuple[Dict[float, float], Dict[float, float]]:
    """
    Build yes-bid / yes-ask books from a snapshot row.

    Observed schema:
      yes_dollars_fp -> YES resting bids
      no_dollars_fp  -> NO resting bids, which imply YES asks via ask = 1 - no_bid
    """
    bids: Dict[float, float] = {}
    asks: Dict[float, float] = {}

    yes_ladder = parse_ladder(row.get("yes_dollars_fp"))
    no_ladder = parse_ladder(row.get("no_dollars_fp"))

    for px, vol in yes_ladder:
        if vol > 1e-5:
            bids[round(px, 2)] = float(vol)

    for no_px, vol in no_ladder:
        yes_ask_px = round(1.0 - float(no_px), 2)
        if vol > 1e-5:
            asks[yes_ask_px] = float(vol)

    return bids, asks


class KalshiMatchingEngine:
    def __init__(self) -> None:
        self.resting_bids: Dict[float, float] = {}
        self.resting_asks: Dict[float, float] = {}

        # Diagnostics only
        self.snapshot_count: int = 0
        self.empty_snapshot_count: int = 0

        # MAKER UPGRADE
        self.active_orders: Dict[str, Optional[Dict[str, Any]]] = {
            'yes': None,  # Our resting Bid
            'no': None    # Our resting Ask
        }


    def get_best_bid(self) -> Optional[float]:
        valid_bids = [px for px, vol in self.resting_bids.items() if vol > 1e-5]
        best_raw = max(valid_bids) if valid_bids else None
        return best_raw


    def get_best_ask(self) -> Optional[float]:
        valid_asks = [px for px, vol in self.resting_asks.items() if vol > 1e-5]
        best_raw = min(valid_asks) if valid_asks else None
        return best_raw

    """
    def get_best_bid(self) -> Optional[float]:
        valid_bids = [px for px, vol in self.resting_bids.items() if vol > 1e-5]
        best_raw = max(valid_bids) if valid_bids else None

        # Keep your current optimistic assumption for now
        my_order = self.active_orders.get('yes')
        if my_order and my_order['receipt']['status'] == 'pending':
            my_px = my_order['order_price']
            return max(best_raw, my_px) if best_raw is not None else my_px
        return best_raw

    def get_best_ask(self) -> Optional[float]:
        valid_asks = [px for px, vol in self.resting_asks.items() if vol > 1e-5]
        best_raw = min(valid_asks) if valid_asks else None

        # Keep your current optimistic assumption for now
        my_order = self.active_orders.get('no')
        if my_order and my_order['receipt']['status'] == 'pending':
            my_px = my_order['order_price']
            return min(best_raw, my_px) if best_raw is not None else my_px
        return best_raw
        """

    def _refresh_queue_ahead_from_snapshot(self) -> None:
        """
        After a non-empty snapshot reset, refresh queue-ahead estimates
        for any still-pending orders from the authoritative external book.
        """
        yes_order = self.active_orders.get('yes')
        if yes_order is not None and yes_order['receipt']['status'] == 'pending':
            px = yes_order['order_price']
            yes_order['v_ahead'] = self.resting_bids.get(px, 0.0)

        no_order = self.active_orders.get('no')
        if no_order is not None and no_order['receipt']['status'] == 'pending':
            px = no_order['order_price']
            no_order['v_ahead'] = self.resting_asks.get(px, 0.0)

    def process_tick(self, row: dict) -> None:
        """
        The master method called on every tick to maintain the universe.

        Important semantics:
        - orderbook_snapshot: full-ladder reset (unless empty, then ignore)
        - orderbook_delta: single-level liquidity change
        - trade: taker volume crossing the spread
        """
        event_type: str = str(row.get('event_type'))

        if event_type not in ['orderbook_snapshot', 'orderbook_delta', 'trade']:
            return

        # ---------------------------------------------------------
        # 1. Maintain the LOB
        # ---------------------------------------------------------
        if event_type == 'orderbook_snapshot':
            self.snapshot_count += 1

            new_bids, new_asks = apply_snapshot_row(row)

            # Ignore terminal / empty snapshots instead of wiping the book
            if len(new_bids) == 0 and len(new_asks) == 0:
                self.empty_snapshot_count += 1
                return

            self.resting_bids = new_bids
            self.resting_asks = new_asks

            # Refresh queue-ahead estimates from the authoritative snapshot
            self._refresh_queue_ahead_from_snapshot()
            return

        if event_type == 'orderbook_delta':
            tick_side: str = str(row.get('side'))
            price_val = safe_float(row.get('price_dollars'), default=np.nan)
            delta = safe_float(row.get('delta_fp'), default=0.0)

            if pd.isna(price_val) or tick_side not in ['yes', 'no']:
                return

            # YES book directly, NO book becomes YES ask via 1 - price
            price: float = round(price_val, 2) if tick_side == 'yes' else round(1.0 - price_val, 2)
            book = self.resting_bids if tick_side == 'yes' else self.resting_asks

            new_vol: float = book.get(price, 0.0) + delta
            if new_vol > 1e-5:
                book[price] = new_vol
            else:
                book.pop(price, None)

            # Queue Physics: Pulls (DELTA ONLY, not snapshots)
            for o_side, order in self.active_orders.items():
                if order is None or order['receipt']['status'] != 'pending':
                    continue

                if tick_side == o_side and price == order['order_price']:
                    if delta < 0:
                        pulled: float = abs(delta)
                        order['v_ahead'] = max(0.0, order['v_ahead'] - (pulled * 0.5))
                        order['receipt']['pulls_while_waiting'] += pulled

            return

        # ---------------------------------------------------------
        # 2. Trade-driven queue / fill logic
        # ---------------------------------------------------------
        if event_type == 'trade':
            raw_price = row.get('yes_price_dollars')
            if raw_price is None or pd.isna(raw_price):
                return

            price: float = float(raw_price)
            tick_side: str = str(row.get('taker_side'))

            for o_side, order in self.active_orders.items():
                if order is None or order['receipt']['status'] != 'pending':
                    continue

                if float(row['local_recv_ts']) <= order['receipt']['arrival_ts']:
                    continue

                traded: float = float(row.get('count_fp', 0.0))

                # Opposite-side taker flow can hit our resting order
                if tick_side != o_side:
                    is_hit = False
                    if o_side == 'yes' and price <= order['order_price']:
                        is_hit = True
                    elif o_side == 'no' and price >= order['order_price']:
                        is_hit = True

                    if is_hit:
                        order['receipt']['trades_ahead_of_us'] += traded

                        # A. Taker volume must eat the queue ahead of us first
                        if order['v_ahead'] > 0:
                            consumed = min(order['v_ahead'], traded)
                            order['v_ahead'] -= consumed
                            traded -= consumed

                        # B. If we are at the front, the remaining taker volume eats our order
                        if order['v_ahead'] <= 1e-5 and traded > 0:
                            order['my_volume'] -= traded
                            if order['my_volume'] <= 1e-5:
                                order['receipt']['status'] = 'filled'
                                order['receipt']['fill_ts'] = float(row.get('local_recv_ts'))
                                order['receipt']['fill_price'] = order['order_price']

    def submit_limit_order(
        self,
        side: str,
        price: float,
        arrival_ts: float,
        signal_ts: float,
        volume: float = 1.0
    ) -> None:
        """Places an independent order into the YES or NO queue."""
        if side not in ['yes', 'no']:
            return

        order_price = round(price, 2)
        book = self.resting_bids if side == 'yes' else self.resting_asks
        v_ahead = book.get(order_price, 0.0)

        self.active_orders[side] = {
            'order_price': order_price,
            'my_volume': volume,
            'v_ahead': v_ahead,
            'receipt': {
                'signal_ts': signal_ts,
                'arrival_ts': arrival_ts,
                'target_price': order_price,
                'initial_v_ahead': v_ahead,
                'pulls_while_waiting': 0.0,
                'trades_ahead_of_us': 0.0,
                'fill_ts': None,
                'status': 'pending'
            }
        }

    def cancel_active_order(self, side: str, cancel_ts: float) -> Optional[Dict[str, Any]]:
        """Cancels an order on a specific side and returns its receipt."""
        order = self.active_orders.get(side)

        if order is not None and order['receipt']['status'] == 'pending':
            order['receipt']['status'] = 'canceled'
            order['receipt']['exit_ts'] = cancel_ts

            final_receipt = order['receipt'].copy()
            self.active_orders[side] = None
            return final_receipt

        return None

In [ ]:
#newer
class KalshiExecutor:
    def __init__(self, predictions_df: pd.DataFrame):
        # Load and sort predictions chronologically
        self.predictions = predictions_df.sort_values(by='local_recv_ts').to_dict('records')
        self.pred_idx = 0

        # FSM State
        self.state = 'WAITING_FOR_SIGNAL'
        self.active_signal = None
        self.filled_leg = None
        self.entry_receipt = None

        # --- MAKER-MAKER HYPERPARAMETERS ---
        self.LATENCY_PENALTY = 0.020  # 20 milliseconds
        self.WHIPSAW_TIMEOUT = 30.0   # Time to wait for both legs to fill

        # Active Inventory Management (The Chase)
        self.PEG_CYCLE_SEC = 3.0
        self.TTL_PUKE_SEC = 60.0
        self.MIN_SPREAD_BAILOUT = 0.01

        self.whipsaw_start_ts = None
        self.peg_start_ts = None
        self.last_peg_ts = None

        # Final Audit Log
        self.completed_trades = []

    def update(self, row: pd.Series, engine: KalshiMatchingEngine) -> None:
        current_ts = float(row['local_recv_ts'])

        # ---------------------------------------------------------
        # STATE 1: WAITING FOR SIGNAL
        # ---------------------------------------------------------
        if self.state == 'WAITING_FOR_SIGNAL':
            if self.pred_idx >= len(self.predictions):
                return

            next_signal = self.predictions[self.pred_idx]

            if current_ts >= next_signal['local_recv_ts'] + self.LATENCY_PENALTY:
                self.active_signal = next_signal
                self.pred_idx += 1

                best_bid = engine.get_best_bid()
                best_ask = engine.get_best_ask()

                # Require a 2c spread so the Arb is actually profitable after fees
                if best_bid is None or best_ask is None or (best_ask - best_bid) < 0.02:
                    self._log_scratch("Skipped - Vacuum or Spread < 2c")
                    return

                # Deploy the Maker-Maker Net
                engine.submit_limit_order('yes', best_bid, current_ts, self.active_signal['local_recv_ts'])
                engine.submit_limit_order('no', best_ask, current_ts, self.active_signal['local_recv_ts'])

                self.state = 'MAKER_PENDING'
                self.whipsaw_start_ts = current_ts

        # ---------------------------------------------------------
        # STATE 2: MAKER PENDING
        # ---------------------------------------------------------
        elif self.state == 'MAKER_PENDING':
            yes_order = engine.active_orders.get('yes')
            no_order = engine.active_orders.get('no')

            yes_filled = yes_order and yes_order['receipt']['status'] == 'filled'
            no_filled = no_order and no_order['receipt']['status'] == 'filled'

            # 1. Both sides filled
            if yes_filled and no_filled:
                yes_receipt = yes_order['receipt']
                no_receipt = no_order['receipt']

                entry_px = yes_receipt['fill_price']
                exit_px = no_receipt['fill_price']
                pnl = round(exit_px - entry_px, 2)

                self._log_full_whipsaw_trade(
                    yes_receipt=yes_receipt,
                    no_receipt=no_receipt,
                    exit_ts=current_ts,
                    pnl=pnl
                )
                return

            # 2. Timeout: 30 seconds have passed
            if current_ts - self.whipsaw_start_ts > self.WHIPSAW_TIMEOUT:

                # Only cancel if BOTH legs are dead
                if not yes_filled and not no_filled:
                    engine.cancel_active_order('yes', current_ts)
                    engine.cancel_active_order('no', current_ts)
                    self._log_scratch("Scratch - Dead Market, No Fills")
                    return

                # We are legged; move to active pegging
                self.filled_leg = 'yes' if yes_filled else 'no'
                self.entry_receipt = yes_order['receipt'] if yes_filled else no_order['receipt']

                self.state = 'ACTIVE_PEGGING'
                self.peg_start_ts = current_ts
                self.last_peg_ts = current_ts - self.PEG_CYCLE_SEC

        # ---------------------------------------------------------
        # STATE 3: ACTIVE PEGGING
        # ---------------------------------------------------------
        elif self.state == 'ACTIVE_PEGGING':
            exit_side = 'no' if self.filled_leg == 'yes' else 'yes'
            exit_order = engine.active_orders.get(exit_side)

            entry_px = self.entry_receipt['fill_price']
            time_in_peg = current_ts - self.peg_start_ts

            # A. Did our peg get hit naturally?
            if exit_order and exit_order['receipt']['status'] == 'filled':
                exit_px = exit_order['receipt']['fill_price']
                pnl = exit_px - entry_px if self.filled_leg == 'yes' else entry_px - exit_px

                self._log_closed_trade(
                    exit_reason="Scratch/Loss - Peg Hit",
                    exit_px=exit_px,
                    exit_ts=current_ts,
                    pnl=round(pnl, 2),
                    exit_receipt=exit_order['receipt']
                )
                return

            # B. TTL hard puke
            if time_in_peg > self.TTL_PUKE_SEC:
                engine.cancel_active_order(exit_side, current_ts)
                exit_px, pnl = self._execute_market_exit(engine, entry_px, self.filled_leg)
                self._log_closed_trade(
                    exit_reason="Loss - 60s Chase Timeout",
                    exit_px=exit_px,
                    exit_ts=current_ts,
                    pnl=pnl,
                    exit_receipt=None
                )
                return

            # C. The 3-second pegging cycle
            if current_ts - self.last_peg_ts >= self.PEG_CYCLE_SEC:
                self.last_peg_ts = current_ts

                best_bid = engine.get_best_bid()
                best_ask = engine.get_best_ask()

                if best_bid is None or best_ask is None:
                    return

                current_spread = round(best_ask - best_bid, 2)

                # Spread bailout
                if current_spread <= self.MIN_SPREAD_BAILOUT:
                    engine.cancel_active_order(exit_side, current_ts)
                    exit_px, pnl = self._execute_market_exit(engine, entry_px, self.filled_leg)
                    self._log_closed_trade(
                        exit_reason="Loss/Scratch - 1c Spread Bailout",
                        exit_px=exit_px,
                        exit_ts=current_ts,
                        pnl=pnl,
                        exit_receipt=None
                    )
                    return

                # Top-of-book queue retention
                current_order_px = exit_order['order_price'] if exit_order and exit_order['receipt']['status'] == 'pending' else None

                if self.filled_leg == 'yes':
                    if current_order_px is not None and current_order_px <= best_ask:
                        return
                    target_px = round(best_ask - 0.01, 2)
                    if target_px <= best_bid:
                        target_px = round(best_bid + 0.01, 2)
                else:
                    if current_order_px is not None and current_order_px >= best_bid:
                        return
                    target_px = round(best_bid + 0.01, 2)
                    if target_px >= best_ask:
                        target_px = round(best_ask - 0.01, 2)

                engine.cancel_active_order(exit_side, current_ts)
                engine.submit_limit_order(exit_side, target_px, current_ts, current_ts)

    # ---------------------------------------------------------
    # Logging helpers
    # ---------------------------------------------------------
    def _log_scratch(self, reason: str):
        self.completed_trades.append({
            'ticker': self.active_signal['ticker'],
            'signal_side': self.active_signal.get('predicted_side', 'maker'),
            'confidence': self.active_signal.get('confidence'),
            'signal_ts': self.active_signal.get('local_recv_ts'),
            'outcome': reason,
            'entry_price': None,
            'exit_price': None,
            'gross_pnl': 0.0,
            'yes_fill_ts': None,
            'no_fill_ts': None,
            'first_fill_ts': None,
            'second_fill_ts': None,
            'time_between_fills': None,
            'time_from_signal_to_first_fill': None,
            'time_from_order_to_first_fill': None,
            'same_timestamp_fill': None,
        })
        self._reset_fsm()

    def _log_full_whipsaw_trade(self, yes_receipt: dict, no_receipt: dict, exit_ts: float, pnl: float):
        yes_fill_ts = yes_receipt.get('fill_ts')
        no_fill_ts = no_receipt.get('fill_ts')

        first_fill_ts = min(yes_fill_ts, no_fill_ts)
        second_fill_ts = max(yes_fill_ts, no_fill_ts)

        signal_ts = self.active_signal.get('local_recv_ts')
        order_arrival_ts = min(yes_receipt.get('arrival_ts'), no_receipt.get('arrival_ts'))

        time_between_fills = second_fill_ts - first_fill_ts
        time_from_signal_to_first_fill = first_fill_ts - signal_ts if signal_ts is not None else None
        time_from_order_to_first_fill = first_fill_ts - order_arrival_ts if order_arrival_ts is not None else None

        self.completed_trades.append({
            'ticker': self.active_signal['ticker'],
            'signal_side': self.active_signal.get('predicted_side', 'maker'),
            'confidence': self.active_signal.get('confidence'),
            'signal_ts': signal_ts,
            'outcome': "Win - Full Whipsaw Arb",
            'entry_price': yes_receipt['fill_price'],
            'exit_price': no_receipt['fill_price'],
            'gross_pnl': pnl,

            'yes_initial_v_ahead': yes_receipt.get('initial_v_ahead'),
            'no_initial_v_ahead': no_receipt.get('initial_v_ahead'),
            'yes_trades_ahead_of_us': yes_receipt.get('trades_ahead_of_us'),
            'no_trades_ahead_of_us': no_receipt.get('trades_ahead_of_us'),
            'yes_pulls_while_waiting': yes_receipt.get('pulls_while_waiting'),
            'no_pulls_while_waiting': no_receipt.get('pulls_while_waiting'),

            'yes_fill_ts': yes_fill_ts,
            'no_fill_ts': no_fill_ts,
            'first_fill_ts': first_fill_ts,
            'second_fill_ts': second_fill_ts,
            'time_between_fills': time_between_fills,
            'time_from_signal_to_first_fill': time_from_signal_to_first_fill,
            'time_from_order_to_first_fill': time_from_order_to_first_fill,
            'same_timestamp_fill': bool(time_between_fills == 0.0),
            'time_in_trade': time_between_fills,
            'exit_ts': exit_ts,
            'legacy_time_from_yes_fill': exit_ts - yes_fill_ts if yes_fill_ts is not None else None,
        })
        self._reset_fsm()

    def _log_closed_trade(self, exit_reason: str, exit_px: float, exit_ts: float, pnl: float, exit_receipt: dict = None):
        entry_fill_ts = self.entry_receipt.get('fill_ts') if self.entry_receipt is not None else None
        exit_fill_ts = exit_receipt.get('fill_ts') if exit_receipt is not None else exit_ts

        signal_ts = self.active_signal.get('local_recv_ts') if self.active_signal is not None else None
        order_arrival_ts = self.entry_receipt.get('arrival_ts') if self.entry_receipt is not None else None

        time_in_trade = (exit_fill_ts - entry_fill_ts) if (entry_fill_ts is not None and exit_fill_ts is not None) else None
        time_from_signal_to_first_fill = (entry_fill_ts - signal_ts) if (entry_fill_ts is not None and signal_ts is not None) else None
        time_from_order_to_first_fill = (entry_fill_ts - order_arrival_ts) if (entry_fill_ts is not None and order_arrival_ts is not None) else None

        self.completed_trades.append({
            'ticker': self.active_signal['ticker'],
            'signal_side': self.active_signal.get('predicted_side', 'maker'),
            'confidence': self.active_signal.get('confidence'),
            'signal_ts': signal_ts,
            'outcome': exit_reason,
            'entry_price': self.entry_receipt['fill_price'],
            'exit_price': exit_px,
            'gross_pnl': pnl,
            'entry_fill_ts': entry_fill_ts,
            'exit_fill_ts': exit_fill_ts,
            'time_in_trade': time_in_trade,
            'time_from_signal_to_first_fill': time_from_signal_to_first_fill,
            'time_from_order_to_first_fill': time_from_order_to_first_fill,
            'exit_ts': exit_ts,
        })
        self._reset_fsm()

    def _reset_fsm(self):
        self.state = 'WAITING_FOR_SIGNAL'
        self.active_signal = None
        self.entry_receipt = None
        self.filled_leg = None
        self.whipsaw_start_ts = None
        self.peg_start_ts = None
        self.last_peg_ts = None

    def _execute_market_exit(self, engine: KalshiMatchingEngine, entry_px: float, filled_leg: str):
        if filled_leg == 'yes':
            exit_px = engine.get_best_bid()
            if exit_px is None:
                exit_px = 0.01
            pnl = exit_px - entry_px
        else:
            exit_px = engine.get_best_ask()
            if exit_px is None:
                exit_px = 0.99
            pnl = entry_px - exit_px

        return round(exit_px, 2), round(pnl, 2)

In [ ]:
import os
import pandas as pd
from tqdm import tqdm
import time

# --- CONFIGURATION ---
PREDICTIONS_PATH = '/content/hot_density_signal_files/predictions_hot_epoch1_top700.parquet'
RAW_LOCAL_DIR = '/content/kalshi_clean_raw_v2'
OUTPUT_AUDIT_PATH = "/content/hot_density_signal_files/trade_receipts_top700.parquet"

print("=== INITIATING ZERO-ASSUMPTION SIMULATOR ===")

# 1. Load the Signals
df_preds = pd.read_parquet(PREDICTIONS_PATH)
tickers = df_preds['ticker'].unique()
print(f"Loaded {len(df_preds)} signals across {len(tickers)} unique markets.")

all_receipts = []
start_time = time.time()

# 2. The Master Loop
for ticker in tqdm(tickers, desc="Simulating Markets"):
    ticker_preds = df_preds[df_preds['ticker'] == ticker].copy()

    clean_ticker = ticker.replace("dense_", "").replace(".parquet", "")
    raw_file_path = os.path.join(RAW_LOCAL_DIR, f"clean_raw_{clean_ticker}.parquet")

    if not os.path.exists(raw_file_path):
        print(f"\n[!] Raw data missing for {clean_ticker}. Skipping.")
        continue

    df_raw = pd.read_parquet(raw_file_path)

    engine = KalshiMatchingEngine()
    executor = KalshiExecutor(ticker_preds)

    raw_records = df_raw.to_dict('records')

    for row in raw_records:
        engine.process_tick(row)
        executor.update(row, engine)

    all_receipts.extend(executor.completed_trades)

# 3. Generate the Final Audit Log
end_time = time.time()
df_audit = pd.DataFrame(all_receipts)

print("\n=== SIMULATION COMPLETE ===")
print(f"Execution Time: {round(end_time - start_time, 2)} seconds")

if not df_audit.empty:
    df_audit.to_parquet(OUTPUT_AUDIT_PATH, index=False)

    print(f"\nTotal Signals Processed: {len(df_audit)}")
    print("\n--- OUTCOME DISTRIBUTION ---")
    print(df_audit['outcome'].value_counts())

    print("\n--- PNL SUMMARY ---")
    gross_pnl = df_audit['gross_pnl'].sum()
    print(f"Total Gross PnL: ${gross_pnl:.2f}")

    actionable = df_audit[~df_audit['outcome'].str.contains('Skipped')]
    if not actionable.empty:
        print(f"\nActionable trades: {len(actionable)}")
        print(f"Gross PnL / actionable trade: ${actionable['gross_pnl'].mean():.4f}")

    filled_trades = df_audit[~df_audit['outcome'].str.contains('Skipped|Scratch')]
    if not filled_trades.empty:
        print("\n--- SAMPLE COMPLETED TRADES ---")
        print(filled_trades.head())
else:
    print("\n[!] No trades were logged. Check the FSM logic or threshold constraints.")

In [ ]:
#volume sim draft starts here





#VOLUME SIM



#VOLUME SIM


#VOLUME SIM

#VOLUME SIM



import ast
import pandas as pd
import numpy as np
from typing import Dict, Optional, Any, Tuple, List


EPS = 1e-5
LIVE_ORDER_STATUSES = {"pending", "partial"}


def safe_float(x: Any, default: Optional[float] = np.nan) -> float:
    try:
        if x is None or pd.isna(x):
            return default
        return float(x)
    except Exception:
        return default


def parse_ladder(raw_ladder: Any) -> List[Tuple[float, float]]:
    """
    Parse snapshot ladders stored as nested numpy arrays / lists / tuples / strings.

    Expected normalized shape:
      [
        ['0.0100', '757201.00'],
        ['0.0200', '4223.00'],
        ...
      ]
    """
    if raw_ladder is None:
        return []

    try:
        if isinstance(raw_ladder, float) and pd.isna(raw_ladder):
            return []
    except Exception:
        pass

    ladder_obj = raw_ladder

    if hasattr(ladder_obj, "as_py"):
        try:
            ladder_obj = ladder_obj.as_py()
        except Exception:
            pass

    if hasattr(ladder_obj, "tolist") and not isinstance(ladder_obj, (str, bytes)):
        try:
            ladder_obj = ladder_obj.tolist()
        except Exception:
            pass

    if isinstance(ladder_obj, str):
        s = ladder_obj.strip()
        if s == "" or s.lower() == "none":
            return []
        try:
            ladder_obj = ast.literal_eval(s)
        except Exception:
            return []

    if not isinstance(ladder_obj, (list, tuple)):
        return []

    out: List[Tuple[float, float]] = []
    for item in ladder_obj:
        if hasattr(item, "as_py"):
            try:
                item = item.as_py()
            except Exception:
                pass

        if hasattr(item, "tolist") and not isinstance(item, (str, bytes)):
            try:
                item = item.tolist()
            except Exception:
                pass

        if not isinstance(item, (list, tuple)) or len(item) != 2:
            continue

        try:
            px = float(item[0])
            vol = float(item[1])
        except Exception:
            continue

        if np.isnan(px) or np.isnan(vol) or vol <= 0:
            continue

        out.append((round(px, 2), float(vol)))

    return out


def apply_snapshot_row(row: dict) -> Tuple[Dict[float, float], Dict[float, float]]:
    """
    Build YES-bid / YES-ask books from a snapshot row.

    Observed schema:
      yes_dollars_fp -> YES resting bids
      no_dollars_fp  -> NO resting bids, which imply YES asks via ask = 1 - no_bid
    """
    bids: Dict[float, float] = {}
    asks: Dict[float, float] = {}

    yes_ladder = parse_ladder(row.get("yes_dollars_fp"))
    no_ladder = parse_ladder(row.get("no_dollars_fp"))

    for px, vol in yes_ladder:
        if vol > EPS:
            bids[round(px, 2)] = float(vol)

    for no_px, vol in no_ladder:
        yes_ask_px = round(1.0 - float(no_px), 2)
        if vol > EPS:
            asks[yes_ask_px] = float(vol)

    return bids, asks


def get_yes_trade_price(row: dict) -> Optional[float]:
    """
    Normalize trade price into YES-dollar space.

    Some rows may have yes_price_dollars; others may only have no_price_dollars.
    For no_price_dollars, YES price = 1 - NO price.
    """
    yes_px = safe_float(row.get("yes_price_dollars"), default=np.nan)
    if not pd.isna(yes_px):
        return round(float(yes_px), 2)

    no_px = safe_float(row.get("no_price_dollars"), default=np.nan)
    if not pd.isna(no_px):
        return round(1.0 - float(no_px), 2)

    return None


def get_trade_count(row: dict) -> float:
    count_fp = safe_float(row.get("count_fp"), default=np.nan)
    if not pd.isna(count_fp):
        return float(count_fp)

    count = safe_float(row.get("count"), default=np.nan)
    if not pd.isna(count):
        return float(count)

    return 0.0


def is_live_order(order: Optional[Dict[str, Any]]) -> bool:
    if order is None:
        return False
    return order.get("receipt", {}).get("status") in LIVE_ORDER_STATUSES


class KalshiMatchingEngine:
    def __init__(self, include_own_orders_in_best: bool = False) -> None:
        self.resting_bids: Dict[float, float] = {}
        self.resting_asks: Dict[float, float] = {}

        # Diagnostics only
        self.snapshot_count: int = 0
        self.empty_snapshot_count: int = 0

        # If True, get_best_bid/get_best_ask include our own live resting orders.
        # For exact comparison to older optimistic logic, set True.
        # For more conservative execution simulation, leave False.
        self.include_own_orders_in_best = include_own_orders_in_best

        self.next_order_id: int = 1

        # yes = our YES bid.
        # no  = our YES ask / NO-side resting order.
        self.active_orders: Dict[str, Optional[Dict[str, Any]]] = {
            "yes": None,
            "no": None,
        }

    def get_best_bid(self) -> Optional[float]:
        valid_bids = [px for px, vol in self.resting_bids.items() if vol > EPS]
        best_raw = max(valid_bids) if valid_bids else None

        if self.include_own_orders_in_best:
            my_order = self.active_orders.get("yes")
            if is_live_order(my_order):
                my_px = my_order["order_price"]
                return max(best_raw, my_px) if best_raw is not None else my_px

        return best_raw

    def get_best_ask(self) -> Optional[float]:
        valid_asks = [px for px, vol in self.resting_asks.items() if vol > EPS]
        best_raw = min(valid_asks) if valid_asks else None

        if self.include_own_orders_in_best:
            my_order = self.active_orders.get("no")
            if is_live_order(my_order):
                my_px = my_order["order_price"]
                return min(best_raw, my_px) if best_raw is not None else my_px

        return best_raw

    def get_order_receipt(self, side: str) -> Optional[Dict[str, Any]]:
        order = self.active_orders.get(side)
        if order is None:
            return None
        return order["receipt"].copy()

    def _refresh_queue_ahead_from_snapshot(self) -> None:
        """
        After a non-empty snapshot reset, refresh queue-ahead estimates
        for any still-live orders from the authoritative external book.
        """
        yes_order = self.active_orders.get("yes")
        if is_live_order(yes_order):
            px = yes_order["order_price"]
            yes_order["v_ahead"] = self.resting_bids.get(px, 0.0)

        no_order = self.active_orders.get("no")
        if is_live_order(no_order):
            px = no_order["order_price"]
            no_order["v_ahead"] = self.resting_asks.get(px, 0.0)

    def _apply_fill(self, order: Dict[str, Any], fill_qty: float, fill_ts: float) -> None:
        """
        Apply a partial or full maker fill to one of our resting orders.
        All fills are at the order's resting limit price in this simulator.
        """
        fill_qty = min(float(fill_qty), float(order["remaining_volume"]))
        if fill_qty <= EPS:
            return

        px = float(order["order_price"])

        order["remaining_volume"] -= fill_qty
        order["remaining_volume"] = max(0.0, order["remaining_volume"])
        order["my_volume"] = order["remaining_volume"]  # backward-compatible alias
        order["filled_qty"] += fill_qty

        receipt = order["receipt"]
        receipt["filled_qty"] = order["filled_qty"]
        receipt["remaining_volume"] = order["remaining_volume"]
        receipt["fill_price"] = px
        receipt["avg_fill_price"] = px
        receipt["last_fill_ts"] = fill_ts
        receipt["fill_ts"] = fill_ts  # backward-compatible: completion ts for full fills, latest fill ts for partials

        if receipt["first_fill_ts"] is None:
            receipt["first_fill_ts"] = fill_ts

        receipt["fill_events"].append({
            "ts": fill_ts,
            "qty": fill_qty,
            "price": px,
        })

        if order["remaining_volume"] <= EPS:
            order["remaining_volume"] = 0.0
            order["my_volume"] = 0.0
            receipt["remaining_volume"] = 0.0
            receipt["status"] = "filled"
        else:
            receipt["status"] = "partial"

    def process_tick(self, row: dict) -> None:
        """
        Master method called on every tick.

        Important semantics:
        - orderbook_snapshot: full-ladder reset unless empty, then ignore.
        - orderbook_delta: single-level liquidity change.
        - trade: taker volume crossing the spread and potentially filling our live orders.
        """
        event_type: str = str(row.get("event_type"))

        if event_type not in ["orderbook_snapshot", "orderbook_delta", "trade"]:
            return

        # ---------------------------------------------------------
        # 1. Maintain the LOB
        # ---------------------------------------------------------
        if event_type == "orderbook_snapshot":
            self.snapshot_count += 1

            new_bids, new_asks = apply_snapshot_row(row)

            # Ignore terminal / empty snapshots instead of wiping the book
            if len(new_bids) == 0 and len(new_asks) == 0:
                self.empty_snapshot_count += 1
                return

            self.resting_bids = new_bids
            self.resting_asks = new_asks

            self._refresh_queue_ahead_from_snapshot()
            return

        if event_type == "orderbook_delta":
            tick_side: str = str(row.get("side"))
            price_val = safe_float(row.get("price_dollars"), default=np.nan)
            delta = safe_float(row.get("delta_fp"), default=0.0)

            if pd.isna(price_val) or tick_side not in ["yes", "no"]:
                return

            # YES book directly, NO book becomes YES ask via 1 - no_price
            price: float = round(price_val, 2) if tick_side == "yes" else round(1.0 - price_val, 2)
            book = self.resting_bids if tick_side == "yes" else self.resting_asks

            new_vol: float = book.get(price, 0.0) + delta
            if new_vol > EPS:
                book[price] = new_vol
            else:
                book.pop(price, None)

            # Queue Physics: pulls at our level reduce estimated volume ahead.
            # Heuristic retained from prior engine.
            for o_side, order in self.active_orders.items():
                if not is_live_order(order):
                    continue

                if tick_side == o_side and price == order["order_price"] and delta < 0:
                    pulled: float = abs(delta)
                    order["v_ahead"] = max(0.0, order["v_ahead"] - (pulled * 0.5))
                    order["receipt"]["pulls_while_waiting"] += pulled

            return

        # ---------------------------------------------------------
        # 2. Trade-driven queue / fill logic
        # ---------------------------------------------------------
        if event_type == "trade":
            price = get_yes_trade_price(row)
            if price is None:
                return

            tick_side: str = str(row.get("taker_side"))
            if tick_side not in ["yes", "no"]:
                return

            current_ts = safe_float(row.get("local_recv_ts"), default=np.nan)
            if pd.isna(current_ts):
                return

            traded: float = get_trade_count(row)
            if traded <= EPS:
                return

            for o_side, order in self.active_orders.items():
                if not is_live_order(order):
                    continue

                # Cannot fill before our order has arrived.
                if current_ts <= order["receipt"]["arrival_ts"]:
                    continue

                # Opposite-side taker flow can hit our resting order.
                if tick_side == o_side:
                    continue

                is_hit = False

                # Our YES bid gets hit by NO-side taker flow selling YES down into bid.
                if o_side == "yes" and price <= order["order_price"]:
                    is_hit = True

                # Our YES ask / NO-side order gets hit by YES-side taker flow buying YES up into ask.
                elif o_side == "no" and price >= order["order_price"]:
                    is_hit = True

                if not is_hit:
                    continue

                remaining_taker_volume = traded
                order["receipt"]["raw_hit_volume_at_or_through_price"] += traded

                # A. Taker volume eats queue ahead first.
                if order["v_ahead"] > EPS:
                    consumed = min(order["v_ahead"], remaining_taker_volume)
                    order["v_ahead"] -= consumed
                    remaining_taker_volume -= consumed
                    order["receipt"]["trades_ahead_of_us"] += consumed

                # B. Remaining taker volume fills us, possibly partially.
                if order["v_ahead"] <= EPS and remaining_taker_volume > EPS:
                    self._apply_fill(order, remaining_taker_volume, float(current_ts))

    def submit_limit_order(
        self,
        side: str,
        price: float,
        arrival_ts: float,
        signal_ts: float,
        volume: float = 1.0,
        tag: str = "entry",
    ) -> Optional[int]:
        """
        Places an independent order into the YES or NO queue.

        side='yes': our YES bid.
        side='no' : our YES ask / NO-side resting order.
        """
        if side not in ["yes", "no"]:
            return None

        order_price = round(float(price), 2)
        volume = float(volume)

        if volume <= EPS:
            return None

        book = self.resting_bids if side == "yes" else self.resting_asks
        v_ahead = float(book.get(order_price, 0.0))

        order_id = self.next_order_id
        self.next_order_id += 1

        self.active_orders[side] = {
            "order_id": order_id,
            "side": side,
            "tag": tag,
            "order_price": order_price,
            "initial_volume": volume,
            "remaining_volume": volume,
            "my_volume": volume,  # backward-compatible alias
            "filled_qty": 0.0,
            "v_ahead": v_ahead,
            "receipt": {
                "order_id": order_id,
                "side": side,
                "tag": tag,
                "signal_ts": signal_ts,
                "arrival_ts": arrival_ts,
                "target_price": order_price,
                "initial_volume": volume,
                "initial_v_ahead": v_ahead,
                "pulls_while_waiting": 0.0,
                "trades_ahead_of_us": 0.0,
                "raw_hit_volume_at_or_through_price": 0.0,
                "filled_qty": 0.0,
                "remaining_volume": volume,
                "fill_price": None,
                "avg_fill_price": None,
                "first_fill_ts": None,
                "last_fill_ts": None,
                "fill_ts": None,
                "fill_events": [],
                "status": "pending",
            },
        }

        return order_id

    def cancel_active_order(self, side: str, cancel_ts: float) -> Optional[Dict[str, Any]]:
        """
        Cancels a live order and returns its receipt.

        If the order was partially filled, the returned receipt preserves the fill history.
        """
        order = self.active_orders.get(side)

        if not is_live_order(order):
            return None

        receipt = order["receipt"]
        receipt["exit_ts"] = cancel_ts
        receipt["cancel_ts"] = cancel_ts
        receipt["filled_qty"] = order["filled_qty"]
        receipt["remaining_volume"] = order["remaining_volume"]

        if order["filled_qty"] > EPS:
            receipt["status"] = "canceled_partial"
        else:
            receipt["status"] = "canceled_unfilled"

        final_receipt = receipt.copy()
        self.active_orders[side] = None
        return final_receipt

In [ ]:
class KalshiExecutor:
    def __init__(self, predictions_df: pd.DataFrame, order_size: float = 1.0):
        # Load and sort predictions chronologically
        self.predictions = predictions_df.sort_values(by="local_recv_ts").to_dict("records")
        self.pred_idx = 0

        self.ORDER_SIZE = float(order_size)

        # FSM State
        self.state = "WAITING_FOR_SIGNAL"
        self.active_signal = None
        self.MAX_SIGNAL_AGE_SEC = 30.0 #temporary, simul track multiple trades later

        # --- MAKER-MAKER HYPERPARAMETERS ---
        self.LATENCY_PENALTY = 0.020
        self.WHIPSAW_TIMEOUT = 30.0

        # Active Inventory Management
        self.PEG_CYCLE_SEC = 3.0
        self.TTL_PUKE_SEC = 60.0
        self.MIN_SPREAD_BAILOUT = 0.01

        self.whipsaw_start_ts = None
        self.peg_start_ts = None
        self.last_peg_ts = None

        # Entry summary after timeout
        self.entry_yes_receipt = None
        self.entry_no_receipt = None
        self.entry_yes_px = None
        self.entry_no_px = None

        self.yes_filled_qty = 0.0
        self.no_filled_qty = 0.0
        self.matched_qty = 0.0
        self.net_yes_qty = 0.0

        self.clean_pnl = 0.0

        # Rescue / pegging state
        self.rescue_side = None
        self.rescue_qty_target = 0.0
        self.rescue_qty_filled = 0.0
        self.rescue_pnl = 0.0
        self.rescue_exit_notional = 0.0
        self.rescue_last_fill_ts = None
        self.rescue_order_counted_qty = {}

        # Final Audit Log
        self.completed_trades = []

    # ============================================================
    # Main update loop
    # ============================================================
    def update(self, row: pd.Series, engine: KalshiMatchingEngine) -> None:
        current_ts = float(row["local_recv_ts"])

        # ---------------------------------------------------------
        # STATE 1: WAITING FOR SIGNAL
        # ---------------------------------------------------------
        if self.state == "WAITING_FOR_SIGNAL":
            if self.pred_idx >= len(self.predictions):
                return

            # Consume and explicitly skip stale signals until we find either:
            #   A. a future signal not ready yet, or
            #   B. a fresh actionable signal.
            while self.pred_idx < len(self.predictions):
                next_signal = self.predictions[self.pred_idx]
                signal_ts = float(next_signal["local_recv_ts"])
                actionable_ts = signal_ts + self.LATENCY_PENALTY

                # Not ready yet.
                if current_ts < actionable_ts:
                    return

                signal_age = current_ts - actionable_ts

                # Too old to act on.
                if signal_age > self.MAX_SIGNAL_AGE_SEC:
                    self.active_signal = next_signal
                    self.pred_idx += 1
                    self._log_scratch("Skipped - Stale Signal")
                    continue

                # Fresh signal.
                break

            if self.pred_idx >= len(self.predictions):
                return

            next_signal = self.predictions[self.pred_idx]
            self.active_signal = next_signal
            self.pred_idx += 1

            best_bid = engine.get_best_bid()
            best_ask = engine.get_best_ask()

            if best_bid is None or best_ask is None:
                self._log_scratch("Skipped - Vacuum")
                return

            spread = round(best_ask - best_bid, 2)

            if spread < 0.02:
                self._log_scratch("Skipped - Spread < 2c")
                return

            engine.submit_limit_order(
                side="yes",
                price=best_bid,
                arrival_ts=current_ts,
                signal_ts=self.active_signal["local_recv_ts"],
                volume=self.ORDER_SIZE,
                tag="entry_yes",
            )

            engine.submit_limit_order(
                side="no",
                price=best_ask,
                arrival_ts=current_ts,
                signal_ts=self.active_signal["local_recv_ts"],
                volume=self.ORDER_SIZE,
                tag="entry_no",
            )

            self.state = "MAKER_PENDING"
            self.whipsaw_start_ts = current_ts

        # ---------------------------------------------------------
        # STATE 2: MAKER PENDING
        # ---------------------------------------------------------
        elif self.state == "MAKER_PENDING":
            yes_order = engine.active_orders.get("yes")
            no_order = engine.active_orders.get("no")

            yes_qty = self._order_filled_qty(yes_order)
            no_qty = self._order_filled_qty(no_order)

            # Case A: full clean whipsaw at requested size.
            if yes_qty >= self.ORDER_SIZE - EPS and no_qty >= self.ORDER_SIZE - EPS:
                yes_receipt = engine.get_order_receipt("yes")
                no_receipt = engine.get_order_receipt("no")

                self._load_entry_summary(yes_receipt, no_receipt)
                self._log_terminal_trade(
                    outcome="Win - Full Clean Whipsaw",
                    exit_reason="both_entry_orders_filled",
                    exit_ts=current_ts,
                )
                return

            # Case B: whipsaw window expired.
            if current_ts - self.whipsaw_start_ts > self.WHIPSAW_TIMEOUT:
                # Snapshot receipts before canceling.
                yes_receipt_before = engine.get_order_receipt("yes")
                no_receipt_before = engine.get_order_receipt("no")

                # Cancel live remainders.
                yes_cancel_receipt = engine.cancel_active_order("yes", current_ts)
                no_cancel_receipt = engine.cancel_active_order("no", current_ts)

                yes_receipt = yes_cancel_receipt or yes_receipt_before
                no_receipt = no_cancel_receipt or no_receipt_before

                self._load_entry_summary(yes_receipt, no_receipt)

                # No fills at all.
                if self.yes_filled_qty <= EPS and self.no_filled_qty <= EPS:
                    self._log_scratch("Scratch - Dead Market, No Fills")
                    return

                # Some matched quantity, but no net inventory.
                if abs(self.net_yes_qty) <= EPS:
                    if self.matched_qty >= self.ORDER_SIZE - EPS:
                        outcome = "Win - Full Clean Whipsaw"
                    else:
                        outcome = "Win - Partial Clean Whipsaw"

                    self._log_terminal_trade(
                        outcome=outcome,
                        exit_reason="timeout_no_net_inventory",
                        exit_ts=current_ts,
                    )
                    return

                # We have inventory to rescue.
                self.state = "ACTIVE_PEGGING"
                self.peg_start_ts = current_ts
                self.last_peg_ts = current_ts - self.PEG_CYCLE_SEC

                self.rescue_side = "no" if self.net_yes_qty > 0 else "yes"
                self.rescue_qty_target = abs(self.net_yes_qty)
                self.rescue_qty_filled = 0.0
                self.rescue_pnl = 0.0
                self.rescue_exit_notional = 0.0
                self.rescue_last_fill_ts = None
                self.rescue_order_counted_qty = {}

                placed = self._place_or_reprice_rescue_order(
                    engine=engine,
                    current_ts=current_ts,
                    force=True,
                )

                if not placed:
                    self._market_exit_remaining_inventory(
                        engine=engine,
                        current_ts=current_ts,
                        reason="Loss/Scratch - Could Not Place Initial Rescue Peg",
                    )
                    return

        # ---------------------------------------------------------
        # STATE 3: ACTIVE PEGGING
        # ---------------------------------------------------------
        elif self.state == "ACTIVE_PEGGING":
            self._sync_rescue_progress(engine)

            # If rescue order fully handled the imbalance, close.
            if self.rescue_qty_filled >= self.rescue_qty_target - EPS:
                self._log_terminal_trade(
                    outcome=self._classify_rescue_outcome("Closed - Rescue Peg Filled"),
                    exit_reason="rescue_peg_filled",
                    exit_ts=current_ts,
                )
                return

            time_in_peg = current_ts - self.peg_start_ts

            # Hard TTL puke.
            if time_in_peg > self.TTL_PUKE_SEC:
                self._market_exit_remaining_inventory(
                    engine=engine,
                    current_ts=current_ts,
                    reason="Loss - Chase Timeout Market Exit",
                )
                return

            # Pegging cycle.
            if current_ts - self.last_peg_ts >= self.PEG_CYCLE_SEC:
                self.last_peg_ts = current_ts

                best_bid = engine.get_best_bid()
                best_ask = engine.get_best_ask()

                if best_bid is None or best_ask is None:
                    return

                current_spread = round(best_ask - best_bid, 2)

                # Spread bailout.
                if current_spread <= self.MIN_SPREAD_BAILOUT:
                    self._market_exit_remaining_inventory(
                        engine=engine,
                        current_ts=current_ts,
                        reason="Loss/Scratch - 1c Spread Bailout",
                    )
                    return

                self._place_or_reprice_rescue_order(
                    engine=engine,
                    current_ts=current_ts,
                    force=False,
                )

    # ============================================================
    # Quantity helpers
    # ============================================================
    def _order_filled_qty(self, order) -> float:
        if order is None:
            return 0.0
        return float(order.get("filled_qty", order.get("receipt", {}).get("filled_qty", 0.0)) or 0.0)

    def _receipt_filled_qty(self, receipt) -> float:
        if receipt is None:
            return 0.0
        return float(receipt.get("filled_qty", 0.0) or 0.0)

    def _receipt_remaining_qty(self, receipt) -> float:
        if receipt is None:
            return 0.0
        return float(receipt.get("remaining_volume", 0.0) or 0.0)

    def _load_entry_summary(self, yes_receipt: dict, no_receipt: dict) -> None:
        self.entry_yes_receipt = yes_receipt or {}
        self.entry_no_receipt = no_receipt or {}

        self.entry_yes_px = self.entry_yes_receipt.get("fill_price") or self.entry_yes_receipt.get("target_price")
        self.entry_no_px = self.entry_no_receipt.get("fill_price") or self.entry_no_receipt.get("target_price")

        self.yes_filled_qty = self._receipt_filled_qty(self.entry_yes_receipt)
        self.no_filled_qty = self._receipt_filled_qty(self.entry_no_receipt)

        self.matched_qty = min(self.yes_filled_qty, self.no_filled_qty)
        self.net_yes_qty = self.yes_filled_qty - self.no_filled_qty

        if self.entry_yes_px is not None and self.entry_no_px is not None:
            self.clean_pnl = round(self.matched_qty * (self.entry_no_px - self.entry_yes_px), 4)
        else:
            self.clean_pnl = 0.0

    # ============================================================
    # Rescue / pegging helpers
    # ============================================================
    def _remaining_rescue_qty(self) -> float:
        return max(0.0, self.rescue_qty_target - self.rescue_qty_filled)

    def _sync_rescue_progress(self, engine: KalshiMatchingEngine) -> None:
        """
        Count newly filled quantity on the current rescue order without double counting.
        This handles partial rescue fills before a cancel/reprice.
        """
        if self.rescue_side not in ["yes", "no"]:
            return

        order = engine.active_orders.get(self.rescue_side)
        if order is None:
            return

        order_id = order.get("order_id")
        if order_id is None:
            return

        total_filled_on_order = float(order.get("filled_qty", 0.0) or 0.0)
        already_counted = float(self.rescue_order_counted_qty.get(order_id, 0.0))
        delta_qty = total_filled_on_order - already_counted

        if delta_qty <= EPS:
            return

        delta_qty = min(delta_qty, self._remaining_rescue_qty())
        exit_px = float(order["order_price"])

        if self.net_yes_qty > 0:
            # Long YES from excess YES bid fills; sell YES to exit.
            pnl_delta = delta_qty * (exit_px - self.entry_yes_px)
        else:
            # Short YES from excess NO/ask fills; buy YES to exit.
            pnl_delta = delta_qty * (self.entry_no_px - exit_px)

        self.rescue_qty_filled += delta_qty
        self.rescue_pnl += pnl_delta
        self.rescue_exit_notional += delta_qty * exit_px
        self.rescue_order_counted_qty[order_id] = already_counted + delta_qty

        receipt = order.get("receipt", {})
        self.rescue_last_fill_ts = receipt.get("last_fill_ts") or receipt.get("fill_ts") or self.rescue_last_fill_ts

    def _current_rescue_order_is_good_enough(self, engine: KalshiMatchingEngine) -> bool:
        order = engine.active_orders.get(self.rescue_side)
        if not is_live_order(order):
            return False

        best_bid = engine.get_best_bid()
        best_ask = engine.get_best_ask()

        if best_bid is None or best_ask is None:
            return True

        current_px = float(order["order_price"])

        if self.rescue_side == "no":
            # Selling YES. If we are at or better than current best ask, keep.
            return current_px <= best_ask

        if self.rescue_side == "yes":
            # Buying YES. If we are at or better than current best bid, keep.
            return current_px >= best_bid

        return False

    def _compute_rescue_target_price(self, engine: KalshiMatchingEngine) -> Optional[float]:
        best_bid = engine.get_best_bid()
        best_ask = engine.get_best_ask()

        if best_bid is None or best_ask is None:
            return None

        best_bid = float(best_bid)
        best_ask = float(best_ask)

        if self.rescue_side == "no":
            # Need to sell YES. Join/improve ask side.
            target_px = round(best_ask - 0.01, 2)
            if target_px <= best_bid:
                target_px = round(best_bid + 0.01, 2)

        elif self.rescue_side == "yes":
            # Need to buy YES. Join/improve bid side.
            target_px = round(best_bid + 0.01, 2)
            if target_px >= best_ask:
                target_px = round(best_ask - 0.01, 2)

        else:
            return None

        return min(0.99, max(0.01, round(target_px, 2)))

    def _place_or_reprice_rescue_order(
        self,
        engine: KalshiMatchingEngine,
        current_ts: float,
        force: bool = False,
    ) -> bool:
        self._sync_rescue_progress(engine)

        remaining_qty = self._remaining_rescue_qty()
        if remaining_qty <= EPS:
            return True

        if not force and self._current_rescue_order_is_good_enough(engine):
            return True

        # Cancel existing live rescue order, but count any partial fills first.
        engine.cancel_active_order(self.rescue_side, current_ts)

        target_px = self._compute_rescue_target_price(engine)
        if target_px is None:
            return False

        engine.submit_limit_order(
            side=self.rescue_side,
            price=target_px,
            arrival_ts=current_ts,
            signal_ts=current_ts,
            volume=remaining_qty,
            tag="rescue_peg",
        )

        return True

    def _market_exit_remaining_inventory(
        self,
        engine: KalshiMatchingEngine,
        current_ts: float,
        reason: str,
    ) -> None:
        self._sync_rescue_progress(engine)

        if self.rescue_side in ["yes", "no"]:
            engine.cancel_active_order(self.rescue_side, current_ts)

        remaining_qty = self._remaining_rescue_qty()

        if remaining_qty > EPS:
            if self.net_yes_qty > 0:
                # Long YES. Sell at bid.
                exit_px = engine.get_best_bid()
                if exit_px is None:
                    exit_px = 0.01
                exit_px = round(float(exit_px), 2)
                pnl_delta = remaining_qty * (exit_px - self.entry_yes_px)

            else:
                # Short YES. Buy at ask.
                exit_px = engine.get_best_ask()
                if exit_px is None:
                    exit_px = 0.99
                exit_px = round(float(exit_px), 2)
                pnl_delta = remaining_qty * (self.entry_no_px - exit_px)

            self.rescue_qty_filled += remaining_qty
            self.rescue_pnl += pnl_delta
            self.rescue_exit_notional += remaining_qty * exit_px
            self.rescue_last_fill_ts = current_ts

        self._log_terminal_trade(
            outcome=self._classify_rescue_outcome(reason),
            exit_reason=reason,
            exit_ts=current_ts,
        )

    def _classify_rescue_outcome(self, label: str) -> str:
        total = self.clean_pnl + self.rescue_pnl

        if total > EPS:
            prefix = "Win"
        elif total < -EPS:
            prefix = "Loss"
        else:
            prefix = "Scratch"

        return f"{prefix} - {label}"

    # ============================================================
    # Logging
    # ============================================================
    def _log_scratch(self, reason: str):
        sig = self.active_signal or {}

        self.completed_trades.append({
            "ticker": sig.get("ticker"),
            "signal_side": sig.get("predicted_side", "maker"),
            "confidence": sig.get("confidence"),
            "signal_ts": sig.get("local_recv_ts"),
            "order_size": self.ORDER_SIZE,

            "outcome": reason,
            "exit_reason": reason,

            "entry_price": None,
            "exit_price": None,
            "gross_pnl": 0.0,

            "yes_filled_qty": 0.0,
            "no_filled_qty": 0.0,
            "matched_qty": 0.0,
            "net_yes_qty": 0.0,
            "clean_pnl": 0.0,
            "rescue_qty_target": 0.0,
            "rescue_qty_filled": 0.0,
            "rescue_pnl": 0.0,

            "yes_remaining_canceled": None,
            "no_remaining_canceled": None,

            "yes_fill_ts": None,
            "no_fill_ts": None,
            "first_fill_ts": None,
            "last_fill_ts": None,
            "time_between_first_last_fill": None,
            "time_from_signal_to_first_fill": None,
            "time_from_order_to_first_fill": None,
            "time_in_trade": None,
            "exit_ts": None,
        })

        self._reset_fsm()

    def _log_terminal_trade(self, outcome: str, exit_reason: str, exit_ts: float):
        sig = self.active_signal or {}

        yes_r = self.entry_yes_receipt or {}
        no_r = self.entry_no_receipt or {}

        gross_pnl = round(self.clean_pnl + self.rescue_pnl, 4)

        first_fill_candidates = [
            yes_r.get("first_fill_ts"),
            no_r.get("first_fill_ts"),
        ]
        first_fill_candidates = [x for x in first_fill_candidates if x is not None]

        last_fill_candidates = [
            yes_r.get("last_fill_ts") or yes_r.get("fill_ts"),
            no_r.get("last_fill_ts") or no_r.get("fill_ts"),
            self.rescue_last_fill_ts,
        ]
        last_fill_candidates = [x for x in last_fill_candidates if x is not None]

        first_fill_ts = min(first_fill_candidates) if first_fill_candidates else None
        last_fill_ts = max(last_fill_candidates) if last_fill_candidates else None

        signal_ts = sig.get("local_recv_ts")

        order_arrival_candidates = [
            yes_r.get("arrival_ts"),
            no_r.get("arrival_ts"),
        ]
        order_arrival_candidates = [x for x in order_arrival_candidates if x is not None]
        first_order_arrival_ts = min(order_arrival_candidates) if order_arrival_candidates else None

        time_between = (
            last_fill_ts - first_fill_ts
            if first_fill_ts is not None and last_fill_ts is not None
            else None
        )

        time_from_signal = (
            first_fill_ts - signal_ts
            if first_fill_ts is not None and signal_ts is not None
            else None
        )

        time_from_order = (
            first_fill_ts - first_order_arrival_ts
            if first_fill_ts is not None and first_order_arrival_ts is not None
            else None
        )

        time_in_trade = (
            exit_ts - first_fill_ts
            if first_fill_ts is not None and exit_ts is not None
            else None
        )

        rescue_exit_vwap = (
            self.rescue_exit_notional / self.rescue_qty_filled
            if self.rescue_qty_filled > EPS
            else None
        )

        # For old compatibility, entry/exit price are the initial YES bid and NO/YES-ask.
        entry_price = self.entry_yes_px
        exit_price = self.entry_no_px if self.rescue_qty_target <= EPS else rescue_exit_vwap

        self.completed_trades.append({
            "ticker": sig.get("ticker"),
            "signal_side": sig.get("predicted_side", "maker"),
            "confidence": sig.get("confidence"),
            "signal_ts": signal_ts,
            "order_size": self.ORDER_SIZE,

            "outcome": outcome,
            "exit_reason": exit_reason,

            "entry_price": entry_price,
            "exit_price": exit_price,
            "gross_pnl": gross_pnl,

            "yes_entry_price": self.entry_yes_px,
            "no_entry_price": self.entry_no_px,

            "yes_filled_qty": round(self.yes_filled_qty, 6),
            "no_filled_qty": round(self.no_filled_qty, 6),
            "matched_qty": round(self.matched_qty, 6),
            "net_yes_qty": round(self.net_yes_qty, 6),

            "clean_pnl": round(self.clean_pnl, 4),
            "rescue_side": self.rescue_side,
            "rescue_qty_target": round(self.rescue_qty_target, 6),
            "rescue_qty_filled": round(self.rescue_qty_filled, 6),
            "rescue_exit_vwap": rescue_exit_vwap,
            "rescue_pnl": round(self.rescue_pnl, 4),

            "yes_remaining_canceled": self._receipt_remaining_qty(yes_r),
            "no_remaining_canceled": self._receipt_remaining_qty(no_r),

            "yes_initial_v_ahead": yes_r.get("initial_v_ahead"),
            "no_initial_v_ahead": no_r.get("initial_v_ahead"),
            "yes_trades_ahead_of_us": yes_r.get("trades_ahead_of_us"),
            "no_trades_ahead_of_us": no_r.get("trades_ahead_of_us"),
            "yes_pulls_while_waiting": yes_r.get("pulls_while_waiting"),
            "no_pulls_while_waiting": no_r.get("pulls_while_waiting"),

            "yes_fill_ts": yes_r.get("first_fill_ts") or yes_r.get("fill_ts"),
            "no_fill_ts": no_r.get("first_fill_ts") or no_r.get("fill_ts"),
            "first_fill_ts": first_fill_ts,
            "last_fill_ts": last_fill_ts,
            "time_between_first_last_fill": time_between,
            "time_from_signal_to_first_fill": time_from_signal,
            "time_from_order_to_first_fill": time_from_order,
            "time_in_trade": time_in_trade,
            "exit_ts": exit_ts,
        })

        self._reset_fsm()

    def _reset_fsm(self):
        self.state = "WAITING_FOR_SIGNAL"
        self.active_signal = None

        self.whipsaw_start_ts = None
        self.peg_start_ts = None
        self.last_peg_ts = None

        self.entry_yes_receipt = None
        self.entry_no_receipt = None
        self.entry_yes_px = None
        self.entry_no_px = None

        self.yes_filled_qty = 0.0
        self.no_filled_qty = 0.0
        self.matched_qty = 0.0
        self.net_yes_qty = 0.0

        self.clean_pnl = 0.0

        self.rescue_side = None
        self.rescue_qty_target = 0.0
        self.rescue_qty_filled = 0.0
        self.rescue_pnl = 0.0
        self.rescue_exit_notional = 0.0
        self.rescue_last_fill_ts = None
        self.rescue_order_counted_qty = {}

In [ ]:
import os
import pandas as pd
from tqdm import tqdm
import time

# --- CONFIGURATION ---
PREDICTIONS_PATH = '/content/hot_density_signal_files/predictions_hot_epoch1_top700.parquet'
RAW_LOCAL_DIR = "/content/kalshi_clean_raw_v2"

ORDER_SIZES = [1, 5, 10, 75]

INCLUDE_OWN_ORDERS_IN_BEST = False

print("=== INITIATING PARTIAL-FILL MAKER-MAKER SIMULATOR ===")

df_preds = pd.read_parquet(PREDICTIONS_PATH)

required_cols = {"ticker", "local_recv_ts"}
missing = required_cols - set(df_preds.columns)
if missing:
    raise ValueError(f"Predictions missing required columns: {missing}")

tickers = df_preds["ticker"].unique()
print(f"Loaded {len(df_preds)} signals across {len(tickers)} unique markets.")
print(f"Order sizes to simulate: {ORDER_SIZES}")

all_size_results = []
global_start_time = time.time()

for order_size in ORDER_SIZES:
    print("\n" + "=" * 80)
    print(f"SIMULATING ORDER_SIZE = {order_size}")
    print("=" * 80)

    all_receipts = []
    size_start_time = time.time()

    for ticker in tqdm(tickers, desc=f"Simulating Q={order_size}"):
        ticker_preds = df_preds[df_preds["ticker"] == ticker].copy()

        clean_ticker = ticker.replace("dense_", "").replace(".parquet", "")
        raw_file_path = os.path.join(
            RAW_LOCAL_DIR,
            f"clean_raw_{clean_ticker}.parquet"
        )

        if not os.path.exists(raw_file_path):
            print(f"\n[!] Raw data missing for {clean_ticker}: {raw_file_path}")
            continue

        df_raw = pd.read_parquet(raw_file_path)

        if df_raw.empty:
            print(f"\n[!] Empty raw data for {clean_ticker}. Skipping.")
            continue

        sort_cols = ["local_recv_ts"]
        if "seq" in df_raw.columns:
            sort_cols.append("seq")

        df_raw = df_raw.sort_values(
            by=sort_cols,
            kind="mergesort"
        ).reset_index(drop=True)

        engine = KalshiMatchingEngine(
            include_own_orders_in_best=INCLUDE_OWN_ORDERS_IN_BEST
        )

        executor = KalshiExecutor(
            predictions_df=ticker_preds,
            order_size=order_size,
        )

        for row in df_raw.to_dict("records"):
            engine.process_tick(row)
            executor.update(row, engine)

        all_receipts.extend(executor.completed_trades)

    size_end_time = time.time()
    df_audit = pd.DataFrame(all_receipts)

    output_path = f"trade_receipts_Q{order_size}.parquet"

    print("\n--- SIZE RUN COMPLETE ---")
    print(f"ORDER_SIZE: {order_size}")
    print(f"Execution Time: {round(size_end_time - size_start_time, 2)} seconds")

    if df_audit.empty:
        print("[!] No trades were logged. Check FSM logic or threshold constraints.")
        continue

    df_audit.to_parquet(output_path, index=False)
    print(f"Saved audit log to {output_path}")

    print(f"\nTotal Signals Processed / Logged: {len(df_audit)}")

    print("\n--- OUTCOME DISTRIBUTION ---")
    print(df_audit["outcome"].value_counts(dropna=False))

    print("\n--- GROSS PNL SUMMARY ---")
    gross_pnl = df_audit["gross_pnl"].sum()
    print(f"Total Gross PnL: ${gross_pnl:.4f}")

    quantity_cols = [
        "yes_filled_qty",
        "no_filled_qty",
        "matched_qty",
        "net_yes_qty",
        "rescue_qty_target",
        "rescue_qty_filled",
        "clean_pnl",
        "rescue_pnl",
        "gross_pnl",
    ]

    existing_quantity_cols = [c for c in quantity_cols if c in df_audit.columns]

    print("\n--- QUANTITY / PNL TOTALS ---")
    print(df_audit[existing_quantity_cols].sum(numeric_only=True))

    print("\n--- EXIT REASON DISTRIBUTION ---")
    if "exit_reason" in df_audit.columns:
        print(df_audit["exit_reason"].value_counts(dropna=False))

    print("\n--- SAMPLE NON-SCRATCH TRADES ---")
    non_scratch = df_audit[
        ~df_audit["outcome"].astype(str).str.contains(
            "Skipped|Dead Market|No Fills",
            na=False
        )
    ]

    if not non_scratch.empty:
        display_cols = [
            "ticker",
            "outcome",
            "order_size",
            "gross_pnl",
            "yes_filled_qty",
            "no_filled_qty",
            "matched_qty",
            "net_yes_qty",
            "clean_pnl",
            "rescue_pnl",
            "rescue_qty_target",
            "rescue_qty_filled",
            "time_in_trade",
        ]
        display_cols = [c for c in display_cols if c in non_scratch.columns]
        print(non_scratch[display_cols].head(20))
    else:
        print("No non-scratch trades.")

    all_size_results.append({
        "order_size": order_size,
        "num_logged": len(df_audit),
        "gross_pnl": gross_pnl,
        "matched_qty": df_audit["matched_qty"].sum() if "matched_qty" in df_audit.columns else None,
        "net_abs_qty": df_audit["net_yes_qty"].abs().sum() if "net_yes_qty" in df_audit.columns else None,
        "rescue_qty_target": df_audit["rescue_qty_target"].sum() if "rescue_qty_target" in df_audit.columns else None,
        "rescue_qty_filled": df_audit["rescue_qty_filled"].sum() if "rescue_qty_filled" in df_audit.columns else None,
    })

global_end_time = time.time()

print("\n" + "=" * 80)
print("ALL SIZE RUNS COMPLETE")
print(f"Total Execution Time: {round(global_end_time - global_start_time, 2)} seconds")
print("=" * 80)

if all_size_results:
    df_size_summary = pd.DataFrame(all_size_results)
    df_size_summary.to_parquet("order_size_summary.parquet", index=False)

    print("\n--- ORDER SIZE SUMMARY ---")
    print(df_size_summary)
else:
    print("[!] No size results generated.")

In [ ]:
####



























####STAGE 2 DEMO####























#####

In [ ]:
import os
import copy
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ============================================================
# CONFIG
# ============================================================

STAGE1_CHECKPOINT = "/content/hot_density_checkpoints/hot_density_epoch_1.pth"
STAGE2_DIR = "/content/stage2_first_pass"
RAW_LOCAL_DIR = "/content/kalshi_clean_raw_v2"

os.makedirs(STAGE2_DIR, exist_ok=True)

STAGE1_P_HOT_THRESHOLD = 0.50
STAGE2_MIN_ENTRY_SPREAD = 0.02
STAGE2_CANDIDATE_COOLDOWN_SEC = 1.0
MIN_TIME_TO_END_SEC = 100.0

SEQ_LEN = 60


# ============================================================
# LOAD STAGE 1 MODEL
# ============================================================

def load_stage1_model(checkpoint_path):
    model = KalshiTransformer(
        feature_dim=23,
        d_model=64,
        nhead=4,
        num_layers=2,
        num_classes=2,
    ).to(device)

    try:
        state = torch.load(checkpoint_path, map_location=device, weights_only=True)
    except TypeError:
        state = torch.load(checkpoint_path, map_location=device)

    model.load_state_dict(state)
    model.eval()
    return model


# ============================================================
# STAGE 1 OUTPUTS + STAGE 2 CANDIDATES
# ============================================================

def generate_stage2_candidates_from_stage1(
    file_paths,
    checkpoint_path,
    split_name,
    p_hot_threshold=0.50,
    min_entry_spread=0.02,
    candidate_cooldown_sec=1.0,
    min_time_to_end_sec=100.0,
    batch_size=512,
):
    """
    Builds a stride-1 Stage 1 inference dataset, runs the frozen Stage 1 model,
    and emits Stage 2 candidates.

    Candidate rule:
        p_hot >= threshold
        current spread >= 2c
        enough time remaining in market
        per-ticker cooldown
    """

    print("\n" + "=" * 90)
    print(f"GENERATING STAGE 2 CANDIDATES: {split_name}")
    print("=" * 90)

    stage1_model = load_stage1_model(checkpoint_path)

    ds = KalshiHotDensityDataset(
        file_paths=file_paths,
        seq_len=SEQ_LEN,
        candidate_lookahead=30,
        fill_lookahead=30,
        stride=1,
        k_hot=5,
        min_spread=0.02,
        inference_mode=True,
    )

    loader = DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
    )

    # Local row index inside each dense file. This lets Stage2Dataset reconstruct features.
    local_idx_array = ds.data.groupby("ticker", sort=False).cumcount().to_numpy()

    rows = []
    cursor = 0

    with torch.no_grad():
        for tickers, ts_exchange_batch, ts_recv_batch, x_batch in tqdm(loader, desc=f"Stage1 inference {split_name}"):
            bsz = len(tickers)
            idxs = np.asarray(ds.indices[cursor:cursor + bsz])
            cursor += bsz

            x_batch = x_batch.to(device)
            logits = stage1_model(x_batch)
            probs = torch.softmax(logits, dim=1)
            p_hot = probs[:, 1].detach().cpu().numpy()

            ts_exchange_np = (
                ts_exchange_batch.detach().cpu().numpy()
                if torch.is_tensor(ts_exchange_batch)
                else np.asarray(ts_exchange_batch)
            )
            ts_recv_np = (
                ts_recv_batch.detach().cpu().numpy()
                if torch.is_tensor(ts_recv_batch)
                else np.asarray(ts_recv_batch)
            )

            for j in range(bsz):
                idx = int(idxs[j])
                bid = float(ds.bid_array[idx])
                ask = float(ds.ask_array[idx])
                spread = ask - bid

                rows.append({
                    "sample_pos": int(cursor - bsz + j),
                    "current_idx": idx,
                    "local_row_idx": int(local_idx_array[idx]),

                    "ticker": str(tickers[j]),
                    "ts_exchange": float(ts_exchange_np[j]),
                    "local_recv_ts": float(ts_recv_np[j]),

                    "p_hot": float(p_hot[j]),
                    "bid_px_1": bid,
                    "ask_px_1": ask,
                    "spread": float(spread),

                    # diagnostics from Stage 1 dataset
                    "stage1_hot_label": int(ds.hot_label[idx]),
                    "hot_count_0_30": int(ds.hot_count_0_30[idx]),
                    "bad_count_0_30": int(ds.bad_count_0_30[idx]),
                    "candidate_good_now": int(ds.candidate_good[idx]),
                    "candidate_bad_now": int(ds.candidate_bad[idx]),
                })

    all_outputs = pd.DataFrame(rows)

    # Market-end guard.
    all_outputs["market_end_ts"] = all_outputs.groupby("ticker")["local_recv_ts"].transform("max")
    all_outputs["time_to_market_end"] = all_outputs["market_end_ts"] - all_outputs["local_recv_ts"]

    cand = all_outputs[
        (all_outputs["p_hot"] >= p_hot_threshold)
        & (all_outputs["spread"] >= min_entry_spread)
        & (all_outputs["time_to_market_end"] >= min_time_to_end_sec)
    ].copy()

    # Greedy per-ticker cooldown.
    cand = cand.sort_values(["ticker", "local_recv_ts"]).reset_index(drop=True)
    keep = []
    last_ts = {}

    for row in cand.to_dict("records"):
        ticker = row["ticker"]
        ts = float(row["local_recv_ts"])

        if ticker not in last_ts or ts - last_ts[ticker] >= candidate_cooldown_sec:
            keep.append(row)
            last_ts[ticker] = ts

    cand = pd.DataFrame(keep).sort_values("local_recv_ts").reset_index(drop=True)

    outputs_path = os.path.join(STAGE2_DIR, f"stage1_outputs_{split_name}.parquet")
    cand_path = os.path.join(STAGE2_DIR, f"stage2_candidates_{split_name}.parquet")

    all_outputs.to_parquet(outputs_path, index=False)
    cand.to_parquet(cand_path, index=False)

    print(f"\nSaved all Stage1 outputs: {outputs_path}")
    print(f"Saved Stage2 candidates:  {cand_path}")
    print(f"All outputs rows: {len(all_outputs)}")
    print(f"Candidate rows:   {len(cand)}")
    print(f"Unique tickers:   {cand['ticker'].nunique() if not cand.empty else 0}")

    if not cand.empty:
        print("\nCandidate p_hot quantiles:")
        print(cand["p_hot"].quantile([0, 0.25, 0.5, 0.75, 0.9, 0.99, 1.0]))
        print("\nCandidate spread quantiles:")
        print(cand["spread"].quantile([0, 0.25, 0.5, 0.75, 0.9, 0.99, 1.0]))

    return ds, all_outputs, cand


train_stage1_ds, train_stage1_outputs, train_candidates = generate_stage2_candidates_from_stage1(
    file_paths=train_files,
    checkpoint_path=STAGE1_CHECKPOINT,
    split_name="train",
    p_hot_threshold=STAGE1_P_HOT_THRESHOLD,
    min_entry_spread=STAGE2_MIN_ENTRY_SPREAD,
    candidate_cooldown_sec=STAGE2_CANDIDATE_COOLDOWN_SEC,
    min_time_to_end_sec=MIN_TIME_TO_END_SEC,
)

val_stage1_ds, val_stage1_outputs, val_candidates = generate_stage2_candidates_from_stage1(
    file_paths=val_files,
    checkpoint_path=STAGE1_CHECKPOINT,
    split_name="val",
    p_hot_threshold=STAGE1_P_HOT_THRESHOLD,
    min_entry_spread=STAGE2_MIN_ENTRY_SPREAD,
    candidate_cooldown_sec=STAGE2_CANDIDATE_COOLDOWN_SEC,
    min_time_to_end_sec=MIN_TIME_TO_END_SEC,
)

In [ ]:
# ============================================================
# FORCED ONE-SLOT LABELER
# ============================================================

def _clone_book_engine(base_engine):
    """
    Clone only the external book state.
    No active simulated orders are copied.
    """
    e = MultiOrderKalshiMatchingEngine()
    e.resting_bids = dict(base_engine.resting_bids)
    e.resting_asks = dict(base_engine.resting_asks)
    e.snapshot_count = base_engine.snapshot_count
    e.empty_snapshot_count = base_engine.empty_snapshot_count
    return e


def _raw_path_for_ticker(ticker, raw_local_dir=RAW_LOCAL_DIR):
    clean_ticker = ticker.replace("dense_", "").replace(".parquet", "")
    return os.path.join(raw_local_dir, f"clean_raw_{clean_ticker}.parquet")


def simulate_forced_q1_slot_from_state(
    base_engine,
    raw_records,
    start_pos,
    current_ts,
    candidate_row,
    min_entry_spread=0.02,
    latency_penalty=0.020,
    whipsaw_timeout=30.0,
    peg_cycle_sec=3.0,
    ttl_puke_sec=60.0,
    min_spread_bailout=0.01,
):
    """
    Independent label:
      Clone current external book state.
      Submit one Q=1 maker-maker slot now.
      Run forward until terminal.
    """

    sim_engine = _clone_book_engine(base_engine)

    best_bid = sim_engine.get_best_bid()
    best_ask = sim_engine.get_best_ask()

    base = dict(candidate_row)
    base.update({
        "actual_entry_ts": current_ts,
        "actual_entry_bid": best_bid,
        "actual_entry_ask": best_ask,
        "actual_entry_spread": None if (best_bid is None or best_ask is None) else round(float(best_ask) - float(best_bid), 2),
    })

    if best_bid is None or best_ask is None or (float(best_ask) - float(best_bid)) < min_entry_spread:
        rec = {
            **base,
            "outcome": "Skipped - Vacuum or Spread < 2c",
            "gross_pnl": 0.0,
            "good_entry": 0,
            "clean_win": 0,
            "rescue_or_loss": 0,
            "dead_no_fill": 0,
            "label_actionable": 0,
        }
        return rec

    slot_id = 1
    signal = {
        "ticker": candidate_row["ticker"],
        "predicted_side": "maker",
        "confidence": candidate_row.get("p_hot", np.nan),
        "local_recv_ts": candidate_row["local_recv_ts"],
    }

    yes_id = sim_engine.submit_limit_order(
        side="yes",
        price=best_bid,
        arrival_ts=current_ts,
        signal_ts=float(candidate_row["local_recv_ts"]),
        volume=1.0,
        slot_id=slot_id,
        role="entry_yes",
    )

    no_id = sim_engine.submit_limit_order(
        side="no",
        price=best_ask,
        arrival_ts=current_ts,
        signal_ts=float(candidate_row["local_recv_ts"]),
        volume=1.0,
        slot_id=slot_id,
        role="entry_no",
    )

    slot = Q1MakerMakerSlot(
        slot_id=slot_id,
        signal=signal,
        yes_order_id=yes_id,
        no_order_id=no_id,
        start_ts=current_ts,
        whipsaw_timeout=whipsaw_timeout,
        peg_cycle_sec=peg_cycle_sec,
        ttl_puke_sec=ttl_puke_sec,
        min_spread_bailout=min_spread_bailout,
    )

    # No fill can occur on the same event row because the order arrives after process_tick.
    slot.update(current_ts, sim_engine)

    terminal_rec = None

    for k in range(start_pos + 1, len(raw_records)):
        row = raw_records[k]
        row_ts = float(row["local_recv_ts"])

        sim_engine.process_tick(row)
        rec = slot.update(row_ts, sim_engine)

        if rec is not None:
            terminal_rec = rec
            break

    if terminal_rec is None:
        terminal_rec = {
            "outcome": "Label Incomplete - Raw End",
            "gross_pnl": 0.0,
        }

    gross = float(terminal_rec.get("gross_pnl", 0.0))
    outcome = str(terminal_rec.get("outcome", ""))

    rec = {
        **base,
        **terminal_rec,
        "gross_pnl": gross,
        "good_entry": int(gross >= 0.02),
        "clean_win": int(outcome == "Win - Full Whipsaw Arb"),
        "rescue_or_loss": int(("Bailout" in outcome) or ("Peg Hit" in outcome) or ("Timeout" in outcome)),
        "dead_no_fill": int("Dead Market" in outcome),
        "label_actionable": int(not outcome.startswith("Skipped") and not outcome.startswith("Label Incomplete")),
    }

    return rec


def label_stage2_candidates(
    candidates_df,
    split_name,
    raw_local_dir=RAW_LOCAL_DIR,
    output_path=None,
    min_entry_spread=0.02,
    latency_penalty=0.020,
    max_candidates=None,
):
    """
    Labels candidate rows independently using forced one-slot simulation.

    The base engine is replayed once per ticker to get the current external book state.
    Each candidate is then simulated independently on a cloned book state.
    """

    print("\n" + "=" * 90)
    print(f"LABELING STAGE 2 CANDIDATES: {split_name}")
    print("=" * 90)

    cand = candidates_df.copy().sort_values(["ticker", "local_recv_ts"]).reset_index(drop=True)

    if max_candidates is not None and len(cand) > max_candidates:
        print(f"[INFO] Sampling {max_candidates} candidates from {len(cand)} for first pass.")
        cand = cand.sample(n=max_candidates, random_state=42).sort_values(["ticker", "local_recv_ts"]).reset_index(drop=True)

    all_labels = []

    for ticker, g in tqdm(cand.groupby("ticker", sort=False), desc=f"Labeling {split_name} by ticker"):
        raw_path = _raw_path_for_ticker(ticker, raw_local_dir)

        if not os.path.exists(raw_path):
            print(f"[WARN] Missing raw file for {ticker}: {raw_path}")
            continue

        df_raw = pd.read_parquet(raw_path).copy()

        sort_cols = ["local_recv_ts"]
        if "seq" in df_raw.columns:
            sort_cols.append("seq")

        df_raw = df_raw.sort_values(sort_cols, kind="mergesort").reset_index(drop=True)
        raw_records = df_raw.to_dict("records")

        base_engine = MultiOrderKalshiMatchingEngine()
        raw_idx = 0
        n_raw = len(raw_records)

        for cand_row in g.to_dict("records"):
            due_ts = float(cand_row["local_recv_ts"]) + latency_penalty

            # Advance base engine until first raw event at/after due_ts.
            while raw_idx < n_raw and float(raw_records[raw_idx]["local_recv_ts"]) < due_ts:
                base_engine.process_tick(raw_records[raw_idx])
                raw_idx += 1

            if raw_idx >= n_raw:
                rec = {
                    **cand_row,
                    "outcome": "Label Incomplete - No Raw After Due",
                    "gross_pnl": 0.0,
                    "good_entry": 0,
                    "clean_win": 0,
                    "rescue_or_loss": 0,
                    "dead_no_fill": 0,
                    "label_actionable": 0,
                    "actual_entry_ts": np.nan,
                    "actual_entry_bid": np.nan,
                    "actual_entry_ask": np.nan,
                    "actual_entry_spread": np.nan,
                }
                all_labels.append(rec)
                continue

            # Match old master-loop semantics:
            # process current row first, then react at this row's clock.
            base_engine.process_tick(raw_records[raw_idx])
            current_ts = float(raw_records[raw_idx]["local_recv_ts"])
            start_pos = raw_idx
            raw_idx += 1

            rec = simulate_forced_q1_slot_from_state(
                base_engine=base_engine,
                raw_records=raw_records,
                start_pos=start_pos,
                current_ts=current_ts,
                candidate_row=cand_row,
                min_entry_spread=min_entry_spread,
                latency_penalty=latency_penalty,
            )

            all_labels.append(rec)

    labels = pd.DataFrame(all_labels)

    if output_path is None:
        output_path = os.path.join(STAGE2_DIR, f"stage2_labels_{split_name}.parquet")

    labels.to_parquet(output_path, index=False)

    print(f"\nSaved Stage 2 labels: {output_path}")
    print(f"Rows: {len(labels)}")

    return labels


train_labels = label_stage2_candidates(
    train_candidates,
    split_name="train",
    raw_local_dir=RAW_LOCAL_DIR,
    output_path=os.path.join(STAGE2_DIR, "stage2_labels_train.parquet"),
    min_entry_spread=STAGE2_MIN_ENTRY_SPREAD,
    max_candidates=None,  # set to e.g. 5000 for a faster smoke test
)

val_labels = label_stage2_candidates(
    val_candidates,
    split_name="val",
    raw_local_dir=RAW_LOCAL_DIR,
    output_path=os.path.join(STAGE2_DIR, "stage2_labels_val.parquet"),
    min_entry_spread=STAGE2_MIN_ENTRY_SPREAD,
    max_candidates=None,
)

In [ ]:
def audit_stage2_labels(df, name="stage2"):
    print("\n" + "=" * 90)
    print(f"STAGE 2 LABEL AUDIT: {name}")
    print("=" * 90)

    if df.empty:
        print("Empty label dataframe.")
        return

    d = df.copy()
    d["gross_pnl"] = pd.to_numeric(d["gross_pnl"], errors="coerce").fillna(0.0)
    d["good_entry"] = pd.to_numeric(d["good_entry"], errors="coerce").fillna(0).astype(int)

    print("Rows:", len(d))
    print("Good entries:", int(d["good_entry"].sum()))
    print("Good rate:", round(d["good_entry"].mean(), 4))
    print("Total gross if trade all candidates:", round(d["gross_pnl"].sum(), 4))
    print("Mean gross if trade all candidates:", round(d["gross_pnl"].mean(), 5))

    print("\nOutcome counts:")
    print(d["outcome"].value_counts(dropna=False).to_string())

    print("\nPnL by outcome:")
    print(
        d.groupby("outcome")["gross_pnl"]
        .agg(["count", "sum", "mean"])
        .sort_values("sum", ascending=False)
        .to_string()
    )

    print("\nGross PnL quantiles:")
    print(d["gross_pnl"].quantile([0, 0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99, 1.0]))

    if "p_hot" in d.columns:
        print("\nGood rate / PnL by p_hot bucket:")
        d["p_hot_bucket"] = pd.cut(
            d["p_hot"],
            bins=[-np.inf, 0.5, 0.6, 0.7, 0.8, 0.9, np.inf],
            labels=["<=.5", ".5-.6", ".6-.7", ".7-.8", ".8-.9", ">.9"],
        )
        print(
            d.groupby("p_hot_bucket", observed=False)
            .agg(
                n=("good_entry", "size"),
                good_rate=("good_entry", "mean"),
                gross_sum=("gross_pnl", "sum"),
                gross_mean=("gross_pnl", "mean"),
                mean_spread=("spread", "mean"),
                mean_actual_spread=("actual_entry_spread", "mean"),
            )
            .reset_index()
            .to_string(index=False)
        )

    if "actual_entry_spread" in d.columns:
        print("\nGood rate / PnL by actual entry spread:")
        d["actual_spread_bucket"] = pd.cut(
            d["actual_entry_spread"],
            bins=[-np.inf, 0.019, 0.029, 0.039, 0.059, 0.099, np.inf],
            labels=["<2c", "2c", "3c", "4-5c", "6-9c", ">=10c"],
        )
        print(
            d.groupby("actual_spread_bucket", observed=False)
            .agg(
                n=("good_entry", "size"),
                good_rate=("good_entry", "mean"),
                gross_sum=("gross_pnl", "sum"),
                gross_mean=("gross_pnl", "mean"),
            )
            .reset_index()
            .to_string(index=False)
        )

    print("\nBy ticker:")
    by_ticker = (
        d.groupby("ticker")
        .agg(
            n=("good_entry", "size"),
            good_rate=("good_entry", "mean"),
            gross_sum=("gross_pnl", "sum"),
            gross_mean=("gross_pnl", "mean"),
        )
        .sort_values("gross_sum", ascending=False)
        .head(20)
    )
    print(by_ticker.to_string())

    oracle = d[d["good_entry"] == 1]
    print("\nOracle candidate selector:")
    print("Oracle trades:", len(oracle))
    print("Oracle gross:", round(oracle["gross_pnl"].sum(), 4))
    print("Oracle gross/trade:", round(oracle["gross_pnl"].mean(), 5) if len(oracle) else np.nan)


audit_stage2_labels(train_labels, "TRAIN")
audit_stage2_labels(val_labels, "VAL")

In [ ]:
class Stage2EntryDataset(Dataset):
    def __init__(self, base_stage1_dataset, labels_df):
        self.base = base_stage1_dataset
        self.labels_df = labels_df.copy().reset_index(drop=True)

        if "sample_pos" not in self.labels_df.columns:
            raise ValueError("labels_df must contain sample_pos from Stage1 inference dataset.")

        self.sample_pos = self.labels_df["sample_pos"].astype(int).to_numpy()
        self.y = self.labels_df["good_entry"].astype(int).to_numpy()

        self.gross_pnl = self.labels_df["gross_pnl"].astype(float).to_numpy()
        self.outcome = self.labels_df["outcome"].astype(str).to_numpy()

    def __len__(self):
        return len(self.labels_df)

    def __getitem__(self, idx):
        sample_pos = int(self.sample_pos[idx])
        current_idx = int(self.base.indices[sample_pos])

        x = self.base.features_array[current_idx - self.base.seq_len : current_idx]
        x = torch.tensor(x, dtype=torch.float32)

        y = torch.tensor(int(self.y[idx]), dtype=torch.long)

        return x, y, torch.tensor(idx, dtype=torch.long)


train_stage2_dataset = Stage2EntryDataset(train_stage1_ds, train_labels)
val_stage2_dataset = Stage2EntryDataset(val_stage1_ds, val_labels)

train_stage2_loader = DataLoader(
    train_stage2_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
)

val_stage2_loader = DataLoader(
    val_stage2_dataset,
    batch_size=256,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

print("Stage2 train candidates:", len(train_stage2_dataset))
print("Stage2 val candidates:", len(val_stage2_dataset))

print("Train good rate:", train_labels["good_entry"].mean())
print("Val good rate:", val_labels["good_entry"].mean())

In [ ]:
# ============================================================
# STAGE 2 TRAIN LOOP
# ============================================================

STAGE2_SAVE_DIR = os.path.join(STAGE2_DIR, "checkpoints")
os.makedirs(STAGE2_SAVE_DIR, exist_ok=True)

stage2_model = KalshiTransformer(
    feature_dim=23,
    d_model=64,
    nhead=4,
    num_layers=2,
    num_classes=2,
).to(device)

optimizer = torch.optim.AdamW(
    stage2_model.parameters(),
    lr=1e-4,
    weight_decay=1e-5,
)

labels_np = train_labels["good_entry"].astype(int).to_numpy()
counts = np.bincount(labels_np, minlength=2)

if np.any(counts == 0):
    raise ValueError(f"Stage2 class missing. Counts: {counts}")

# Mild class weighting. If positives are rare, this helps, but do not go insane.
weights = counts.sum() / (2.0 * counts)
weights = np.minimum(weights, np.array([5.0, 5.0]))

criterion = nn.CrossEntropyLoss(
    weight=torch.tensor(weights, dtype=torch.float32).to(device)
)

print("Stage2 class counts:", counts)
print("Stage2 class weights:", weights)


def evaluate_stage2_model(model, loader, labels_df, name="val"):
    model.eval()

    all_targets = []
    all_preds = []
    all_p_good = []
    all_row_idx = []

    val_loss = 0.0

    with torch.no_grad():
        for x, y, row_idx in tqdm(loader, desc=f"Evaluating Stage2 {name}", leave=False):
            x = x.to(device)
            y = y.to(device)

            logits = model(x)
            loss = criterion(logits, y)
            val_loss += loss.item()

            probs = torch.softmax(logits, dim=1)
            p_good = probs[:, 1]
            preds = torch.argmax(probs, dim=1)

            all_targets.extend(y.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_p_good.extend(p_good.cpu().numpy())
            all_row_idx.extend(row_idx.cpu().numpy())

    avg_loss = val_loss / max(1, len(loader))

    result_df = labels_df.iloc[np.asarray(all_row_idx)].copy().reset_index(drop=True)
    result_df["stage2_p_good"] = np.asarray(all_p_good)
    result_df["stage2_pred"] = np.asarray(all_preds)
    result_df["stage2_target"] = np.asarray(all_targets)

    print("\n" + "=" * 90)
    print(f"STAGE2 EVAL: {name}")
    print("=" * 90)
    print("Loss:", round(avg_loss, 5))

    print("\nConfusion matrix:")
    print(confusion_matrix(all_targets, all_preds, labels=[0, 1]))

    print("\nClassification report:")
    print(classification_report(all_targets, all_preds, labels=[0, 1], target_names=["bad", "good"], zero_division=0))

    print("\nStage2 p_good quantiles:")
    print(result_df["stage2_p_good"].quantile([0, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99, 1.0]))

    print("\nThreshold approval table:")
    rows = []
    for thr in [0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]:
        sel = result_df[result_df["stage2_p_good"] >= thr]
        if len(sel) == 0:
            rows.append({
                "thr": thr,
                "n": 0,
                "good_rate": np.nan,
                "gross_sum": 0.0,
                "gross_mean": np.nan,
                "wins": 0,
                "rescue": 0,
            })
            continue

        rows.append({
            "thr": thr,
            "n": len(sel),
            "good_rate": sel["good_entry"].mean(),
            "gross_sum": sel["gross_pnl"].sum(),
            "gross_mean": sel["gross_pnl"].mean(),
            "wins": int((sel["outcome"] == "Win - Full Whipsaw Arb").sum()),
            "rescue": int(sel["outcome"].astype(str).str.contains("Bailout|Peg Hit|Timeout", na=False).sum()),
        })

    table = pd.DataFrame(rows)
    print(table.to_string(index=False))

    print("\nTop-N approval table:")
    ranked = result_df.sort_values("stage2_p_good", ascending=False).reset_index(drop=True)
    rows = []
    for n in [50, 100, 200, 300, 500, 700, 1000, 1500, 2000]:
        if n > len(ranked):
            continue
        sel = ranked.head(n)
        rows.append({
            "top_n": n,
            "good_rate": sel["good_entry"].mean(),
            "gross_sum": sel["gross_pnl"].sum(),
            "gross_mean": sel["gross_pnl"].mean(),
            "p_good_cutoff": sel["stage2_p_good"].min(),
            "wins": int((sel["outcome"] == "Win - Full Whipsaw Arb").sum()),
            "rescue": int(sel["outcome"].astype(str).str.contains("Bailout|Peg Hit|Timeout", na=False).sum()),
        })

    top_table = pd.DataFrame(rows)
    print(top_table.to_string(index=False))

    # Baseline: trade all Stage2 candidates
    print("\nTrade-all-candidates baseline:")
    print("n:", len(result_df))
    print("gross:", round(result_df["gross_pnl"].sum(), 4))
    print("gross/candidate:", round(result_df["gross_pnl"].mean(), 5))

    oracle = result_df[result_df["good_entry"] == 1]
    print("\nOracle baseline:")
    print("oracle n:", len(oracle))
    print("oracle gross:", round(oracle["gross_pnl"].sum(), 4))
    print("oracle gross/trade:", round(oracle["gross_pnl"].mean(), 5) if len(oracle) else np.nan)

    return avg_loss, result_df


EPOCHS_STAGE2 = 5
best_val_gross = -1e9
best_path = os.path.join(STAGE2_SAVE_DIR, "best_stage2_by_val_gross.pth")

for epoch in range(1, EPOCHS_STAGE2 + 1):
    print("\n" + "=" * 90)
    print(f"STAGE2 EPOCH {epoch} / {EPOCHS_STAGE2}")
    print("=" * 90)

    stage2_model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for x, y, row_idx in tqdm(train_stage2_loader, desc="Training Stage2", leave=False):
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = stage2_model(x)
        loss = criterion(logits, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(stage2_model.parameters(), max_norm=1.0)
        optimizer.step()

        train_loss += loss.item()
        preds = torch.argmax(logits, dim=1)
        train_correct += (preds == y).sum().item()
        train_total += y.numel()

    avg_train_loss = train_loss / max(1, len(train_stage2_loader))
    train_acc = train_correct / max(1, train_total)

    print(f"Train loss: {avg_train_loss:.5f} | train acc: {train_acc:.4f}")

    val_loss, val_scored = evaluate_stage2_model(
        stage2_model,
        val_stage2_loader,
        val_labels,
        name=f"val epoch {epoch}",
    )

    # Simple model-selection proxy: use best gross at threshold 0.50 on labeled candidates.
    sel = val_scored[val_scored["stage2_p_good"] >= 0.50]
    val_gross_at_50 = sel["gross_pnl"].sum() if len(sel) else 0.0

    epoch_path = os.path.join(STAGE2_SAVE_DIR, f"stage2_epoch_{epoch}.pth")
    torch.save(stage2_model.state_dict(), epoch_path)
    print("Saved:", epoch_path)

    if val_gross_at_50 > best_val_gross:
        best_val_gross = val_gross_at_50
        torch.save(stage2_model.state_dict(), best_path)
        print(f"[+] New best val gross@0.50: {best_val_gross:.4f}. Saved {best_path}")

print("\nStage2 training complete.")
print("Best checkpoint:", best_path)

In [ ]:
def write_stage2_approved_predictions(
    scored_df,
    threshold=0.50,
    output_path="/content/stage2_approved_predictions.parquet",
):
    df = scored_df[scored_df["stage2_p_good"] >= threshold].copy()

    if df.empty:
        print("No approved predictions.")
        return df

    df = df.sort_values("local_recv_ts").reset_index(drop=True)

    out = pd.DataFrame({
        "ticker": df["ticker"],
        "ts": df["ts_exchange"] if "ts_exchange" in df.columns else df["local_recv_ts"],
        "ts_exchange": df["ts_exchange"] if "ts_exchange" in df.columns else df["local_recv_ts"],
        "local_recv_ts": df["local_recv_ts"],
        "predicted_side": "maker",
        "confidence": df["stage2_p_good"],
        "p_hot": df["p_hot"],
        "stage2_p_good": df["stage2_p_good"],
    })

    out.to_parquet(output_path, index=False)

    print("Saved Stage2-approved predictions:", output_path)
    print("Rows:", len(out))
    print("Unique tickers:", out["ticker"].nunique())
    print("stage2_p_good cutoff:", threshold)

    return out


# Re-evaluate best checkpoint if desired.
best_stage2 = KalshiTransformer(
    feature_dim=23,
    d_model=64,
    nhead=4,
    num_layers=2,
    num_classes=2,
).to(device)

best_stage2.load_state_dict(torch.load(best_path, map_location=device))
best_stage2.eval()

_, val_scored_best = evaluate_stage2_model(
    best_stage2,
    val_stage2_loader,
    val_labels,
    name="val best checkpoint",
)

approved = write_stage2_approved_predictions(
    val_scored_best,
    threshold=0.50,
    output_path=os.path.join(STAGE2_DIR, "stage2_approved_val_thr0p50.parquet"),
)

In [ ]:
df_stage2_sim = run_multislot_zero_assumption_sim(
    predictions_path=os.path.join(STAGE2_DIR, "stage2_approved_val_thr0p50.parquet"),
    raw_local_dir=RAW_LOCAL_DIR,
    output_audit_path=os.path.join(STAGE2_DIR, "receipts_stage2_val_thr0p50.parquet"),
    max_active_slots=1,
    entry_cooldown_sec=1.0,
    min_entry_spread=0.04,
    log_slot_skips=True,
)

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ============================================================
# CONFIG: set these to the exact old/live weights you want tested
# ============================================================

STAGE1_CHECKPOINT_TO_TEST = "/content/stage1_weights.pth"
STAGE2_CHECKPOINT_TO_TEST = "/content/stage2_weights.pth"

RUN_TAG = "live_weights_apr27"
STAGE2_TEST_DIR = f"/content/stage2_weight_tests/{RUN_TAG}"
os.makedirs(STAGE2_TEST_DIR, exist_ok=True)

P_HOT_CANDIDATE_THRESHOLD = 0.50      # broad candidate generation
P_HOT_POLICY_THRESHOLD = 0.60         # live policy gate
P_GOOD_POLICY_THRESHOLD = 0.60        # live policy gate
MIN_SIGNAL_SPREAD = 0.04              # current dense signal spread filter
EXECUTOR_MIN_ENTRY_SPREAD = 0.04      # simulator live book spread filter


# ============================================================
# Stage 2 scorer dataset
# ============================================================

class Stage2CandidateScoreDataset(Dataset):
    def __init__(self, base_stage1_dataset, candidates_df):
        self.base = base_stage1_dataset
        self.df = candidates_df.copy().reset_index(drop=True)

        if "sample_pos" not in self.df.columns:
            raise ValueError("candidates_df must contain sample_pos from Stage 1 candidate generation.")

        self.sample_pos = self.df["sample_pos"].astype(int).to_numpy()

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        sample_pos = int(self.sample_pos[idx])
        current_idx = int(self.base.indices[sample_pos])

        x = self.base.features_array[current_idx - self.base.seq_len : current_idx]
        x = torch.tensor(x, dtype=torch.float32)

        return x, torch.tensor(idx, dtype=torch.long)


def load_binary_transformer(checkpoint_path):
    model = KalshiTransformer(
        feature_dim=23,
        d_model=64,
        nhead=4,
        num_layers=2,
        num_classes=2,
    ).to(device)

    try:
        state = torch.load(checkpoint_path, map_location=device, weights_only=True)
    except TypeError:
        state = torch.load(checkpoint_path, map_location=device)

    model.load_state_dict(state)
    model.eval()
    return model


def score_stage2_candidates_with_checkpoint(
    base_stage1_dataset,
    candidates_df,
    stage2_checkpoint_path,
    batch_size=512,
):
    model = load_binary_transformer(stage2_checkpoint_path)

    ds = Stage2CandidateScoreDataset(base_stage1_dataset, candidates_df)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)

    p_good = np.full(len(candidates_df), np.nan, dtype=float)

    with torch.no_grad():
        for x, row_idx in tqdm(loader, desc="Scoring Stage2 candidates"):
            x = x.to(device)
            logits = model(x)
            probs = torch.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()
            idx = row_idx.cpu().numpy()
            p_good[idx] = probs

    out = candidates_df.copy().reset_index(drop=True)
    out["stage2_p_good"] = p_good

    return out


def write_twostage_policy_predictions(
    scored_df,
    output_path,
    p_hot_min=0.60,
    p_good_min=0.60,
    min_signal_spread=0.04,
):
    df = scored_df[
        (scored_df["p_hot"] >= p_hot_min)
        & (scored_df["stage2_p_good"] >= p_good_min)
        & (scored_df["spread"] >= min_signal_spread)
    ].copy()

    df = df.sort_values("local_recv_ts").reset_index(drop=True)

    if df.empty:
        print("No approved predictions.")
        empty = pd.DataFrame(columns=[
            "ticker", "ts", "ts_exchange", "local_recv_ts",
            "predicted_side", "confidence", "p_hot", "stage2_p_good", "spread"
        ])
        empty.to_parquet(output_path, index=False)
        return empty

    pred = pd.DataFrame({
        "ticker": df["ticker"],
        "ts": df["ts_exchange"] if "ts_exchange" in df.columns else df["local_recv_ts"],
        "ts_exchange": df["ts_exchange"] if "ts_exchange" in df.columns else df["local_recv_ts"],
        "local_recv_ts": df["local_recv_ts"],
        "predicted_side": "maker",
        "confidence": df["stage2_p_good"],
        "p_hot": df["p_hot"],
        "stage2_p_good": df["stage2_p_good"],
        "spread": df["spread"],
    })

    pred.to_parquet(output_path, index=False)

    print("Saved predictions:", output_path)
    print("Rows:", len(pred))
    print("Unique tickers:", pred["ticker"].nunique())
    print("p_hot_min:", p_hot_min)
    print("p_good_min:", p_good_min)
    print("min_signal_spread:", min_signal_spread)

    print("\nPrediction score quantiles:")
    print(pred[["p_hot", "stage2_p_good", "spread"]].quantile([0, .25, .5, .75, .9, .99, 1.0]))

    return pred

In [ ]:
# ============================================================
# 1. Generate Stage 1 candidates using the checkpoint under test
# ============================================================

test_stage1_ds, test_stage1_outputs, test_candidates = generate_stage2_candidates_from_stage1(
    file_paths=val_files,
    checkpoint_path=STAGE1_CHECKPOINT_TO_TEST,
    split_name=RUN_TAG,
    p_hot_threshold=P_HOT_CANDIDATE_THRESHOLD,
    min_entry_spread=0.02,              # broad; policy filter below handles 4c
    candidate_cooldown_sec=1.0,
    min_time_to_end_sec=100.0,
    batch_size=512,
)

# ============================================================
# 2. Score candidates with Stage 2 checkpoint under test
# ============================================================

test_scored = score_stage2_candidates_with_checkpoint(
    base_stage1_dataset=test_stage1_ds,
    candidates_df=test_candidates,
    stage2_checkpoint_path=STAGE2_CHECKPOINT_TO_TEST,
    batch_size=512,
)

scored_path = os.path.join(STAGE2_TEST_DIR, f"scored_{RUN_TAG}.parquet")
test_scored.to_parquet(scored_path, index=False)
print("Saved scored candidates:", scored_path)

print("\nScored candidate quantiles:")
print(test_scored[["p_hot", "stage2_p_good", "spread"]].quantile([0, .25, .5, .75, .9, .99, 1.0]))

# ============================================================
# 3. Write live-policy prediction parquet
# ============================================================

pred_path = os.path.join(
    STAGE2_TEST_DIR,
    f"preds_{RUN_TAG}_pHot{P_HOT_POLICY_THRESHOLD}_pGood{P_GOOD_POLICY_THRESHOLD}_spread{EXECUTOR_MIN_ENTRY_SPREAD}.parquet"
)

approved = write_twostage_policy_predictions(
    scored_df=test_scored,
    output_path=pred_path,
    p_hot_min=P_HOT_POLICY_THRESHOLD,
    p_good_min=P_GOOD_POLICY_THRESHOLD,
    min_signal_spread=MIN_SIGNAL_SPREAD,
)

# ============================================================
# 4. Run current fresh-slot simulator
# ============================================================

receipts_path = os.path.join(STAGE2_TEST_DIR, f"receipts_{RUN_TAG}.parquet")

df_stage2_sim = run_multislot_zero_assumption_sim(
    predictions_path=pred_path,
    raw_local_dir=RAW_LOCAL_DIR,
    output_audit_path=receipts_path,
    max_active_slots=1,
    entry_cooldown_sec=1.0,
    min_entry_spread=EXECUTOR_MIN_ENTRY_SPREAD,
    log_slot_skips=True,
)

print("Saved sim receipts:", receipts_path)

In [ ]:
import pandas as pd
import numpy as np

def audit_worst_stage2_labels(labels_df, n=40):
    df = labels_df.copy()
    df["gross_pnl"] = pd.to_numeric(df["gross_pnl"], errors="coerce").fillna(0.0)

    cols = [
        "ticker",
        "local_recv_ts",
        "p_hot",
        "stage2_p_good",
        "spread",
        "actual_entry_spread",
        "actual_entry_bid",
        "actual_entry_ask",
        "outcome",
        "gross_pnl",
        "entry_price",
        "exit_price",
        "time_from_signal_to_first_fill",
        "time_in_trade",
    ]
    cols = [c for c in cols if c in df.columns]

    worst = df.sort_values("gross_pnl").head(n)

    print("\n=== WORST STAGE2 LABELS ===")
    print(worst[cols].to_string(index=False))

    print("\n=== WORST BY OUTCOME ===")
    print(
        df.groupby("outcome")
        .agg(
            n=("gross_pnl", "size"),
            gross_sum=("gross_pnl", "sum"),
            gross_mean=("gross_pnl", "mean"),
            min_gross=("gross_pnl", "min"),
            mean_p_hot=("p_hot", "mean"),
            mean_p_good=("stage2_p_good", "mean"),
            mean_spread=("actual_entry_spread", "mean"),
        )
        .sort_values("gross_sum")
        .to_string()
    )

    print("\n=== WORST BY TICKER ===")
    print(
        df.groupby("ticker")
        .agg(
            n=("gross_pnl", "size"),
            gross_sum=("gross_pnl", "sum"),
            gross_mean=("gross_pnl", "mean"),
            min_gross=("gross_pnl", "min"),
            mean_p_hot=("p_hot", "mean"),
            mean_p_good=("stage2_p_good", "mean"),
            mean_spread=("actual_entry_spread", "mean"),
        )
        .sort_values("gross_sum")
        .head(20)
        .to_string()
    )

    return worst

worst_apr24 = audit_worst_stage2_labels(val_scored_best, n=40)

In [ ]:
import os
import pandas as pd
import numpy as np

APR24_CAP_DIR = "/content/stage2_first_pass/apr24_cap_sweeps"
os.makedirs(APR24_CAP_DIR, exist_ok=True)

def write_filtered_preds(scored_df, name, p_hot_min=0.50, p_good_min=None, min_signal_spread=0.04, max_signal_spread=None):
    mask = (
        (scored_df["p_hot"] >= p_hot_min)
        & (scored_df["spread"] >= min_signal_spread)
    )

    if p_good_min is not None:
        mask &= scored_df["stage2_p_good"] >= p_good_min

    if max_signal_spread is not None:
        mask &= scored_df["spread"] <= max_signal_spread

    df = scored_df[mask].copy().sort_values("local_recv_ts").reset_index(drop=True)

    out = pd.DataFrame({
        "ticker": df["ticker"],
        "ts": df["ts_exchange"] if "ts_exchange" in df.columns else df["local_recv_ts"],
        "ts_exchange": df["ts_exchange"] if "ts_exchange" in df.columns else df["local_recv_ts"],
        "local_recv_ts": df["local_recv_ts"],
        "predicted_side": "maker",
        "confidence": df["stage2_p_good"],
        "p_hot": df["p_hot"],
        "stage2_p_good": df["stage2_p_good"],
        "spread": df["spread"],
    })

    path = os.path.join(APR24_CAP_DIR, f"preds_{name}.parquet")
    out.to_parquet(path, index=False)

    print(f"{name}: {len(out)} signals")
    return path

def summarize_receipts_for_policy(rec, name, p_hot_min, p_good_min, min_signal_spread, max_signal_spread, exec_min_spread, max_slots):
    rec = rec.copy()
    rec["outcome"] = rec["outcome"].astype(str)
    rec["gross_pnl"] = pd.to_numeric(rec["gross_pnl"], errors="coerce").fillna(0.0)

    actionable = rec[~rec["outcome"].str.contains("Skipped", na=False)]

    return {
        "policy": name,
        "p_hot_min": p_hot_min,
        "p_good_min": p_good_min,
        "min_signal_spread": min_signal_spread,
        "max_signal_spread": max_signal_spread,
        "exec_min_spread": exec_min_spread,
        "max_slots": max_slots,
        "signals_logged": len(rec),
        "actionable": len(actionable),
        "gross_pnl": rec["gross_pnl"].sum(),
        "gross_per_actionable": actionable["gross_pnl"].mean() if len(actionable) else np.nan,
        "wins": int((rec["outcome"] == "Win - Full Whipsaw Arb").sum()),
        "rescue": int(rec["outcome"].str.contains("Bailout|Peg Hit|Timeout", na=False).sum()),
        "dead": int(rec["outcome"].str.contains("Dead Market", na=False).sum()),
        "skipped_max_slots": int((rec["outcome"] == "Skipped - Max Active Slots").sum()),
        "skipped_spread": int((rec["outcome"] == "Skipped - Vacuum or Spread < 2c").sum()),
    }

rows = []

for p_hot_min in [0.6]:
    for p_good_min in [0.60]:
        for min_sig_spread in [0.04]:
            for max_sig_spread in [None]:
                for max_slots in [3, 20]:
                    name = (
                        f"pHot{str(p_hot_min).replace('.', 'p')}"
                        f"_pGood{str(p_good_min).replace('.', 'p') if p_good_min is not None else 'None'}"
                        f"_sigMin{int(min_sig_spread*100)}c"
                        f"_sigMax{int(max_sig_spread*100)}c" if max_sig_spread is not None else
                        f"pHot{str(p_hot_min).replace('.', 'p')}"
                        f"_pGood{str(p_good_min).replace('.', 'p') if p_good_min is not None else 'None'}"
                        f"_sigMin{int(min_sig_spread*100)}c_sigMaxNone"
                    )

                    pred_path = write_filtered_preds(
                        val_scored_best,
                        name=name,
                        p_hot_min=p_hot_min,
                        p_good_min=p_good_min,
                        min_signal_spread=min_sig_spread,
                        max_signal_spread=max_sig_spread,
                    )

                    if pd.read_parquet(pred_path).empty:
                        continue

                    # Executor min spread matches signal min spread for live-safe behavior.
                    rec = run_multislot_zero_assumption_sim(
                        predictions_path=pred_path,
                        raw_local_dir=RAW_LOCAL_DIR,
                        output_audit_path=os.path.join(APR24_CAP_DIR, f"receipts_{name}.parquet"),
                        max_active_slots=max_slots,
                        entry_cooldown_sec=1.0,
                        min_entry_spread=min_sig_spread,
                        log_slot_skips=True,
                    )

                    rows.append(
                        summarize_receipts_for_policy(
                            rec=rec,
                            name=name,
                            p_hot_min=p_hot_min,
                            p_good_min=p_good_min,
                            min_signal_spread=min_sig_spread,
                            max_signal_spread=max_sig_spread,
                            exec_min_spread=min_sig_spread,
                            max_slots=max_slots,
                        )
                    )

cap_summary = pd.DataFrame(rows).sort_values(["gross_pnl", "gross_per_actionable"], ascending=False)
cap_path = os.path.join(APR24_CAP_DIR, "apr24_cap_sweep_summary.csv")
cap_summary.to_csv(cap_path, index=False)

print("\n=== APR27 CAP SWEEP SUMMARY ===")
print(cap_summary.head(40).to_string(index=False))
print("Saved:", cap_path)

In [ ]:
import os
import numpy as np
import pandas as pd

POLICY_DIR = "/content/stage2_first_pass/policy_sweeps"
os.makedirs(POLICY_DIR, exist_ok=True)

def write_policy_predictions(scored_df, mask, name):
    df = scored_df[mask].copy().sort_values("local_recv_ts").reset_index(drop=True)

    out = pd.DataFrame({
        "ticker": df["ticker"],
        "ts": df["ts_exchange"] if "ts_exchange" in df.columns else df["local_recv_ts"],
        "ts_exchange": df["ts_exchange"] if "ts_exchange" in df.columns else df["local_recv_ts"],
        "local_recv_ts": df["local_recv_ts"],
        "predicted_side": "maker",
        "confidence": df["stage2_p_good"] if "stage2_p_good" in df.columns else df["p_hot"],
        "p_hot": df["p_hot"],
        "stage2_p_good": df["stage2_p_good"] if "stage2_p_good" in df.columns else np.nan,
        "spread": df["spread"],
        "actual_entry_spread": df["actual_entry_spread"] if "actual_entry_spread" in df.columns else np.nan,
    })

    path = os.path.join(POLICY_DIR, f"preds_{name}.parquet")
    out.to_parquet(path, index=False)

    print(f"{name}: wrote {len(out)} rows -> {path}")
    return path


def summarize_receipts(df, name):
    if df.empty:
        return {
            "policy": name,
            "signals_logged": 0,
            "actionable": 0,
            "gross_pnl": 0.0,
            "gross_per_actionable": np.nan,
            "wins": 0,
            "rescue": 0,
            "skipped_max_slots": 0,
        }

    actionable = df[~df["outcome"].astype(str).str.contains("Skipped", na=False)]
    return {
        "policy": name,
        "signals_logged": len(df),
        "actionable": len(actionable),
        "gross_pnl": float(df["gross_pnl"].sum()),
        "gross_per_actionable": float(actionable["gross_pnl"].mean()) if len(actionable) else np.nan,
        "wins": int((df["outcome"] == "Win - Full Whipsaw Arb").sum()),
        "rescue": int(df["outcome"].astype(str).str.contains("Bailout|Peg Hit|Timeout", na=False).sum()),
        "skipped_max_slots": int((df["outcome"] == "Skipped - Max Active Slots").sum()),
    }


def run_policy(name, mask):
    pred_path = write_policy_predictions(val_scored_best, mask, name)
    rec_path = os.path.join(POLICY_DIR, f"receipts_{name}.parquet")

    receipts = run_multislot_zero_assumption_sim(
        predictions_path=pred_path,
        raw_local_dir=RAW_LOCAL_DIR,
        output_audit_path=rec_path,
        max_active_slots=1,
        entry_cooldown_sec=1.0,
        min_entry_spread=0.02,
        log_slot_skips=True,
    )

    return summarize_receipts(receipts, name)


results = []

# A. trade all Stage2 candidates
results.append(run_policy(
    "all_stage2_candidates_pHot50_spread2c",
    val_scored_best["p_hot"] >= 0.50
))

# B. Stage2 thresholds
for thr in [0.60, 0.70, 0.80]:
    results.append(run_policy(
        f"stage2_pGood_ge_{str(thr).replace('.', 'p')}",
        val_scored_best["stage2_p_good"] >= thr
    ))

# C. dumb spread proxies
for spread_thr in [0.04, 0.06]:
    results.append(run_policy(
        f"spread_ge_{int(spread_thr * 100)}c",
        val_scored_best["actual_entry_spread"] >= spread_thr
    ))

# D. combined
for p_thr in [0.60, 0.70]:
    for spread_thr in [0.04, 0.06]:
        results.append(run_policy(
            f"stage2_ge_{str(p_thr).replace('.', 'p')}_spread_ge_{int(spread_thr * 100)}c",
            (val_scored_best["stage2_p_good"] >= p_thr)
            & (val_scored_best["actual_entry_spread"] >= spread_thr)
        ))

summary = pd.DataFrame(results).sort_values("gross_pnl", ascending=False)
summary_path = os.path.join(POLICY_DIR, "policy_sweep_summary.csv")
summary.to_csv(summary_path, index=False)

print("\n=== POLICY SWEEP SUMMARY ===")
print(summary.to_string(index=False))
print("Saved:", summary_path)

In [ ]:
def run_random_matched(scored_df, n_signals, name_prefix, n_repeats=10, seed0=100):
    rows = []

    for i in range(n_repeats):
        rng = np.random.default_rng(seed0 + i)
        sample_idx = rng.choice(scored_df.index.to_numpy(), size=min(n_signals, len(scored_df)), replace=False)
        mask = scored_df.index.isin(sample_idx)

        name = f"{name_prefix}_seed{i}"
        pred_path = write_policy_predictions(scored_df, mask, name)
        rec_path = os.path.join(POLICY_DIR, f"receipts_{name}.parquet")

        receipts = run_multislot_zero_assumption_sim(
            predictions_path=pred_path,
            raw_local_dir=RAW_LOCAL_DIR,
            output_audit_path=rec_path,
            max_active_slots=1,
            entry_cooldown_sec=1.0,
            min_entry_spread=0.02,
            log_slot_skips=True,
        )

        rows.append(summarize_receipts(receipts, name))

    out = pd.DataFrame(rows)
    print("\n=== RANDOM MATCHED SUMMARY ===")
    print(out.to_string(index=False))
    print("\nMean:")
    print(out[["actionable", "gross_pnl", "gross_per_actionable", "wins", "rescue", "skipped_max_slots"]].mean(numeric_only=True))
    return out


# Example: random baseline matching stage2 threshold 0.50 signal count.
n_stage2_050 = int((val_scored_best["stage2_p_good"] >= 0.50).sum())

random_050 = run_random_matched(
    val_scored_best,
    n_signals=n_stage2_050,
    name_prefix=f"random_matched_n{n_stage2_050}",
    n_repeats=10,
)

In [ ]:
import os
import pandas as pd
import numpy as np

SPREAD_EXEC_DIR = "/content/stage2_first_pass/live_safe_spread_sweeps"
os.makedirs(SPREAD_EXEC_DIR, exist_ok=True)

policy_files = {
    "all_stage2_candidates": "/content/stage2_first_pass/policy_sweeps/preds_all_stage2_candidates_pHot50_spread2c.parquet",
    "stage2_ge_0p6": "/content/stage2_first_pass/policy_sweeps/preds_stage2_pGood_ge_0p6.parquet",
    "stage2_ge_0p7": "/content/stage2_first_pass/policy_sweeps/preds_stage2_pGood_ge_0p7.parquet",
    "stage2_ge_0p8": "/content/stage2_first_pass/policy_sweeps/preds_stage2_pGood_ge_0p8.parquet",
}

rows = []

for policy_name, pred_path in policy_files.items():
    for min_spread in [0.02, 0.04, 0.06]:
        for max_slots in [1]:
            run_name = f"{policy_name}_execSpread{int(min_spread*100)}c_slots{max_slots}"
            receipt_path = os.path.join(SPREAD_EXEC_DIR, f"receipts_{run_name}.parquet")

            rec = run_multislot_zero_assumption_sim(
                predictions_path=pred_path,
                raw_local_dir=RAW_LOCAL_DIR,
                output_audit_path=receipt_path,
                max_active_slots=max_slots,
                entry_cooldown_sec=1.0,
                min_entry_spread=min_spread,
                log_slot_skips=True,
            )

            actionable = rec[~rec["outcome"].astype(str).str.contains("Skipped", na=False)]

            rows.append({
                "policy": policy_name,
                "min_entry_spread": min_spread,
                "max_slots": max_slots,
                "signals_logged": len(rec),
                "actionable": len(actionable),
                "gross_pnl": rec["gross_pnl"].sum(),
                "gross_per_actionable": actionable["gross_pnl"].mean() if len(actionable) else np.nan,
                "wins": int((rec["outcome"] == "Win - Full Whipsaw Arb").sum()),
                "rescue": int(rec["outcome"].astype(str).str.contains("Bailout|Peg Hit|Timeout", na=False).sum()),
                "skipped_max_slots": int((rec["outcome"] == "Skipped - Max Active Slots").sum()),
                "skipped_spread": int((rec["outcome"] == "Skipped - Vacuum or Spread < 2c").sum()),
            })

summary_live_safe = pd.DataFrame(rows).sort_values("gross_pnl", ascending=False)
summary_path = os.path.join(SPREAD_EXEC_DIR, "live_safe_spread_sweep_summary.csv")
summary_live_safe.to_csv(summary_path, index=False)

print(summary_live_safe.to_string(index=False))
print("Saved:", summary_path)

In [ ]:
CLEAN_POLICY_FILES = {
    "stage2_ge_0p6": "/content/stage2_first_pass/policy_sweeps/preds_stage2_pGood_ge_0p6.parquet",
    "stage2_ge_0p7": "/content/stage2_first_pass/policy_sweeps/preds_stage2_pGood_ge_0p7.parquet",
    "stage2_ge_0p8": "/content/stage2_first_pass/policy_sweeps/preds_stage2_pGood_ge_0p8.parquet",
}

rows = []

for policy_name, pred_path in CLEAN_POLICY_FILES.items():
    for min_spread in [0.04, 0.06]:
        for max_slots in [1, 2]:
            run_name = f"{policy_name}_spread{int(min_spread*100)}c_slots{max_slots}"
            receipt_path = f"/content/stage2_first_pass/slot_sweeps_receipts_{run_name}.parquet"

            rec = run_multislot_zero_assumption_sim(
                predictions_path=pred_path,
                raw_local_dir=RAW_LOCAL_DIR,
                output_audit_path=receipt_path,
                max_active_slots=max_slots,
                entry_cooldown_sec=1.0,
                min_entry_spread=min_spread,
                log_slot_skips=True,
            )

            actionable = rec[~rec["outcome"].astype(str).str.contains("Skipped", na=False)]

            rows.append({
                "policy": policy_name,
                "min_spread": min_spread,
                "max_slots": max_slots,
                "actionable": len(actionable),
                "gross_pnl": rec["gross_pnl"].sum(),
                "gross_per_actionable": actionable["gross_pnl"].mean() if len(actionable) else np.nan,
                "wins": int((rec["outcome"] == "Win - Full Whipsaw Arb").sum()),
                "rescue": int(rec["outcome"].astype(str).str.contains("Bailout|Peg Hit|Timeout", na=False).sum()),
                "skipped_max_slots": int((rec["outcome"] == "Skipped - Max Active Slots").sum()),
            })

slot_sweep = pd.DataFrame(rows).sort_values("gross_pnl", ascending=False)
print(slot_sweep.to_string(index=False))

In [ ]:
import os
import numpy as np
import pandas as pd

SWEEP_DIR = "/content/stage2_first_pass/final_policy_sweeps"
os.makedirs(SWEEP_DIR, exist_ok=True)

# These should already exist from your earlier policy sweep.
policy_files = {
    "all_stage2_candidates": "/content/stage2_first_pass/policy_sweeps/preds_all_stage2_candidates_pHot50_spread2c.parquet",
    "stage2_ge_0p6": "/content/stage2_first_pass/policy_sweeps/preds_stage2_pGood_ge_0p6.parquet",
    "stage2_ge_0p7": "/content/stage2_first_pass/policy_sweeps/preds_stage2_pGood_ge_0p7.parquet",
    "stage2_ge_0p8": "/content/stage2_first_pass/policy_sweeps/preds_stage2_pGood_ge_0p8.parquet",
}

def summarize_sim_receipts(rec, policy, min_spread, max_slots):
    if rec.empty:
        return {
            "policy": policy,
            "min_spread": min_spread,
            "max_slots": max_slots,
            "signals_logged": 0,
            "actionable": 0,
            "gross_pnl": 0.0,
            "gross_per_actionable": np.nan,
            "wins": 0,
            "rescue": 0,
            "dead": 0,
            "skipped_max_slots": 0,
            "skipped_spread": 0,
            "skipped_cooldown": 0,
        }

    out = rec.copy()
    out["outcome"] = out["outcome"].astype(str)
    out["gross_pnl"] = pd.to_numeric(out["gross_pnl"], errors="coerce").fillna(0.0)

    actionable = out[~out["outcome"].str.contains("Skipped", na=False)]

    return {
        "policy": policy,
        "min_spread": min_spread,
        "max_slots": max_slots,
        "signals_logged": len(out),
        "actionable": len(actionable),
        "gross_pnl": float(out["gross_pnl"].sum()),
        "gross_per_actionable": float(actionable["gross_pnl"].mean()) if len(actionable) else np.nan,
        "wins": int((out["outcome"] == "Win - Full Whipsaw Arb").sum()),
        "rescue": int(out["outcome"].str.contains("Bailout|Peg Hit|Timeout", na=False).sum()),
        "dead": int(out["outcome"].str.contains("Dead Market", na=False).sum()),
        "skipped_max_slots": int((out["outcome"] == "Skipped - Max Active Slots").sum()),
        "skipped_spread": int((out["outcome"] == "Skipped - Vacuum or Spread < 2c").sum()),
        "skipped_cooldown": int((out["outcome"] == "Skipped - Entry Cooldown").sum()),
    }

rows = []

for policy_name, pred_path in policy_files.items():
    for min_spread in [0.03, 0.04, 0.05, 0.06]:
        for max_slots in [1, 2]:
            run_name = f"{policy_name}_spread{int(min_spread*100)}c_slots{max_slots}"
            receipt_path = os.path.join(SWEEP_DIR, f"receipts_{run_name}.parquet")

            rec = run_multislot_zero_assumption_sim(
                predictions_path=pred_path,
                raw_local_dir=RAW_LOCAL_DIR,
                output_audit_path=receipt_path,
                max_active_slots=max_slots,
                entry_cooldown_sec=1.0,
                min_entry_spread=min_spread,
                log_slot_skips=True,
            )

            rows.append(summarize_sim_receipts(rec, policy_name, min_spread, max_slots))

summary = pd.DataFrame(rows).sort_values(["gross_pnl", "gross_per_actionable"], ascending=False)
summary_path = os.path.join(SWEEP_DIR, "summary_policy_spread_slots.csv")
summary.to_csv(summary_path, index=False)

print("\n=== POLICY / SPREAD / SLOT SWEEP ===")
print(summary.to_string(index=False))
print("Saved:", summary_path)

In [ ]:
def audit_slot_hogging(receipts_path, preds_path, name="policy"):
    rec = pd.read_parquet(receipts_path).copy()
    preds = pd.read_parquet(preds_path).copy()

    rec["signal_ts_round"] = pd.to_numeric(rec["signal_ts"], errors="coerce").round(6)
    preds["signal_ts_round"] = pd.to_numeric(preds["local_recv_ts"], errors="coerce").round(6)

    merged = rec.merge(
        preds,
        on=["ticker", "signal_ts_round"],
        how="left",
        suffixes=("", "_pred"),
    )

    merged["outcome"] = merged["outcome"].astype(str)
    merged["gross_pnl"] = pd.to_numeric(merged["gross_pnl"], errors="coerce").fillna(0.0)

    entered = merged[~merged["outcome"].str.contains("Skipped", na=False)].copy()
    skipped_slot = merged[merged["outcome"].eq("Skipped - Max Active Slots")].copy()
    skipped_spread = merged[merged["outcome"].eq("Skipped - Vacuum or Spread < 2c")].copy()

    def summarize_block(df, label):
        if df.empty:
            return {
                "group": label,
                "n": 0,
                "mean_stage2": np.nan,
                "mean_p_hot": np.nan,
                "mean_spread": np.nan,
                "gross_sum": 0.0,
                "gross_mean": np.nan,
            }

        return {
            "group": label,
            "n": len(df),
            "mean_stage2": pd.to_numeric(df.get("stage2_p_good"), errors="coerce").mean(),
            "mean_p_hot": pd.to_numeric(df.get("p_hot"), errors="coerce").mean(),
            "mean_spread": pd.to_numeric(df.get("spread"), errors="coerce").mean(),
            "gross_sum": df["gross_pnl"].sum(),
            "gross_mean": df["gross_pnl"].mean(),
        }

    table = pd.DataFrame([
        summarize_block(entered, "entered_actionable"),
        summarize_block(skipped_slot, "skipped_max_slots"),
        summarize_block(skipped_spread, "skipped_spread"),
    ])

    print("\n" + "=" * 90)
    print(f"SLOT HOGGING AUDIT: {name}")
    print("=" * 90)
    print(table.to_string(index=False))

    if not skipped_slot.empty:
        print("\nSkipped-by-slot stage2/spread quantiles:")
        cols = [c for c in ["stage2_p_good", "p_hot", "spread", "actual_entry_spread"] if c in skipped_slot.columns]
        if cols:
            print(skipped_slot[cols].quantile([0.1, 0.25, 0.5, 0.75, 0.9]).to_string())

    if not entered.empty:
        print("\nEntered/actionable stage2/spread quantiles:")
        cols = [c for c in ["stage2_p_good", "p_hot", "spread", "actual_entry_spread"] if c in entered.columns]
        if cols:
            print(entered[cols].quantile([0.1, 0.25, 0.5, 0.75, 0.9]).to_string())

    return merged

audit_slot_hogging(
    receipts_path="/content/stage2_first_pass/live_safe_spread_sweeps/receipts_all_stage2_candidates_execSpread2c_slots1.parquet",
    preds_path="/content/stage2_first_pass/policy_sweeps/preds_all_stage2_candidates_pHot50_spread2c.parquet",
    name="all candidates spread 2c slots1",
)

In [ ]:
def audit_stage1_threshold_vs_spread(scored_df):
    df = scored_df.copy()

    df["p_hot_bucket"] = pd.cut(
        df["p_hot"],
        bins=[-np.inf, 0.50, 0.60, 0.70, 0.80, 0.90, np.inf],
        labels=["<=.50", ".50-.60", ".60-.70", ".70-.80", ".80-.90", ">.90"],
    )

    df["spread_bucket"] = pd.cut(
        df["actual_entry_spread"],
        bins=[-np.inf, 0.019, 0.029, 0.039, 0.049, 0.059, 0.079, np.inf],
        labels=["<2c", "2c", "3c", "4c", "5c", "6-7c", ">=8c"],
    )

    out = (
        df.groupby(["p_hot_bucket", "spread_bucket"], observed=False)
        .agg(
            n=("good_entry", "size"),
            good_rate=("good_entry", "mean"),
            gross_sum=("gross_pnl", "sum"),
            gross_mean=("gross_pnl", "mean"),
            mean_stage2=("stage2_p_good", "mean"),
        )
        .reset_index()
    )

    print(out.to_string(index=False))
    return out

p_hot_spread_audit = audit_stage1_threshold_vs_spread(val_scored_best)

In [ ]:
import os
import numpy as np
import pandas as pd

STAGE1_GATE_DIR = "/content/stage2_first_pass/stage1_gate_sweeps"
os.makedirs(STAGE1_GATE_DIR, exist_ok=True)

def write_gate_preds(scored_df, name, p_hot_min, p_good_min=None):
    mask = scored_df["p_hot"] >= p_hot_min

    if p_good_min is not None:
        mask &= scored_df["stage2_p_good"] >= p_good_min

    df = scored_df[mask].copy().sort_values("local_recv_ts").reset_index(drop=True)

    out = pd.DataFrame({
        "ticker": df["ticker"],
        "ts": df["ts_exchange"] if "ts_exchange" in df.columns else df["local_recv_ts"],
        "ts_exchange": df["ts_exchange"] if "ts_exchange" in df.columns else df["local_recv_ts"],
        "local_recv_ts": df["local_recv_ts"],
        "predicted_side": "maker",
        "confidence": df["stage2_p_good"] if p_good_min is not None else df["p_hot"],
        "p_hot": df["p_hot"],
        "stage2_p_good": df["stage2_p_good"],
        "spread": df["spread"],
    })

    path = os.path.join(STAGE1_GATE_DIR, f"preds_{name}.parquet")
    out.to_parquet(path, index=False)

    print(f"{name}: wrote {len(out)} rows")
    return path

def summarize_receipts_simple(rec, name, p_hot_min, p_good_min, min_spread):
    actionable = rec[~rec["outcome"].astype(str).str.contains("Skipped", na=False)]

    return {
        "policy": name,
        "p_hot_min": p_hot_min,
        "p_good_min": p_good_min,
        "min_spread": min_spread,
        "signals_logged": len(rec),
        "actionable": len(actionable),
        "gross_pnl": rec["gross_pnl"].sum(),
        "gross_per_actionable": actionable["gross_pnl"].mean() if len(actionable) else np.nan,
        "wins": int((rec["outcome"] == "Win - Full Whipsaw Arb").sum()),
        "rescue": int(rec["outcome"].astype(str).str.contains("Bailout|Peg Hit|Timeout", na=False).sum()),
        "dead": int(rec["outcome"].astype(str).str.contains("Dead Market", na=False).sum()),
        "skipped_max_slots": int((rec["outcome"] == "Skipped - Max Active Slots").sum()),
        "skipped_spread": int((rec["outcome"] == "Skipped - Vacuum or Spread < 2c").sum()),
    }

rows = []

configs = []
for p_hot_min in [0.50, 0.60, 0.70, 0.80]:
    configs.append((p_hot_min, None))
    configs.append((p_hot_min, 0.60))
    configs.append((p_hot_min, 0.70))

for p_hot_min, p_good_min in configs:
    gate_name = f"pHot{str(p_hot_min).replace('.', 'p')}"
    if p_good_min is not None:
        gate_name += f"_pGood{str(p_good_min).replace('.', 'p')}"
    else:
        gate_name += "_noStage2"

    pred_path = write_gate_preds(
        val_scored_best,
        name=gate_name,
        p_hot_min=p_hot_min,
        p_good_min=p_good_min,
    )

    for min_spread in [0.02, 0.03, 0.04, 0.05, 0.06]:
        run_name = f"{gate_name}_execSpread{int(min_spread * 100)}c"

        rec = run_multislot_zero_assumption_sim(
            predictions_path=pred_path,
            raw_local_dir=RAW_LOCAL_DIR,
            output_audit_path=os.path.join(STAGE1_GATE_DIR, f"receipts_{run_name}.parquet"),
            max_active_slots=1,
            entry_cooldown_sec=1.0,
            min_entry_spread=min_spread,
            log_slot_skips=True,
        )

        rows.append(
            summarize_receipts_simple(
                rec=rec,
                name=run_name,
                p_hot_min=p_hot_min,
                p_good_min=p_good_min,
                min_spread=min_spread,
            )
        )

stage1_gate_summary = pd.DataFrame(rows).sort_values(
    ["gross_pnl", "gross_per_actionable"],
    ascending=False,
)

summary_path = os.path.join(STAGE1_GATE_DIR, "stage1_gate_summary.csv")
stage1_gate_summary.to_csv(summary_path, index=False)

print(stage1_gate_summary.to_string(index=False))
print("Saved:", summary_path)

In [ ]:
%run /content/microquant_executor_sweep_colab.py

In [ ]:
# %% Cell 1: configure paths and thresholds
import os

RAW_LOCAL_DIR = "/content/kalshi_clean_raw_v2"
RUN_DIR = "/content/executor_sweep_apr29"
os.makedirs(RUN_DIR, exist_ok=True)

STAGE1_WEIGHTS = "/content/hot_density_checkpoints/hot_density_epoch_1.pth"     # edit
STAGE2_WEIGHTS = "/content/stage2_first_pass/checkpoints/best_stage2_by_val_gross.pth"            # edit

# Candidate generation is intentionally broad. Final policy thresholds are below.
cand_cfg = CandidateGenConfig(
    stage1_p_hot_threshold=0.50,
    candidate_min_spread=0.02,
    candidate_cooldown_sec=1.0,
    min_time_to_end_sec=100.0,
    batch_size=512,
)

In [ ]:
# %% Cell 2: generate VAL candidates only and score stage 2
# This is the slowest part, but only runs on val_files. It caches outputs in RUN_DIR.
val_ds, val_stage1_outputs, val_candidates = generate_val_candidates_from_stage1(
    file_paths=val_files,
    stage1_checkpoint=STAGE1_WEIGHTS,
    split_name="apr29_val",
    out_dir=RUN_DIR,
    cfg=cand_cfg,
    force_recompute=False,
)

val_scored = score_stage2_candidates(
    base_stage1_dataset=val_ds,
    candidates_df=val_candidates,
    stage2_checkpoint=STAGE2_WEIGHTS,
    out_dir=RUN_DIR,
    tag="apr29_val",
    force_recompute=False,
)

print("val_candidates:", len(val_candidates))
print("val_scored:", len(val_scored))
print(val_scored[["p_hot", "stage2_p_good", "spread"]].quantile([0, .25, .5, .75, .9, .99, 1.0]))

In [ ]:
# %% Cell 3: define policy gates and executor variants
# Adjust p_hot / p_good / min spread here.
gates = [
    PolicyGate("pHot06_pGood06_spread4c", p_hot_min=0.60, p_good_min=0.60, min_signal_spread=0.04),
    PolicyGate("pHot07_pGood06_spread4c", p_hot_min=0.70, p_good_min=0.60, min_signal_spread=0.04),
    PolicyGate("pHot06_pGood065_spread4c", p_hot_min=0.60, p_good_min=0.65, min_signal_spread=0.04),
    PolicyGate("pHot06_pGood06_spread5c", p_hot_min=0.60, p_good_min=0.60, min_signal_spread=0.05),
    PolicyGate("pHot06_pGood06_spread6c", p_hot_min=0.60, p_good_min=0.60, min_signal_spread=0.06),
]

exec_cfgs = {}
for slots in [1, 2, 3]:
    for name, cfg in standard_executor_variants(order_size=1.0, max_slots=slots, min_entry_spread=0.04).items():
        exec_cfgs[f"slots{slots}_{name}"] = cfg

# Optional: test volume=2, but do this after Q=1 sanity checks.
for name, cfg in standard_executor_variants(order_size=2.0, max_slots=1, min_entry_spread=0.04).items():
    exec_cfgs[f"Q2_slots1_{name}"] = cfg

In [ ]:
# %% Cell 4: run global chronological executor sweep
summary = run_executor_policy_sweep(
    scored_df=val_scored,
    raw_local_dir=RAW_LOCAL_DIR,
    out_dir=os.path.join(RUN_DIR, "policy_sweep"),
    gates=gates,
    executor_cfgs=exec_cfgs,
    global_mode=True,
    save_all_receipts=True,
    show_progress=True,
)

summary.head(30)

In [ ]:
# %% Cell 5: inspect top receipts
best_policy = summary.iloc[0]["policy"]
receipt_path = os.path.join(RUN_DIR, "policy_sweep", f"receipts_{best_policy}.parquet")
rec = pd.read_parquet(receipt_path)
print("Best policy:", best_policy)
print(rec["outcome"].value_counts())
print("gross:", rec["gross_pnl"].sum())
print(rec.sort_values("gross_pnl").head(20).to_string(index=False))

In [ ]:
gates = [
    PolicyGate("pHot06_pGood06_spread4c", 0.60, 0.60, 0.04),
    PolicyGate("pHot06_pGood065_spread4c", 0.60, 0.65, 0.04),
    PolicyGate("pHot06_pGood06_spread5c", 0.60, 0.60, 0.05),
]

exec_cfgs = {}
for slots in [3, 5, 10, 20]:
    variants = standard_executor_variants(
        order_size=1.0,
        max_slots=slots,
        min_entry_spread=0.04,
    )
    for name in ["baseline_peg3", "early1s_adv1c", "early2s_adv2c"]:
        exec_cfgs[f"slots{slots}_{name}"] = variants[name]

summary_slots = run_executor_policy_sweep(
    scored_df=val_scored,
    raw_local_dir=RAW_LOCAL_DIR,
    out_dir=os.path.join(RUN_DIR, "slots_3_5_10_20"),
    gates=gates,
    executor_cfgs=exec_cfgs,
    global_mode=True,
    save_all_receipts=True,
    show_progress=True,
)

In [ ]:
exec_cfgs_q = {}

for q, slots in [(50.0, 20)]:
    variants = standard_executor_variants(
        order_size=q,
        max_slots=slots,
        min_entry_spread=0.04,
    )
    for name in ["baseline_peg3", "early2s_adv2c", "early1s_adv1c"]:
        exec_cfgs_q[f"Q{int(q)}_slots{slots}_{name}"] = variants[name]

summary_q = run_executor_policy_sweep(
    scored_df=val_scored,
    raw_local_dir=RAW_LOCAL_DIR,
    out_dir=os.path.join(RUN_DIR, "q_sweep"),
    gates=[PolicyGate("pHot06_pGood06_spread4c", 0.60, 0.60, 0.04)],
    executor_cfgs=exec_cfgs_q,
    global_mode=True,
    save_all_receipts=True,
    show_progress=True,
)

In [ ]:
%run /content/microquant_executor_sweep_colab.py
%run /content/microquant_dual_market_colab.py

In [ ]:
# %% Cell 0: imports
%run /content/microquant_executor_sweep_colab.py
%run /content/microquant_dual_market_colab.py

import os
import torch
from torch.utils.data import DataLoader

RUN_DIR = "/content/dual_market_first_pass"
os.makedirs(RUN_DIR, exist_ok=True)

RAW_LOCAL_DIR = "/content/kalshi_clean_raw_v2"

# Choose holdout/list variables from your existing split.
# train_files = ...
# val_files = ...

In [ ]:
# %% Cell 1: build dual datasets and audit label balance
# First pass recommendation: k_hot=8 or 10, because there are now three action families.
dual_cfg = DualDatasetConfig(
    seq_len=60,
    candidate_lookahead=30,
    fill_lookahead=30,
    stride=2,
    k_hot=8,
    min_single_spread=0.02,
    min_cross_edge=0.02,
    pair_tolerance_sec=3.0,
    require_pair_available=True,
)

train_dual_ds = DualMarketHotDataset(train_files, cfg=dual_cfg, inference_mode=False, return_diagnostics=True)
val_dual_ds = DualMarketHotDataset(val_files, cfg=dual_cfg, inference_mode=False, return_diagnostics=True)

train_diag = audit_dual_dataset(train_dual_ds, "train")
val_diag = audit_dual_dataset(val_dual_ds, "val")


In [ ]:
# %% Cell 2: train dual Stage 1
# This intentionally mirrors your current Stage 1 loop but uses feature_dim=46.
import numpy as np
import pandas as pd
import torch.nn as nn
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
STAGE1_DUAL_DIR = os.path.join(RUN_DIR, "stage1_dual_checkpoints")
os.makedirs(STAGE1_DUAL_DIR, exist_ok=True)

train_loader = DataLoader(train_dual_ds, batch_size=64, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dual_ds, batch_size=128, shuffle=False, num_workers=0, pin_memory=True)

stage1_dual = KalshiTransformerFlex(feature_dim=46, d_model=64, nhead=4, num_layers=2, num_classes=2).to(DEVICE)
opt = torch.optim.AdamW(stage1_dual.parameters(), lr=1e-4, weight_decay=1e-5)

labels_np = train_dual_ds.hot_label[train_dual_ds.indices]
counts = np.bincount(labels_np.astype(int), minlength=2)
weights = counts.sum() / (2.0 * np.maximum(counts, 1))
weights = np.minimum(weights, np.array([5.0, 12.0]))
criterion = nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float32).to(DEVICE))
print("Dual Stage1 counts:", counts, "weights:", weights)

best_top500 = -1.0
best_stage1_path = os.path.join(STAGE1_DUAL_DIR, "best_dual_stage1_top500.pth")

for epoch in range(1, 6):
    stage1_dual.train()
    loss_sum = 0.0
    for x, y, diag in tqdm(train_loader, desc=f"dual stage1 train {epoch}", leave=False):
        x, y = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad(set_to_none=True)
        logits = stage1_dual(x)
        loss = criterion(logits, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(stage1_dual.parameters(), 1.0)
        opt.step()
        loss_sum += float(loss.item())
    print(f"Epoch {epoch} train_loss={loss_sum/max(1,len(train_loader)):.5f}")

    stage1_dual.eval()
    all_y, all_p = [], []
    with torch.no_grad():
        for x, y, diag in tqdm(val_loader, desc=f"dual stage1 val {epoch}", leave=False):
            x = x.to(DEVICE)
            p = torch.softmax(stage1_dual(x), dim=1)[:,1].detach().cpu().numpy()
            all_p.extend(p); all_y.extend(y.numpy())
    all_y = np.asarray(all_y); all_p = np.asarray(all_p)
    pred = (all_p >= 0.5).astype(int)
    print("Confusion:")
    print(confusion_matrix(all_y, pred, labels=[0,1]))
    print(classification_report(all_y, pred, labels=[0,1], target_names=["cold","hot"], zero_division=0))
    print("p_hot quantiles:", pd.Series(all_p).quantile([.5,.75,.9,.95,.975,.99,.995,.999]).to_string())
    order = np.argsort(-all_p)
    topn = min(500, len(order))
    top500_prec = float(all_y[order[:topn]].mean()) if topn else 0.0
    print("top500 precision:", top500_prec, "cutoff:", float(all_p[order[topn-1]]) if topn else None)
    epoch_path = os.path.join(STAGE1_DUAL_DIR, f"dual_stage1_epoch_{epoch}.pth")
    torch.save(stage1_dual.state_dict(), epoch_path)
    if top500_prec > best_top500:
        best_top500 = top500_prec
        torch.save(stage1_dual.state_dict(), best_stage1_path)
        print("[+] saved best", best_stage1_path)

print("Best dual Stage1:", best_stage1_path)

In [ ]:
# %% Cell 3: generate dual Stage1 candidates on train/val
# If train candidate generation is slow, run val first; train can be sampled later.
cand_cfg = DualCandidateConfig(
    p_hot_threshold=0.50,
    candidate_cooldown_sec=1.0,
    min_time_to_end_sec=100.0,
    batch_size=512,
)

train_dual_inf_ds, train_dual_outputs, train_dual_candidates = generate_dual_stage1_candidates(
    file_paths=train_files,
    stage1_checkpoint=best_stage1_path,
    out_dir=RUN_DIR,
    dataset_cfg=dual_cfg,
    cand_cfg=cand_cfg,
    split_name="train",
    force_recompute=False,
)

val_dual_inf_ds, val_dual_outputs, val_dual_candidates = generate_dual_stage1_candidates(
    file_paths=val_files,
    stage1_checkpoint=best_stage1_path,
    out_dir=RUN_DIR,
    dataset_cfg=dual_cfg,
    cand_cfg=cand_cfg,
    split_name="val",
    force_recompute=False,
)

In [ ]:
# %% Cell 4: create action-conditioned Stage2 labels from dense diagnostics
label_cfg = DualActionLabelConfig(
    min_good_pnl=0.02,
    include_single=True,
    include_yes_yes=True,
    include_no_no=True,
    cross_actions_canonical_only=True,
)

train_action_labels = expand_dual_action_labels_from_dense(train_dual_candidates, cfg=label_cfg)
val_action_labels = expand_dual_action_labels_from_dense(val_dual_candidates, cfg=label_cfg)

train_action_labels.to_parquet(os.path.join(RUN_DIR, "dual_action_labels_train_dense.parquet"), index=False)
val_action_labels.to_parquet(os.path.join(RUN_DIR, "dual_action_labels_val_dense.parquet"), index=False)

print("Train action label good rate:", train_action_labels["good_entry"].mean())
print("Val action label good rate:", val_action_labels["good_entry"].mean())
print("Val action mix:")
print(val_action_labels.groupby("action_type").agg(n=("good_entry","size"), good_rate=("good_entry","mean"), gross=("gross_pnl","sum"), mean_edge=("action_edge","mean")).to_string())

In [ ]:
# %% Cell 5: train action-conditioned Stage2
STAGE2_DUAL_DIR = os.path.join(RUN_DIR, "stage2_dual_action")
stage2_dual_model, best_dual_stage2_path = train_action_stage2_model(
    train_base=train_dual_inf_ds,
    train_labels=train_action_labels,
    val_base=val_dual_inf_ds,
    val_labels=val_action_labels,
    out_dir=STAGE2_DUAL_DIR,
    epochs=5,
    batch_size=128,
    lr=1e-4,
    weight_decay=1e-5,
)

print("Best dual action Stage2:", best_dual_stage2_path)

In [ ]:
# %% Cell 6: score val actions and create approved action predictions
val_action_scored = score_dual_action_stage2(
    base_ds=val_dual_inf_ds,
    action_labels=val_action_labels,
    checkpoint_path=best_dual_stage2_path,
    batch_size=512,
)

val_action_scored.to_parquet(os.path.join(RUN_DIR, "dual_action_scored_val.parquet"), index=False)

approved_dual_actions = make_dual_action_policy_predictions(
    val_action_scored,
    p_hot_min=0.60,
    p_good_min=0.60,
    min_action_edge=0.02,
    output_path=os.path.join(RUN_DIR, "dual_action_predictions_val_pHot06_pGood06_edge2c.parquet"),
)

print("Approved action rows:", len(approved_dual_actions))
print(approved_dual_actions.groupby("action_type").agg(n=("action_type","size"), mean_p_good=("stage2_p_good","mean"), mean_edge=("action_edge","mean")).to_string())

In [ ]:
# %% Cell 7: run first-pass raw dual action sim
sim_cfg = DualActionSimConfig(
    order_size=1.0,
    max_active_slots=3,
    entry_cooldown_sec=1.0,
    latency_penalty_sec=0.020,
    whipsaw_timeout_sec=30.0,
    min_action_edge=0.02,
    force_exit_one_leg=True,
)

receipts_dual = run_dual_action_policy_sim(
    predictions=approved_dual_actions,
    raw_local_dir=RAW_LOCAL_DIR,
    cfg=sim_cfg,
    output_path=os.path.join(RUN_DIR, "dual_action_receipts_val_slots3_Q1.parquet"),
    show_progress=True,
)

summary_dual = summarize_dual_action_receipts(receipts_dual, name="dual_action_slots3_Q1")
print(summary_dual.to_string(index=False))
print("Outcome counts:")
print(receipts_dual["outcome"].value_counts(dropna=False).to_string())
print("By action:")
print(receipts_dual.groupby("action_type").agg(n=("gross_pnl","size"), gross=("gross_pnl","sum"), mean=("gross_pnl","mean"), wins=("outcome", lambda s: s.astype(str).str.contains("Win").sum()), losses=("outcome", lambda s: s.astype(str).str.contains("Loss").sum())).to_string())


In [ ]:
# %% Cell 8: threshold/action sweeps after scoring is cached
rows = []
for p_hot in [0.60]:
    for p_good in [0.90]:
        for edge in [0.04]:
            pred = make_dual_action_policy_predictions(val_action_scored, p_hot_min=p_hot, p_good_min=p_good, min_action_edge=edge)
            if len(pred) == 0:
                continue
            cfg = DualActionSimConfig(order_size=1.0, max_active_slots=5, entry_cooldown_sec=1.0, whipsaw_timeout_sec=30.0, min_action_edge=edge, force_exit_one_leg=True)
            rec = run_dual_action_policy_sim(pred, raw_local_dir=RAW_LOCAL_DIR, cfg=cfg, show_progress=False)
            summ = summarize_dual_action_receipts(rec, name=f"pHot{p_hot}_pGood{p_good}_edge{edge}").iloc[0].to_dict()
            summ.update({"p_hot": p_hot, "p_good": p_good, "edge": edge, "signals": len(pred)})
            rows.append(summ)

sweep_dual = pd.DataFrame(rows).sort_values(["gross_pnl", "gross_per_actionable"], ascending=False)
sweep_dual.to_csv(os.path.join(RUN_DIR, "dual_action_policy_sweep_summary.csv"), index=False)
print(sweep_dual.head(30).to_string(index=False))

In [ ]:
sel = val_scored[val_scored["stage2_p_good"] >= 0.70].copy()

print(sel.groupby("action_type").agg(
    n=("gross_pnl", "size"),
    good_rate=("good_entry", "mean"),
    gross=("gross_pnl", "sum"),
    mean=("gross_pnl", "mean"),
).sort_values("gross", ascending=False))

print(sel.groupby("game_id").agg(
    n=("gross_pnl", "size"),
    good_rate=("good_entry", "mean"),
    gross=("gross_pnl", "sum"),
    mean=("gross_pnl", "mean"),
).sort_values("gross", ascending=False).head(20))

In [ ]:
# %% Cell 8: threshold/action sweeps after scoring is cached
rows = []
for p_hot in [0.60, 0.70]:
    for p_good in [0.70, 0.90]:
        for edge in [0.03, 0.04]:
            pred = make_dual_action_policy_predictions(val_action_scored, p_hot_min=p_hot, p_good_min=p_good, min_action_edge=edge)
            if len(pred) == 0:
                continue
            cfg = DualActionSimConfig(order_size=50.0, max_active_slots=10, entry_cooldown_sec=1.0, whipsaw_timeout_sec=30.0, min_action_edge=edge, force_exit_one_leg=True)
            rec = run_dual_action_policy_sim(pred, raw_local_dir=RAW_LOCAL_DIR, cfg=cfg, show_progress=False)
            summ = summarize_dual_action_receipts(rec, name=f"pHot{p_hot}_pGood{p_good}_edge{edge}").iloc[0].to_dict()
            summ.update({"p_hot": p_hot, "p_good": p_good, "edge": edge, "signals": len(pred)})
            rows.append(summ)

sweep_dual = pd.DataFrame(rows).sort_values(["gross_pnl", "gross_per_actionable"], ascending=False)
sweep_dual.to_csv(os.path.join(RUN_DIR, "dual_action_policy_sweep_summary.csv"), index=False)
print(sweep_dual.head(30).to_string(index=False))

In [ ]:
import pandas as pd
import numpy as np
import os, glob

# Change this to the receipt file for the policy you want.
# Example:
# receipts_path = "/content/.../receipts_pHot0.6_pGood0.9_edge0.04.parquet"

receipt_candidates = sorted(glob.glob("/content/**/*receipts*.parquet", recursive=True))
print("\n".join(receipt_candidates[-20:]))

receipts_path = receipt_candidates[-1]  # replace manually if needed
df = pd.read_parquet(receipts_path).copy()

print("Loaded:", receipts_path)
print("Rows:", len(df))
print("Columns:", list(df.columns))

df["gross_pnl"] = pd.to_numeric(df["gross_pnl"], errors="coerce").fillna(0.0)
df["outcome"] = df["outcome"].astype(str)

if "action_type" not in df.columns:
    print("WARNING: no action_type column found. Need to patch dual sim receipts to include action_type.")
else:
    print("\n=== By action_type ===")
    print(
        df.groupby("action_type")
        .agg(
            rows=("gross_pnl", "size"),
            actionable=("outcome", lambda s: (~s.str.contains("Skipped", na=False)).sum()),
            gross=("gross_pnl", "sum"),
            mean=("gross_pnl", "mean"),
            wins=("outcome", lambda s: s.str.contains("Win", na=False).sum()),
            losses=("outcome", lambda s: s.str.contains("Loss", na=False).sum()),
            one_leg=("outcome", lambda s: s.str.contains("One-Leg", na=False).sum()),
            clean=("outcome", lambda s: s.str.contains("Clean", na=False).sum()),
            skipped=("outcome", lambda s: s.str.contains("Skipped", na=False).sum()),
            max_loss=("gross_pnl", "min"),
        )
        .sort_values("gross", ascending=False)
        .to_string()
    )

print("\n=== Outcome counts ===")
print(df["outcome"].value_counts().to_string())

if "action_type" in df.columns:
    print("\n=== Outcome by action_type ===")
    print(pd.crosstab(df["action_type"], df["outcome"]).to_string())

print("\n=== Worst trades ===")
cols = [
    "action_type", "ticker", "pair_ticker", "game_id",
    "local_recv_ts", "p_hot", "stage2_p_good",
    "action_edge", "gross_pnl", "outcome",
    "entry_px_1", "entry_px_2",
    "fill_qty_1", "fill_qty_2",
    "fill_ts_1", "fill_ts_2",
]
cols = [c for c in cols if c in df.columns]
print(df.sort_values("gross_pnl").head(30)[cols].to_string(index=False))

print("\n=== Best trades ===")
print(df.sort_values("gross_pnl", ascending=False).head(30)[cols].to_string(index=False))

In [ ]:
print("rec rows:", len(rec))
print("columns:")
print(rec.columns.tolist())

print("\nOutcome counts:")
print(rec["outcome"].value_counts(dropna=False))

if "action_type" in rec.columns:
    print("\nBy action_type:")
    print(
        rec.groupby("action_type")
        .agg(
            rows=("gross_pnl", "size"),
            actionable=("outcome", lambda s: (~s.astype(str).str.contains("Skipped", na=False)).sum()),
            gross=("gross_pnl", "sum"),
            mean=("gross_pnl", "mean"),
            wins=("outcome", lambda s: s.astype(str).str.contains("Win", na=False).sum()),
            losses=("outcome", lambda s: s.astype(str).str.contains("Loss", na=False).sum()),
            skipped=("outcome", lambda s: s.astype(str).str.contains("Skipped", na=False).sum()),
        )
        .sort_values("gross", ascending=False)
    )

    print("\nOutcome x action_type:")
    print(pd.crosstab(rec["action_type"], rec["outcome"]))
else:
    print("\nWARNING: rec has no action_type column.")

In [ ]:
# %% Cell 0: imports
%run /content/microquant_executor_sweep_colab.py
%run /content/microquant_dual_market_colab_v2.py

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix

RUN_DIR = "/content/dual_market_v2"
os.makedirs(RUN_DIR, exist_ok=True)
RAW_LOCAL_DIR = "/content/kalshi_clean_raw_v2"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

In [ ]:
# %% Cell 1: build dual datasets and audit label balance
# k_hot=8 is a first pass because dual_hot_count includes single + yes_yes + no_no.
dual_cfg = DualDatasetConfig(
    seq_len=60,
    candidate_lookahead=30,
    fill_lookahead=30,
    stride=2,
    k_hot=8,
    min_single_spread=0.02,
    min_cross_edge=0.02,
    pair_tolerance_sec=3.0,
    require_pair_available=True,
)

train_dual_ds = DualMarketHotDataset(train_files, cfg=dual_cfg, inference_mode=False, return_diagnostics=True)
val_dual_ds = DualMarketHotDataset(val_files, cfg=dual_cfg, inference_mode=False, return_diagnostics=True)

train_diag = audit_dual_dataset(train_dual_ds, "train")
val_diag = audit_dual_dataset(val_dual_ds, "val")

print("\nBad-count quantiles train:")
print(train_diag[["dual_hot_count", "dual_bad_count", "single_bad_count", "yes_yes_bad_count", "no_no_bad_count"]].quantile([.5,.75,.9,.95,.99]).to_string())
print("\nBad-count quantiles val:")
print(val_diag[["dual_hot_count", "dual_bad_count", "single_bad_count", "yes_yes_bad_count", "no_no_bad_count"]].quantile([.5,.75,.9,.95,.99]).to_string())

In [ ]:
# %% Cell 2: train dual Stage 1 with hard-negative CE
STAGE1_DUAL_DIR = os.path.join(RUN_DIR, "stage1_dual_checkpoints")
os.makedirs(STAGE1_DUAL_DIR, exist_ok=True)

train_loader = DataLoader(train_dual_ds, batch_size=64, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dual_ds, batch_size=128, shuffle=False, num_workers=0, pin_memory=True)

stage1_dual = KalshiTransformerFlex(feature_dim=46, d_model=64, nhead=4, num_layers=2, num_classes=2).to(DEVICE)
opt = torch.optim.AdamW(stage1_dual.parameters(), lr=1e-4, weight_decay=1e-5)

labels_np = train_dual_ds.hot_label[train_dual_ds.indices]
counts = np.bincount(labels_np.astype(int), minlength=2)
print("Dual Stage1 counts:", counts, "hot_rate:", counts[1] / max(1, counts.sum()))

class DualHardNegativeCELoss(nn.Module):
    def __init__(self, pos_weight=8.0, toxic_boost=2.0, bad_count_scale=10.0, max_weight=12.0):
        super().__init__()
        self.pos_weight = float(pos_weight)
        self.toxic_boost = float(toxic_boost)
        self.bad_count_scale = float(bad_count_scale)
        self.max_weight = float(max_weight)

    def forward(self, logits, targets, bad_counts):
        ce = torch.nn.functional.cross_entropy(logits, targets, reduction="none")
        targets_f = targets.float()
        bad_counts = bad_counts.float()
        bad_strength = torch.clamp(bad_counts / self.bad_count_scale, 0.0, 1.0)
        weights = torch.where(
            targets == 1,
            torch.full_like(targets_f, self.pos_weight),
            1.0 + self.toxic_boost * bad_strength,
        )
        weights = torch.clamp(weights, 1.0, self.max_weight)
        return (ce * weights).mean()

criterion = DualHardNegativeCELoss(
    pos_weight=8.0,
    toxic_boost=2.0,
    bad_count_scale=10.0,
    max_weight=12.0,
).to(DEVICE)
print("Using DualHardNegativeCELoss")

stage1_history = []
DUAL_STAGE1_CHECKPOINT = None

for epoch in range(1, 6):
    stage1_dual.train()
    loss_sum = 0.0
    for x, y, diag in tqdm(train_loader, desc=f"dual stage1 train {epoch}", leave=False):
        x, y = x.to(DEVICE), y.to(DEVICE)
        bad_counts = diag["dual_bad_count"].to(DEVICE)
        opt.zero_grad(set_to_none=True)
        logits = stage1_dual(x)
        loss = criterion(logits, y, bad_counts)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(stage1_dual.parameters(), 1.0)
        opt.step()
        loss_sum += float(loss.item())
    avg_train_loss = loss_sum / max(1, len(train_loader))
    print(f"Epoch {epoch} train_loss={avg_train_loss:.5f}")

    stage1_dual.eval()
    all_y, all_p = [], []
    val_loss_sum = 0.0
    with torch.no_grad():
        for x, y, diag in tqdm(val_loader, desc=f"dual stage1 val {epoch}", leave=False):
            x = x.to(DEVICE)
            y_dev = y.to(DEVICE)
            bad_counts = diag["dual_bad_count"].to(DEVICE)
            logits = stage1_dual(x)
            loss = criterion(logits, y_dev, bad_counts)
            val_loss_sum += float(loss.item())
            p = torch.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()
            all_p.extend(p)
            all_y.extend(y.numpy())

    avg_val_loss = val_loss_sum / max(1, len(val_loader))
    all_y = np.asarray(all_y); all_p = np.asarray(all_p)
    pred = (all_p >= 0.5).astype(int)
    print(f"Epoch {epoch} val_loss={avg_val_loss:.5f}")
    print("Confusion:")
    print(confusion_matrix(all_y, pred, labels=[0,1]))
    print(classification_report(all_y, pred, labels=[0,1], target_names=["cold","hot"], zero_division=0))
    print("p_hot quantiles:", pd.Series(all_p).quantile([.5,.75,.9,.95,.975,.99,.995,.999]).to_string())
    order = np.argsort(-all_p)
    top_rows = []
    for n in [100, 300, 500, 1000, 2000, 5000]:
        nn = min(n, len(order))
        if nn <= 0:
            continue
        top_rows.append({"top_n": nn, "precision": float(all_y[order[:nn]].mean()), "cutoff": float(all_p[order[nn-1]])})
    print(pd.DataFrame(top_rows).to_string(index=False))

    epoch_path = os.path.join(STAGE1_DUAL_DIR, f"dual_stage1_epoch_{epoch}.pth")
    torch.save(stage1_dual.state_dict(), epoch_path)
    DUAL_STAGE1_CHECKPOINT = epoch_path
    stage1_history.append({"epoch": epoch, "train_loss": avg_train_loss, "val_loss": avg_val_loss, "checkpoint": epoch_path})
    print("Saved:", epoch_path)

pd.DataFrame(stage1_history).to_csv(os.path.join(STAGE1_DUAL_DIR, "dual_stage1_history.csv"), index=False)
print("Stage1 history saved:", os.path.join(STAGE1_DUAL_DIR, "dual_stage1_history.csv"))
print("Default DUAL_STAGE1_CHECKPOINT is last epoch:", DUAL_STAGE1_CHECKPOINT)
print("Override DUAL_STAGE1_CHECKPOINT manually here if you prefer a different epoch.")

In [ ]:
DUAL_STAGE1_CHECKPOINT = "/content/dual_market_v2/stage1_dual_checkpoints/dual_stage1_epoch_1.pth"

In [ ]:
# %% Cell 3: generate dual Stage1 candidates on train/val
# If train candidate generation is slow, run val first; train can be sampled later.
cand_cfg = DualCandidateConfig(
    p_hot_threshold=0.50,
    candidate_cooldown_sec=1.0,
    min_time_to_end_sec=100.0,
    batch_size=512,
)

train_dual_inf_ds, train_dual_outputs, train_dual_candidates = generate_dual_stage1_candidates(
    file_paths=train_files,
    stage1_checkpoint=DUAL_STAGE1_CHECKPOINT,
    out_dir=RUN_DIR,
    dataset_cfg=dual_cfg,
    cand_cfg=cand_cfg,
    split_name="train",
    force_recompute=False,
)

val_dual_inf_ds, val_dual_outputs, val_dual_candidates = generate_dual_stage1_candidates(
    file_paths=val_files,
    stage1_checkpoint=DUAL_STAGE1_CHECKPOINT,
    out_dir=RUN_DIR,
    dataset_cfg=dual_cfg,
    cand_cfg=cand_cfg,
    split_name="val",
    force_recompute=False,
)

In [ ]:
# Re-load the two dependency files into the current notebook namespace
exec(compile(open("/content/microquant_executor_sweep_colab.py").read(),
             "/content/microquant_executor_sweep_colab.py", "exec"), globals())

exec(compile(open("/content/microquant_dual_market_colab_v2.py").read(),
             "/content/microquant_dual_market_colab_v2.py", "exec"), globals())

print("Before patch:")
print("DualActionSimConfig:", "DualActionSimConfig" in globals())
print("DualActionGlobalExecutor:", "DualActionGlobalExecutor" in globals())
print("run_dual_action_policy_sim:", "run_dual_action_policy_sim" in globals())

# Now load the patch into the SAME namespace
exec(compile(open("/content/microquant_dual_raw_labels_patch.py").read(),
             "/content/microquant_dual_raw_labels_patch.py", "exec"), globals())

print("After patch:")
print("label_dual_actions_from_raw_executor:", "label_dual_actions_from_raw_executor" in globals())
print("RawDualActionLabelConfig:", "RawDualActionLabelConfig" in globals())

In [ ]:
# %% Cell 4 replacement: raw-executor Stage2 labels (NO dense shortcut)
raw_label_cfg = DualRawLabelConfig(
    min_good_pnl=0.019,          # ~2c with float tolerance
    min_action_edge=0.02,        # same initial edge/spread gate as old Stage2 candidate min
    include_single=True,
    include_yes_yes=True,
    include_no_no=True,
    cross_actions_canonical_only=True,
    catastrophic_pnl=-0.05,
    random_state=42,
)

label_sim_cfg = DualActionSimConfig(
    order_size=1.0,
    max_active_slots=1,          # forced independent label
    entry_cooldown_sec=0.0,      # forced independent label
    latency_penalty_sec=0.020,
    whipsaw_timeout_sec=30.0,
    peg_cycle_sec=3.0,
    ttl_puke_sec=60.0,
    min_spread_bailout=0.01,
    min_action_edge=0.02,
    force_exit_one_leg=True,
)

# For a fast first pass, set max_action_rows=30000 or 50000 for train.
# For the serious run, set max_action_rows=None.
train_action_labels = label_dual_actions_from_raw_executor(
    candidates=train_dual_candidates,
    raw_local_dir=RAW_LOCAL_DIR,
    cfg=label_sim_cfg,
    label_cfg=raw_label_cfg,
    split_name="train",
    output_path=os.path.join(RUN_DIR, "dual_action_labels_train_RAW.parquet"),
    max_action_rows=50000,
    force_recompute=False,
    show_progress=True,
)

val_action_labels = label_dual_actions_from_raw_executor(
    candidates=val_dual_candidates,
    raw_local_dir=RAW_LOCAL_DIR,
    cfg=label_sim_cfg,
    label_cfg=raw_label_cfg,
    split_name="val",
    output_path=os.path.join(RUN_DIR, "dual_action_labels_val_RAW.parquet"),
    max_action_rows=None,
    force_recompute=False,
    show_progress=True,
)

print("Train raw action label good rate:", train_action_labels["good_entry"].mean() if len(train_action_labels) else None)
print("Val raw action label good rate:", val_action_labels["good_entry"].mean() if len(val_action_labels) else None)
print("Val raw action mix:")
if len(val_action_labels):
    print(val_action_labels.groupby("action_type").agg(
        n=("good_entry", "size"),
        actionable=("label_actionable", "sum"),
        good_rate=("good_entry", "mean"),
        gross=("gross_pnl", "sum"),
        mean_gross=("gross_pnl", "mean"),
        clean=("clean_pnl", "sum"),
        rescue=("rescue_pnl", "sum"),
    ).to_string())

In [ ]:
# %% Cell 5 replacement/keep: train action-conditioned Stage2 on RAW labels
STAGE2_DUAL_DIR = os.path.join(RUN_DIR, "stage2_dual_action_RAW")
stage2_dual_model, DUAL_STAGE2_CHECKPOINT = train_action_stage2_model(
    train_base=train_dual_inf_ds,
    train_labels=train_action_labels,
    val_base=val_dual_inf_ds,
    val_labels=val_action_labels,
    out_dir=STAGE2_DUAL_DIR,
    epochs=5,
    batch_size=128,
    lr=1e-4,
    weight_decay=1e-5,
)
print("Default RAW-label DUAL_STAGE2_CHECKPOINT is last epoch:", DUAL_STAGE2_CHECKPOINT)

In [ ]:
# %% Cell 6 replacement/keep: score RAW-label Stage2 and write predictions
val_action_scored = score_dual_action_stage2(
    base_ds=val_dual_inf_ds,
    action_labels=val_action_labels,
    checkpoint_path=DUAL_STAGE2_CHECKPOINT,
    batch_size=512,
)
val_action_scored.to_parquet(os.path.join(RUN_DIR, "dual_action_scored_val_RAW.parquet"), index=False)

approved_dual_actions = make_dual_action_policy_predictions(
    val_action_scored,
    p_hot_min=0.60,
    p_good_min=0.60,
    min_action_edge=0.02,
    output_path=os.path.join(RUN_DIR, "dual_action_predictions_val_RAW_pHot06_pGood06_edge2c.parquet"),
)
print("Approved RAW action rows:", len(approved_dual_actions))
if len(approved_dual_actions):
    print(approved_dual_actions.groupby("action_type").agg(
        n=("action_type","size"), mean_p_good=("stage2_p_good","mean"), mean_edge=("action_edge","mean")
    ).to_string())

In [ ]:
# %% Cell 7: required single-action control before interpreting dual results
sim_cfg = DualActionSimConfig(
    order_size=1.0,
    max_active_slots=3,
    entry_cooldown_sec=1.0,
    latency_penalty_sec=0.020,
    whipsaw_timeout_sec=30.0,
    peg_cycle_sec=3.0,
    ttl_puke_sec=60.0,
    min_spread_bailout=0.01,
    min_action_edge=0.02,
    force_exit_one_leg=True,
)

single_control = compare_dual_single_to_baseline_executor(
    approved_dual_actions[approved_dual_actions["action_type"] == "single"].copy(),
    raw_local_dir=RAW_LOCAL_DIR,
    dual_cfg=sim_cfg,
    out_dir=os.path.join(RUN_DIR, "single_control_RAW"),
    tag="raw_stage2_single_control",
    show_progress=True,
)

In [ ]:
# %% Cell 8: raw-label dual policy sim and sweep
receipts_dual = run_dual_action_policy_sim(
    predictions=approved_dual_actions,
    raw_local_dir=RAW_LOCAL_DIR,
    cfg=sim_cfg,
    output_path=os.path.join(RUN_DIR, "dual_action_receipts_val_RAW_slots3_Q1.parquet"),
    show_progress=True,
)
summary_dual = summarize_dual_action_receipts(receipts_dual, name="dual_action_RAW_slots3_Q1")
print(summary_dual.to_string(index=False))
print("Outcome counts:")
print(receipts_dual["outcome"].value_counts(dropna=False).to_string())
print("By action:")
if len(receipts_dual):
    print(receipts_dual.groupby("action_type").agg(
        n=("gross_pnl","size"), gross=("gross_pnl","sum"), mean=("gross_pnl","mean"),
        clean=("clean_pnl","sum"), rescue=("rescue_pnl","sum"),
        wins=("outcome", lambda s: s.astype(str).str.contains("Win").sum()),
        losses=("outcome", lambda s: s.astype(str).str.contains("Loss").sum()),
    ).to_string())

In [ ]:
# %% Cell 9: raw-label threshold sweep with receipts saved
rows = []
all_receipts = []
for p_hot in [0.50, 0.60, 0.70]:
    for p_good in [0.7, 0.80]:
        for edge in [0.03, 0.04]:
            policy_name = f"RAW_pHot{p_hot}_pGood{p_good}_edge{edge}"
            pred = make_dual_action_policy_predictions(val_action_scored, p_hot_min=p_hot, p_good_min=p_good, min_action_edge=edge)
            if len(pred) == 0:
                continue
            cfg = DualActionSimConfig(
                order_size=100.0,
                max_active_slots=20,
                entry_cooldown_sec=0,
                whipsaw_timeout_sec=30.0,
                peg_cycle_sec=3.0,
                ttl_puke_sec=60.0,
                min_spread_bailout=0.01,
                min_action_edge=edge,
                force_exit_one_leg=True,
            )
            rec = run_dual_action_policy_sim(pred, raw_local_dir=RAW_LOCAL_DIR, cfg=cfg, show_progress=False)
            rec = rec.copy()
            rec["policy"] = policy_name
            receipt_path = os.path.join(RUN_DIR, f"receipts_{policy_name}.parquet")
            rec.to_parquet(receipt_path, index=False)
            all_receipts.append(rec)
            summ_df = summarize_dual_action_receipts(rec, name=policy_name).copy()
            summ_df["p_hot"] = p_hot
            summ_df["p_good"] = p_good
            summ_df["edge"] = edge
            summ_df["signals"] = len(pred)
            summ_df["receipt_path"] = receipt_path
            rows.extend(summ_df.to_dict("records"))

sweep_dual = pd.DataFrame(rows).sort_values(["gross_pnl", "gross_per_actionable"], ascending=False)
sweep_dual.to_csv(os.path.join(RUN_DIR, "dual_action_policy_sweep_summary_RAW.csv"), index=False)
all_receipts_df = pd.concat(all_receipts, ignore_index=True) if all_receipts else pd.DataFrame()
all_receipts_df.to_parquet(os.path.join(RUN_DIR, "dual_action_all_receipts_RAW.parquet"), index=False)
print(sweep_dual.head(80).to_string(index=False))
print("Saved RAW sweep:", os.path.join(RUN_DIR, "dual_action_policy_sweep_summary_RAW.csv"))
print("Saved RAW all receipts:", os.path.join(RUN_DIR, "dual_action_all_receipts_RAW.parquet"))

In [ ]:
import os
import pandas as pd
import numpy as np

RECEIPTS_PATH = "/content/dual_market_v2/dual_action_all_receipts_RAW.parquet"
df = pd.read_parquet(RECEIPTS_PATH).copy()

for c in ["gross_pnl", "clean_pnl", "rescue_pnl", "p_hot", "stage2_p_good", "action_edge"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

df["outcome"] = df["outcome"].astype(str)

actionable = df[~df["outcome"].str.contains("Skipped", na=False)].copy()

print("Rows:", len(df), "Actionable:", len(actionable))
print("Columns:", df.columns.tolist())

show_cols = [
    "policy", "slot_id", "action_type", "game_id", "ticker", "pair_ticker",
    "signal_ts", "entry_ts", "exit_ts",
    "outcome", "gross_pnl", "clean_pnl", "rescue_pnl",
    "matched_qty", "remaining_unmatched_qty",
    "p_hot", "stage2_p_good", "action_edge",
    "entry_px_1", "entry_px_2",
    "fill_qty_1", "fill_qty_2",
    "fill_ts_1", "fill_ts_2",
    "force_exit_attempts", "event_chain",
]
show_cols = [c for c in show_cols if c in actionable.columns]

worst = actionable.sort_values("gross_pnl").head(30)
print("\n=== WORST 30 TRADES ===")
print(worst[show_cols].to_string(index=False))

print("\n=== WORST TRADE EVENT CHAINS ===")
for i, row in worst.head(10).iterrows():
    print("\n" + "=" * 120)
    print("policy:", row.get("policy"))
    print("action:", row.get("action_type"))
    print("ticker:", row.get("ticker"), "pair:", row.get("pair_ticker"), "game:", row.get("game_id"))
    print("gross:", row.get("gross_pnl"), "clean:", row.get("clean_pnl"), "rescue:", row.get("rescue_pnl"))
    print("p_hot:", row.get("p_hot"), "p_good:", row.get("stage2_p_good"), "edge:", row.get("action_edge"))
    print("entry px:", row.get("entry_px_1"), row.get("entry_px_2"))
    print("fill qty:", row.get("fill_qty_1"), row.get("fill_qty_2"))
    print("fill ts:", row.get("fill_ts_1"), row.get("fill_ts_2"))
    chain = str(row.get("event_chain", ""))
    if len(chain) > 1000:
        chain = chain[:1000] + "...TRUNCATED..."
    print("chain:", chain)

In [ ]:
id_cols = [c for c in ["action_type", "game_id", "ticker", "pair_ticker", "signal_ts"] if c in actionable.columns]

dedup = (
    actionable.sort_values("gross_pnl")
    .drop_duplicates(subset=id_cols, keep="first")
    .copy()
)

print("Actionable rows:", len(actionable))
print("Dedup trades:", len(dedup))

print("\n=== WORST UNIQUE TRADES ===")
print(dedup.sort_values("gross_pnl").head(30)[show_cols].to_string(index=False))

print("\n=== Worst unique by action ===")
print(
    dedup.groupby("action_type")
    .agg(
        n=("gross_pnl", "size"),
        gross=("gross_pnl", "sum"),
        mean=("gross_pnl", "mean"),
        min=("gross_pnl", "min"),
        clean=("clean_pnl", "sum"),
        rescue=("rescue_pnl", "sum"),
    )
    .sort_values("gross", ascending=False)
    .to_string()
)

In [ ]:
w = dedup.sort_values("gross_pnl").head(100).copy()

print(w.groupby(["game_id", "action_type"]).agg(
    n=("gross_pnl", "size"),
    gross=("gross_pnl", "sum"),
    min_loss=("gross_pnl", "min"),
    mean_p_hot=("p_hot", "mean"),
    mean_p_good=("stage2_p_good", "mean"),
    mean_edge=("action_edge", "mean"),
).sort_values("gross").to_string())

print(pd.crosstab(w["action_type"], w["outcome"]).to_string())

In [ ]:
bad_completed = dedup[
    (dedup["action_type"].isin(["no_no", "yes_yes"]))
    & (dedup["matched_qty"] > 0)
    & (dedup["gross_pnl"] < -5)
].copy()

print(bad_completed[[
    "policy","action_type","game_id","ticker","pair_ticker",
    "gross_pnl","clean_pnl","rescue_pnl","matched_qty",
    "entry_px_1","entry_px_2","fill_qty_1","fill_qty_2",
    "p_hot","stage2_p_good","action_edge","event_chain"
]].sort_values("gross_pnl").head(30).to_string(index=False))

In [ ]:
# %% Cell 4: create action-conditioned Stage2 labels from dense diagnostics
label_cfg = DualActionLabelConfig(
    min_good_pnl=0.02,
    include_single=True,
    include_yes_yes=True,
    include_no_no=True,
    cross_actions_canonical_only=True,
)

train_action_labels = expand_dual_action_labels_from_dense(train_dual_candidates, cfg=label_cfg)
val_action_labels = expand_dual_action_labels_from_dense(val_dual_candidates, cfg=label_cfg)

train_action_labels.to_parquet(os.path.join(RUN_DIR, "dual_action_labels_train_dense.parquet"), index=False)
val_action_labels.to_parquet(os.path.join(RUN_DIR, "dual_action_labels_val_dense.parquet"), index=False)

print("Train action label good rate:", train_action_labels["good_entry"].mean())
print("Val action label good rate:", val_action_labels["good_entry"].mean())
print("Val action mix:")
print(val_action_labels.groupby("action_type").agg(n=("good_entry","size"), good_rate=("good_entry","mean"), gross=("gross_pnl","sum"), mean_edge=("action_edge","mean")).to_string())

In [ ]:
# %% Cell 5: train action-conditioned Stage2
# Saves every epoch. It returns the last checkpoint by default; override manually if desired.
STAGE2_DUAL_DIR = os.path.join(RUN_DIR, "stage2_dual_action")
stage2_dual_model, DUAL_STAGE2_CHECKPOINT = train_action_stage2_model(
    train_base=train_dual_inf_ds,
    train_labels=train_action_labels,
    val_base=val_dual_inf_ds,
    val_labels=val_action_labels,
    out_dir=STAGE2_DUAL_DIR,
    epochs=5,
    batch_size=128,
    lr=1e-4,
    weight_decay=1e-5,
)

print("Default DUAL_STAGE2_CHECKPOINT is last epoch:", DUAL_STAGE2_CHECKPOINT)
print("Override DUAL_STAGE2_CHECKPOINT manually here if you prefer a different epoch.")


In [ ]:
# %% Cell 6: score val actions and create approved action predictions
val_action_scored = score_dual_action_stage2(
    base_ds=val_dual_inf_ds,
    action_labels=val_action_labels,
    checkpoint_path=DUAL_STAGE2_CHECKPOINT,
    batch_size=512,
)

val_action_scored.to_parquet(os.path.join(RUN_DIR, "dual_action_scored_val.parquet"), index=False)

approved_dual_actions = make_dual_action_policy_predictions(
    val_action_scored,
    p_hot_min=0.60,
    p_good_min=0.60,
    min_action_edge=0.02,
    output_path=os.path.join(RUN_DIR, "dual_action_predictions_val_pHot06_pGood06_edge2c.parquet"),
)

print("Approved action rows:", len(approved_dual_actions))
if len(approved_dual_actions):
    print(approved_dual_actions.groupby("action_type").agg(n=("action_type","size"), mean_p_good=("stage2_p_good","mean"), mean_edge=("action_edge","mean")).to_string())

In [ ]:
# %% Cell 7: run raw dual action sim with baseline-style pegging/rescue
sim_cfg = DualActionSimConfig(
    order_size=1.0,
    max_active_slots=3,
    entry_cooldown_sec=1.0,
    latency_penalty_sec=0.020,
    whipsaw_timeout_sec=30.0,
    peg_cycle_sec=3.0,
    ttl_puke_sec=60.0,
    min_spread_bailout=0.01,
    min_action_edge=0.02,
    force_exit_one_leg=True,
)

receipts_dual = run_dual_action_policy_sim(
    predictions=approved_dual_actions,
    raw_local_dir=RAW_LOCAL_DIR,
    cfg=sim_cfg,
    output_path=os.path.join(RUN_DIR, "dual_action_receipts_val_slots3_Q1.parquet"),
    show_progress=True,
)

summary_dual = summarize_dual_action_receipts(receipts_dual, name="dual_action_slots3_Q1")
print(summary_dual.to_string(index=False))
print("Outcome counts:")
print(receipts_dual["outcome"].value_counts(dropna=False).to_string())
print("By action:")
print(receipts_dual.groupby("action_type").agg(n=("gross_pnl","size"), gross=("gross_pnl","sum"), mean=("gross_pnl","mean"), clean=("clean_pnl","sum"), rescue=("rescue_pnl","sum"), wins=("outcome", lambda s: s.astype(str).str.contains("Win").sum()), losses=("outcome", lambda s: s.astype(str).str.contains("Loss").sum())).to_string())

In [ ]:
# %% Cell 8: threshold/action sweeps after scoring is cached; save receipts for every policy
rows = []
all_receipts = []
for p_hot in [0.50, 0.60, 0.70]:
    for p_good in [0.50, 0.60, 0.70, 0.80, 0.90]:
        for edge in [0.01, 0.02, 0.03, 0.04]:
            policy_name = f"pHot{p_hot}_pGood{p_good}_edge{edge}"
            pred = make_dual_action_policy_predictions(val_action_scored, p_hot_min=p_hot, p_good_min=p_good, min_action_edge=edge)
            if len(pred) == 0:
                continue
            cfg = DualActionSimConfig(
                order_size=1.0,
                max_active_slots=3,
                entry_cooldown_sec=1.0,
                whipsaw_timeout_sec=30.0,
                peg_cycle_sec=3.0,
                ttl_puke_sec=60.0,
                min_spread_bailout=0.01,
                min_action_edge=edge,
                force_exit_one_leg=True,
            )
            rec = run_dual_action_policy_sim(pred, raw_local_dir=RAW_LOCAL_DIR, cfg=cfg, show_progress=False)
            rec = rec.copy()
            rec["policy"] = policy_name
            receipt_path = os.path.join(RUN_DIR, f"receipts_{policy_name}.parquet")
            rec.to_parquet(receipt_path, index=False)
            all_receipts.append(rec)
            summ_df = summarize_dual_action_receipts(rec, name=policy_name).copy()
            summ_df["p_hot"] = p_hot
            summ_df["p_good"] = p_good
            summ_df["edge"] = edge
            summ_df["signals"] = len(pred)
            summ_df["receipt_path"] = receipt_path
            rows.extend(summ_df.to_dict("records"))

sweep_dual = pd.DataFrame(rows).sort_values(["gross_pnl", "gross_per_actionable"], ascending=False)
sweep_dual.to_csv(os.path.join(RUN_DIR, "dual_action_policy_sweep_summary.csv"), index=False)
all_receipts_df = pd.concat(all_receipts, ignore_index=True) if all_receipts else pd.DataFrame()
all_receipts_df.to_parquet(os.path.join(RUN_DIR, "dual_action_all_receipts.parquet"), index=False)
print(sweep_dual.head(60).to_string(index=False))
print("Saved sweep:", os.path.join(RUN_DIR, "dual_action_policy_sweep_summary.csv"))
print("Saved all receipts:", os.path.join(RUN_DIR, "dual_action_all_receipts.parquet"))

In [ ]:
# %% Cell 9: receipt diagnostics
if len(all_receipts_df):
    actionable = all_receipts_df[~all_receipts_df["outcome"].astype(str).str.contains("Skipped", na=False)].copy()
    print("\nOutcome x action type:")
    print(pd.crosstab(all_receipts_df["action_type"], all_receipts_df["outcome"]).to_string())
    print("\nActionable by action/outcome:")
    print(actionable.groupby(["action_type", "outcome"]).agg(n=("gross_pnl","size"), gross=("gross_pnl","sum"), mean=("gross_pnl","mean"), clean=("clean_pnl","sum"), rescue=("rescue_pnl","sum"), max_loss=("gross_pnl","min")).to_string())
    show_cols = [c for c in ["policy", "action_type", "ticker", "pair_ticker", "outcome", "gross_pnl", "clean_pnl", "rescue_pnl", "matched_qty", "remaining_unmatched_qty", "p_hot", "stage2_p_good", "action_edge", "fill_qty_1", "fill_qty_2", "entry_px_1", "entry_px_2", "event_chain"] if c in actionable.columns]
    print("\nWorst trades:")
    print(actionable.sort_values("gross_pnl").head(30)[show_cols].to_string(index=False))
    print("\nBest trades:")
    print(actionable.sort_values("gross_pnl", ascending=False).head(30)[show_cols].to_string(index=False))